# One-fold validation

This notebook is the cleaned validation notebook supplied with the project. It expects the external dataset under `data/raw/` and writes generated artifacts under `results/`. Raw data and generated outputs are not committed to the repository.


In [ ]:
import os

BASE = "data/raw"

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for f in files:
        print(f"{indent}    {f}")

In [ ]:
import os
import glob

DATA_DIR = "data/raw"

MASTER_FILE = os.path.join(DATA_DIR, "SP500_master_panel.csv")
MACRO_FILE = os.path.join(DATA_DIR, "SP500_macro_panel.csv")
SECTOR_FILE = os.path.join(DATA_DIR, "SP500_sectors_clean.csv")
OHLC_FILE = os.path.join(DATA_DIR, "OHLC.xlsx")

print("DATA DIRECTORY:")
print(DATA_DIR)

print("\nMAIN FILES:")
print("Master:", os.path.exists(MASTER_FILE))
print("Macro:", os.path.exists(MACRO_FILE))
print("Sectors:", os.path.exists(SECTOR_FILE))
print("OHLC:", os.path.exists(OHLC_FILE))

SPX_FILES = sorted(
    glob.glob(os.path.join(DATA_DIR, "SPX as of Aug*.xlsx"))
)

print("\nHISTORICAL SPX FILES:", len(SPX_FILES))

for f in SPX_FILES:
    print(os.path.basename(f))

In [ ]:
import pandas as pd
import os

MASTER_FILE = "data/raw/SP500_master_panel.csv"

# Read only the header first
master_header = pd.read_csv(
    MASTER_FILE,
    nrows=0,
    encoding="utf-8-sig"
)

print("NUMBER OF COLUMNS:", len(master_header.columns))
print("\nRAW COLUMN NAMES:\n")

for i, col in enumerate(master_header.columns):
    print(i, repr(col))

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "data/raw"

MASTER_FILE = f"{DATA_DIR}/SP500_master_panel.csv"

master = pd.read_csv(
    MASTER_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

# Standardize column names
master.columns = master.columns.str.strip().str.lower()

# Convert date properly
master["date"] = pd.to_datetime(
    master["date"],
    errors="coerce"
)

# Sort chronologically
master = master.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

print("Shape:", master.shape)
print("Unique tickers:", master["ticker"].nunique())

print("\nDate range:")
print(master["date"].min(), "→", master["date"].max())

print("\nMissing dates:", master["date"].isna().sum())

print("\nColumns:")
print(master.columns.tolist())

In [ ]:
required_columns = [
    "date",
    "ticker",
    "total_return_index",
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "gics_sector",
    "gics_industry",
    "spx",
    "vix",
    "move",
    "credit_spread",
    "ust10y",
    "curve_2y10y",
    "fed_funds",
    "risk_free_rate",
    "monthly_return",
    "bid_ask_spread",
    "dollar_volume",
    "momentum_12_1",
    "excess_return",
    "in_sp500",
]

missing = [
    c for c in required_columns
    if c not in master.columns
]

if missing:
    print("❌ Missing columns:")
    print(missing)
else:
    print("✓ All 31 required columns are present.")

In [ ]:
print(
    master.groupby("ticker")["date"]
    .count()
    .describe()
)

In [ ]:
print("\nUnique dates:", master["date"].nunique())

print("\nFirst 20 dates:")
print(
    master["date"]
    .drop_duplicates()
    .sort_values()
    .head(20)
    .to_list()
)

In [ ]:
print("\nObservations by year:")

print(
    master.assign(
        year=master["date"].dt.year
    )
    .groupby("year")
    .size()
)

In [ ]:
print(
    master[
        [
            "date",
            "ticker",
            "monthly_return",
            "risk_free_rate",
            "excess_return"
        ]
    ].head(20)
)

In [ ]:
print("\nMissing target values:")

print(
    master[
        ["monthly_return", "risk_free_rate", "excess_return"]
    ].isna().sum()
)

In [ ]:
print("\nExcess return summary:")
print(master["excess_return"].describe())

In [ ]:
print(
    master["in_sp500"]
    .value_counts(dropna=False)
)

In [ ]:
membership_by_date = (
    master
    .groupby("date")["in_sp500"]
    .agg(
        observations="count",
        sp500_members="sum"
    )
)

print(membership_by_date.head(20))

In [ ]:
import glob
import os

SPX_FILES = sorted(
    glob.glob(
        os.path.join(
            DATA_DIR,
            "SPX as of Aug*.xlsx"
        )
    )
)

print("Number of SPX files:", len(SPX_FILES))

for i, f in enumerate(SPX_FILES, 1):
    print(f"{i:02d}. {os.path.basename(f)}")

In [ ]:
import pandas as pd

# Inspect the first SPX file
test_file = SPX_FILES[0]

df_test = pd.read_excel(test_file)

print("File:", os.path.basename(test_file))
print("Shape:", df_test.shape)

print("\nColumns:")
for i, col in enumerate(df_test.columns):
    print(i, repr(col))

print("\nFirst 5 rows:")
display(df_test.head())

In [ ]:
import glob
import os

SPX_FILES = sorted(
    glob.glob(
        os.path.join(
            DATA_DIR,
            "SPX as of Aug*.xlsx"
        )
    )
)

# Keep only actual Excel files
SPX_FILES = [
    f for f in SPX_FILES
    if f.lower().endswith((".xlsx", ".xls"))
]

print("Number of SPX Excel files:", len(SPX_FILES))

for i, f in enumerate(SPX_FILES, 1):
    print(f"{i:02d}. {os.path.basename(f)}")
    
for f in SPX_FILES:
    df = pd.read_excel(f)

    print(
        f"{os.path.basename(f):40s}",
        "rows =", len(df),
        "columns =", len(df.columns)
    )

In [ ]:
import openpyxl

print("openpyxl version:", openpyxl.__version__)

test_file = SPX_FILES[0]

df_test = pd.read_excel(
    test_file,
    engine="openpyxl"
)

print("File:", os.path.basename(test_file))
print("Shape:", df_test.shape)

print("\nColumns:")
for i, col in enumerate(df_test.columns):
    print(i, repr(col))

print("\nFirst 5 rows:")
display(df_test.head())

In [ ]:
for f in SPX_FILES:
    df = pd.read_excel(f, engine="openpyxl")

    print(
        f"{os.path.basename(f):40s}",
        "rows =", len(df),
        "columns =", len(df.columns)
    )

In [ ]:
import pandas as pd
import os

all_spx = []

for f in SPX_FILES:
    df = pd.read_excel(f, engine="openpyxl")

    # Extract the snapshot date from the filename
    filename = os.path.basename(f)
    snapshot_date = pd.to_datetime(
        filename.replace("SPX as of ", "").replace(".xlsx", ""),
        format="%b %d %Y"
    )

    # Store the snapshot date
    df["snapshot_date"] = snapshot_date

    all_spx.append(df)

print("Files loaded:", len(all_spx))

In [ ]:
spx_raw = pd.concat(
    all_spx,
    ignore_index=True
)

print("Combined shape:", spx_raw.shape)

In [ ]:
print("Columns:", len(spx_raw.columns))

for i, col in enumerate(spx_raw.columns):
    print(i, repr(col))

In [ ]:
print("\nSnapshot dates:")
print(
    spx_raw["snapshot_date"]
    .sort_values()
    .drop_duplicates()
    .to_string(index=False)
)

In [ ]:
print("\nRows per snapshot:")

print(
    spx_raw
    .groupby("snapshot_date")
    .size()
    .to_string()
)

In [ ]:
print(spx_raw.columns.tolist())

In [ ]:
for i, col in enumerate(spx_raw.columns):
    print(i, repr(col))

In [ ]:
# ============================================================
# STEP 1: LOAD AND INSPECT ALL S&P 500 SNAPSHOT FILES
# ============================================================

import pandas as pd
import os
import glob

# ------------------------------------------------------------
# 1. Find all SPX snapshot Excel files
# ------------------------------------------------------------

SPX_FILES = sorted(
    glob.glob(
        os.path.join(
            DATA_DIR,
            "SPX as of Aug*.xlsx"
        )
    )
)

# Keep only actual Excel files
SPX_FILES = [
    f for f in SPX_FILES
    if f.lower().endswith((".xlsx", ".xls"))
]

print("=" * 70)
print("S&P 500 FILE DISCOVERY")
print("=" * 70)

print("Number of SPX Excel files:", len(SPX_FILES))

for i, f in enumerate(SPX_FILES, 1):
    print(f"{i:02d}. {os.path.basename(f)}")


# ------------------------------------------------------------
# 2. Load every SPX file
# ------------------------------------------------------------

all_spx = []

print("\n" + "=" * 70)
print("LOADING FILES")
print("=" * 70)

for f in SPX_FILES:

    # Read Excel explicitly with openpyxl
    df = pd.read_excel(
        f,
        engine="openpyxl"
    )

    # --------------------------------------------------------
    # Extract snapshot date from filename
    #
    # Example:
    # "SPX as of Aug 10 2025.xlsx"
    # becomes:
    # 2025-08-10
    # --------------------------------------------------------

    filename = os.path.basename(f)

    date_string = (
        filename
        .replace("SPX as of ", "")
        .replace(".xlsx", "")
        .replace(".xls", "")
        .strip()
    )

    snapshot_date = pd.to_datetime(
        date_string,
        format="%b %d %Y"
    )

    # Add snapshot date to the dataframe
    df["snapshot_date"] = snapshot_date

    # Store the dataframe
    all_spx.append(df)

    print(
        f"{filename:40s} "
        f"rows = {len(df):5d} | "
        f"columns = {len(df.columns):2d} | "
        f"date = {snapshot_date.date()}"
    )


# ------------------------------------------------------------
# 3. Combine all files into one dataframe
# ------------------------------------------------------------

spx_raw = pd.concat(
    all_spx,
    ignore_index=True
)

print("\n" + "=" * 70)
print("COMBINED DATASET")
print("=" * 70)

print("Combined shape:", spx_raw.shape)
print("Number of rows:", len(spx_raw))
print("Number of columns:", len(spx_raw.columns))


# ------------------------------------------------------------
# 4. Display column names
# ------------------------------------------------------------

print("\nColumns:")

for i, col in enumerate(spx_raw.columns):
    print(f"{i}: {repr(col)}")


# ------------------------------------------------------------
# 5. Standardize column names
# ------------------------------------------------------------

spx_raw.columns = (
    spx_raw.columns
    .str.strip()
)

print("\nStandardized columns:")
print(spx_raw.columns.tolist())


# ------------------------------------------------------------
# 6. Check that required columns exist
# ------------------------------------------------------------

required_columns = [
    "Ticker",
    "Name",
    "Weight",
    "Shares",
    "Price",
    "snapshot_date"
]

missing_required = [
    col
    for col in required_columns
    if col not in spx_raw.columns
]

print("\nRequired-column check:")

if len(missing_required) == 0:
    print("✓ All expected columns are present.")
else:
    print("WARNING: Missing columns:")
    for col in missing_required:
        print("  -", col)


# ------------------------------------------------------------
# 7. Inspect the first rows
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRST 20 ROWS")
print("=" * 70)

display(
    spx_raw.head(20)
)


# ------------------------------------------------------------
# 8. Inspect the last rows
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LAST 20 ROWS")
print("=" * 70)

display(
    spx_raw.tail(20)
)


# ------------------------------------------------------------
# 9. Check snapshot dates
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SNAPSHOT DATES")
print("=" * 70)

snapshot_dates = (
    spx_raw["snapshot_date"]
    .drop_duplicates()
    .sort_values()
)

print(
    snapshot_dates.to_string(index=False)
)

print(
    "\nNumber of unique snapshot dates:",
    len(snapshot_dates)
)

print(
    "Earliest snapshot:",
    snapshot_dates.min().date()
)

print(
    "Latest snapshot:",
    snapshot_dates.max().date()
)


# ------------------------------------------------------------
# 10. Number of rows per snapshot
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ROWS PER SNAPSHOT")
print("=" * 70)

rows_per_snapshot = (
    spx_raw
    .groupby("snapshot_date")
    .size()
    .sort_index()
)

print(
    rows_per_snapshot.to_string()
)


# ------------------------------------------------------------
# 11. Unique companies
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNIQUE TICKERS")
print("=" * 70)

unique_tickers = spx_raw["Ticker"].nunique()

print(
    "Number of unique tickers:",
    unique_tickers
)


# ------------------------------------------------------------
# 12. Check duplicate ticker-date observations
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

duplicate_mask = spx_raw.duplicated(
    subset=[
        "snapshot_date",
        "Ticker"
    ],
    keep=False
)

duplicate_count = duplicate_mask.sum()

print(
    "Number of duplicated rows:",
    duplicate_count
)

if duplicate_count == 0:

    print(
        "✓ No duplicate ticker-date observations found."
    )

else:

    print(
        "WARNING: Duplicate ticker-date observations found."
    )

    print("\nDuplicate observations:")

    display(
        spx_raw.loc[
            duplicate_mask
        ]
        .sort_values(
            [
                "snapshot_date",
                "Ticker"
            ]
        )
    )


# ------------------------------------------------------------
# 13. Check missing values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing_values = (
    spx_raw
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

print(
    missing_values.to_string()
)


# ------------------------------------------------------------
# 14. Missing-value percentages
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUE PERCENTAGES")
print("=" * 70)

missing_percentage = (
    spx_raw
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
)

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percent": missing_percentage.round(2)
})

display(
    missing_summary
)


# ------------------------------------------------------------
# 15. Basic data types
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(
    spx_raw.dtypes.to_string()
)


# ------------------------------------------------------------
# 16. Basic numerical statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMERICAL SUMMARY")
print("=" * 70)

display(
    spx_raw.describe(
        include="all"
    ).T
)


# ------------------------------------------------------------
# 17. Check ticker values
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE TICKERS")
print("=" * 70)

print(
    spx_raw["Ticker"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .head(50)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 18. Check whether tickers are missing
# ------------------------------------------------------------

missing_tickers = (
    spx_raw["Ticker"]
    .isna()
    .sum()
)

print(
    "\nMissing ticker values:",
    missing_tickers
)


# ------------------------------------------------------------
# 19. Check whether snapshot dates are missing
# ------------------------------------------------------------

missing_dates = (
    spx_raw["snapshot_date"]
    .isna()
    .sum()
)

print(
    "Missing snapshot dates:",
    missing_dates
)


# ------------------------------------------------------------
# 20. Check weights
# ------------------------------------------------------------

if "Weight" in spx_raw.columns:

    print("\n" + "=" * 70)
    print("WEIGHT CHECK")
    print("=" * 70)

    print(
        "Weight data type:",
        spx_raw["Weight"].dtype
    )

    print(
        "\nSample weights:"
    )

    print(
        spx_raw["Weight"]
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 21. Check shares
# ------------------------------------------------------------

if "Shares" in spx_raw.columns:

    print("\n" + "=" * 70)
    print("SHARES CHECK")
    print("=" * 70)

    print(
        "Shares data type:",
        spx_raw["Shares"].dtype
    )

    print(
        "\nSample shares:"
    )

    print(
        spx_raw["Shares"]
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 22. Check prices
# ------------------------------------------------------------

if "Price" in spx_raw.columns:

    print("\n" + "=" * 70)
    print("PRICE CHECK")
    print("=" * 70)

    print(
        "Price data type:",
        spx_raw["Price"].dtype
    )

    print(
        "\nSample prices:"
    )

    print(
        spx_raw["Price"]
        .head(20)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 23. Check each snapshot's number of unique companies
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNIQUE COMPANIES PER SNAPSHOT")
print("=" * 70)

unique_per_snapshot = (
    spx_raw
    .groupby("snapshot_date")["Ticker"]
    .nunique()
    .sort_index()
)

print(
    unique_per_snapshot.to_string()
)


# ------------------------------------------------------------
# 24. Check for suspiciously small snapshots
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SNAPSHOT SIZE CHECK")
print("=" * 70)

for date, count in rows_per_snapshot.items():

    if count < 400:

        print(
            f"WARNING: {date.date()} has only "
            f"{count} rows."
        )

    else:

        print(
            f"OK: {date.date()} has "
            f"{count} rows."
        )


# ------------------------------------------------------------
# 25. Create a clean copy for subsequent processing
# ------------------------------------------------------------

spx = spx_raw.copy()

# Remove completely empty rows
spx = spx.dropna(
    how="all"
).reset_index(
    drop=True
)

print("\n" + "=" * 70)
print("FINAL INITIAL SPX DATASET")
print("=" * 70)

print(
    "Shape after removing completely empty rows:",
    spx.shape
)

print(
    "\nColumns:",
    spx.columns.tolist()
)

print(
    "\nDate range:",
    spx["snapshot_date"].min().date(),
    "to",
    spx["snapshot_date"].max().date()
)

print(
    "\nUnique tickers:",
    spx["Ticker"].nunique()
)

print(
    "\nUnique snapshots:",
    spx["snapshot_date"].nunique()
)

print("\n✓ SPX snapshot loading and diagnostics complete.")

In [ ]:
# ============================================================
# STEP 2: FIND AND INSPECT THE BLOOMBERG STOCK-LEVEL DATA
# ============================================================

import os
import glob
import pandas as pd

print("=" * 70)
print("SEARCHING DATA DIRECTORY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Show all files in DATA_DIR
# ------------------------------------------------------------

all_files = sorted(
    glob.glob(
        os.path.join(DATA_DIR, "*")
    )
)

print("Total files found:", len(all_files))

for i, f in enumerate(all_files, 1):
    print(
        f"{i:03d}. {os.path.basename(f)}"
    )


# ------------------------------------------------------------
# 2. Separate Excel / CSV files
# ------------------------------------------------------------

data_files = [
    f for f in all_files
    if f.lower().endswith(
        (".xlsx", ".xls", ".csv")
    )
]

print("\n" + "=" * 70)
print("DATA FILES")
print("=" * 70)

print(
    "Number of Excel/CSV files:",
    len(data_files)
)

for i, f in enumerate(data_files, 1):
    print(
        f"{i:03d}. {os.path.basename(f)}"
    )


# ------------------------------------------------------------
# 3. Identify files that are NOT SPX snapshots
# ------------------------------------------------------------

non_spx_files = [
    f for f in data_files
    if not os.path.basename(f).startswith(
        "SPX as of Aug"
    )
]

print("\n" + "=" * 70)
print("NON-SPX DATA FILES")
print("=" * 70)

print(
    "Number of non-SPX files:",
    len(non_spx_files)
)

for i, f in enumerate(non_spx_files, 1):
    print(
        f"{i:03d}. {os.path.basename(f)}"
    )


# ------------------------------------------------------------
# 4. Inspect every non-SPX file
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INSPECTING NON-SPX FILES")
print("=" * 70)

for f in non_spx_files:

    filename = os.path.basename(f)

    print("\n" + "-" * 70)
    print("FILE:", filename)
    print("-" * 70)

    try:

        # ----------------------------------------------------
        # Excel
        # ----------------------------------------------------

        if f.lower().endswith(
            (".xlsx", ".xls")
        ):

            # Get sheet names first
            excel_file = pd.ExcelFile(
                f,
                engine="openpyxl"
            )

            print(
                "Sheets:",
                excel_file.sheet_names
            )

            # Read first sheet only for inspection
            df_temp = pd.read_excel(
                f,
                sheet_name=0,
                engine="openpyxl"
            )

        # ----------------------------------------------------
        # CSV
        # ----------------------------------------------------

        elif f.lower().endswith(".csv"):

            df_temp = pd.read_csv(
                f
            )

        else:

            continue


        # ----------------------------------------------------
        # Basic information
        # ----------------------------------------------------

        print(
            "Shape:",
            df_temp.shape
        )

        print(
            "\nColumns:"
        )

        for j, col in enumerate(
            df_temp.columns
        ):
            print(
                f"  {j}: {repr(col)}"
            )


        # ----------------------------------------------------
        # First 5 rows
        # ----------------------------------------------------

        print(
            "\nFirst 5 rows:"
        )

        display(
            df_temp.head()
        )


        # ----------------------------------------------------
        # Data types
        # ----------------------------------------------------

        print(
            "\nData types:"
        )

        print(
            df_temp.dtypes.to_string()
        )


    except Exception as e:

        print(
            "ERROR reading file:",
            repr(e)
        )


# ------------------------------------------------------------
# 5. Specifically search for the 31-column dataset
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SEARCHING FOR EXPECTED BLOOMBERG VARIABLES")
print("=" * 70)

expected_columns = [
    "date",
    "ticker",
    "total_return_index",
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "gics_sector"
]

candidate_files = []

for f in data_files:

    filename = os.path.basename(f)

    # Skip the SPX constituent files
    if filename.startswith(
        "SPX as of Aug"
    ):
        continue

    try:

        if f.lower().endswith(
            (".xlsx", ".xls")
        ):

            df_check = pd.read_excel(
                f,
                nrows=5,
                engine="openpyxl"
            )

        elif f.lower().endswith(".csv"):

            df_check = pd.read_csv(
                f,
                nrows=5
            )

        else:

            continue


        # Strip whitespace for comparison
        cols_clean = [
            str(c).strip().lower()
            for c in df_check.columns
        ]

        matches = [
            c
            for c in expected_columns
            if c.lower() in cols_clean
        ]

        if len(matches) >= 5:

            candidate_files.append(
                (
                    f,
                    len(matches),
                    matches
                )
            )

    except Exception:
        pass


# ------------------------------------------------------------
# 6. Print candidate files
# ------------------------------------------------------------

if len(candidate_files) == 0:

    print(
        "No obvious Bloomberg stock-level file "
        "was automatically identified."
    )

    print(
        "\nLook at the NON-SPX DATA FILES section above."
    )

else:

    print(
        "Potential Bloomberg data files:"
    )

    for f, n, matches in candidate_files:

        print(
            f"\n{os.path.basename(f)}"
        )

        print(
            "Matching expected columns:",
            n
        )

        print(
            matches
        )


print("\n" + "=" * 70)
print("STEP 2 COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 3: IDENTIFY THE CORRECT MASTER DATASET
# ============================================================

import pandas as pd
import os

FILES_TO_CHECK = [
    "OHLC.xlsx",
    "SP500_macro_panel.csv",
    "SP500_master_panel.csv",
    "SP500_sectors_clean.csv"
]

for filename in FILES_TO_CHECK:

    filepath = os.path.join(DATA_DIR, filename)

    print("\n" + "=" * 80)
    print(f"FILE: {filename}")
    print("=" * 80)

    try:

        # ----------------------------------------------------
        # Excel
        # ----------------------------------------------------

        if filename.endswith(".xlsx"):

            xls = pd.ExcelFile(
                filepath,
                engine="openpyxl"
            )

            print("Sheets:", xls.sheet_names)

            for sheet in xls.sheet_names:

                temp = pd.read_excel(
                    filepath,
                    sheet_name=sheet,
                    engine="openpyxl",
                    nrows=5
                )

                print(
                    f"\nSheet: {sheet}"
                )

                print(
                    "Columns:",
                    len(temp.columns)
                )

                print(
                    [str(c) for c in temp.columns]
                )

                print(
                    "Shape of first 5 rows:",
                    temp.shape
                )

                display(temp.head(2))

        # ----------------------------------------------------
        # CSV
        # ----------------------------------------------------

        elif filename.endswith(".csv"):

            temp = pd.read_csv(
                filepath,
                nrows=5
            )

            print(
                "Columns:",
                len(temp.columns)
            )

            print(
                [str(c) for c in temp.columns]
            )

            display(temp.head(2))

            # Read only metadata about the full file
            full_temp = pd.read_csv(
                filepath,
                usecols=[0]
            )

            print(
                "Total rows:",
                len(full_temp)
            )

    except Exception as e:

        print(
            "ERROR:",
            repr(e)
        )


print("\n" + "=" * 80)
print("STEP 3 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 4: INSPECT THE MASTER STOCK-LEVEL PANEL
# ============================================================

import pandas as pd
import os

MASTER_FILE = os.path.join(
    DATA_DIR,
    "SP500_master_panel.csv"
)

# ------------------------------------------------------------
# 1. Load the master panel
# ------------------------------------------------------------

master = pd.read_csv(
    MASTER_FILE
)

print("=" * 80)
print("MASTER PANEL")
print("=" * 80)

print("Shape:", master.shape)
print("Rows:", len(master))
print("Columns:", len(master.columns))


# ------------------------------------------------------------
# 2. Show all columns
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMN NAMES")
print("=" * 80)

for i, col in enumerate(master.columns):
    print(f"{i:02d}  {repr(col)}")


# ------------------------------------------------------------
# 3. First 10 rows
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FIRST 10 ROWS")
print("=" * 80)

display(master.head(10))


# ------------------------------------------------------------
# 4. Last 10 rows
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LAST 10 ROWS")
print("=" * 80)

display(master.tail(10))


# ------------------------------------------------------------
# 5. Data types
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)

print(
    master.dtypes.to_string()
)


# ------------------------------------------------------------
# 6. Missing values
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

missing = (
    master
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

missing_pct = (
    master
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
)

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_pct.round(2)
})

display(
    missing_table
)


# ------------------------------------------------------------
# 7. Date inspection
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATE INSPECTION")
print("=" * 80)

if "date" in master.columns:

    master["date"] = pd.to_datetime(
        master["date"],
        errors="coerce"
    )

    print(
        "Earliest date:",
        master["date"].min()
    )

    print(
        "Latest date:",
        master["date"].max()
    )

    print(
        "Unique dates:",
        master["date"].nunique()
    )

    print(
        "Missing dates:",
        master["date"].isna().sum()
    )

    print("\nSample dates:")

    print(
        master["date"]
        .drop_duplicates()
        .sort_values()
        .head(20)
        .to_string(index=False)
    )

else:

    print("WARNING: 'date' column not found.")


# ------------------------------------------------------------
# 8. Ticker inspection
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TICKER INSPECTION")
print("=" * 80)

if "ticker" in master.columns:

    master["ticker"] = (
        master["ticker"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    print(
        "Unique tickers:",
        master["ticker"].nunique()
    )

    print(
        "Missing tickers:",
        master["ticker"].isna().sum()
    )

    print("\nFirst 50 tickers:")

    print(
        master["ticker"]
        .drop_duplicates()
        .sort_values()
        .head(50)
        .to_string(index=False)
    )

else:

    print("WARNING: 'ticker' column not found.")


# ------------------------------------------------------------
# 9. Duplicate date-ticker observations
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DUPLICATE DATE-TICKER CHECK")
print("=" * 80)

if (
    "date" in master.columns
    and
    "ticker" in master.columns
):

    duplicate_mask = master.duplicated(
        subset=[
            "date",
            "ticker"
        ],
        keep=False
    )

    duplicate_count = duplicate_mask.sum()

    print(
        "Duplicated rows:",
        duplicate_count
    )

    if duplicate_count > 0:

        print(
            "\nSample duplicates:"
        )

        display(
            master.loc[
                duplicate_mask
            ]
            .sort_values(
                [
                    "date",
                    "ticker"
                ]
            )
            .head(50)
        )

    else:

        print(
            "✓ No duplicate date-ticker observations."
        )


# ------------------------------------------------------------
# 10. Number of observations per ticker
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVATIONS PER TICKER")
print("=" * 80)

if "ticker" in master.columns:

    observations_per_ticker = (
        master
        .groupby("ticker")
        .size()
        .sort_values()
    )

    print(
        observations_per_ticker.describe()
    )

    print("\nLowest observation counts:")

    print(
        observations_per_ticker
        .head(20)
        .to_string()
    )

    print("\nHighest observation counts:")

    print(
        observations_per_ticker
        .tail(20)
        .to_string()
    )


# ------------------------------------------------------------
# 11. Number of stocks per date
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVATIONS PER DATE")
print("=" * 80)

if "date" in master.columns:

    stocks_per_date = (
        master
        .groupby("date")["ticker"]
        .nunique()
    )

    print(
        stocks_per_date.describe()
    )

    print("\nFirst 20 dates:")

    print(
        stocks_per_date
        .head(20)
        .to_string()
    )

    print("\nLast 20 dates:")

    print(
        stocks_per_date
        .tail(20)
        .to_string()
    )


# ------------------------------------------------------------
# 12. Numeric summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMERIC VARIABLES")
print("=" * 80)

numeric_columns = master.select_dtypes(
    include="number"
).columns.tolist()

print(
    "Number of numeric columns:",
    len(numeric_columns)
)

print(
    numeric_columns
)

display(
    master[
        numeric_columns
    ].describe().T
)


# ------------------------------------------------------------
# 13. Check the expected research variables
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("EXPECTED RESEARCH VARIABLES")
print("=" * 80)

expected_variables = [
    "date",
    "ticker",
    "total_return_index",
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "gics_sector"
]

for variable in expected_variables:

    if variable in master.columns:

        print(
            f"✓ {variable}"
        )

    else:

        print(
            f"✗ {variable} -- NOT FOUND"
        )


# ------------------------------------------------------------
# 14. Sector information
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SECTOR INFORMATION")
print("=" * 80)

if "gics_sector" in master.columns:

    print(
        "Unique sectors:",
        master["gics_sector"].nunique()
    )

    print(
        "\nSector counts:"
    )

    print(
        master["gics_sector"]
        .value_counts(dropna=False)
        .to_string()
    )


# ------------------------------------------------------------
# 15. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 4 COMPLETE")
print("=" * 80)

print(
    "Master panel shape:",
    master.shape
)

if "date" in master.columns:
    print(
        "Date range:",
        master["date"].min(),
        "to",
        master["date"].max()
    )

if "ticker" in master.columns:
    print(
        "Unique tickers:",
        master["ticker"].nunique()
    )

print(
    "✓ Master panel loaded and inspected."
)

In [ ]:
# ============================================================
# STEP 5: IDENTIFY VARIABLES AND UNDERSTAND THE RETURN DATA
# ============================================================

print("=" * 80)
print("ALL 31 MASTER-PANEL VARIABLES")
print("=" * 80)

for i, col in enumerate(master.columns):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 1. Inspect the first stock through time
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAMPLE STOCK THROUGH TIME")
print("=" * 80)

sample_ticker = master["ticker"].dropna().iloc[0]

print("Sample ticker:", sample_ticker)

sample_stock = (
    master[
        master["ticker"] == sample_ticker
    ]
    .sort_values("date")
)

display(
    sample_stock.head(15)
)


# ------------------------------------------------------------
# 2. Show the same stock's latest observations
# ------------------------------------------------------------

print("\nLatest observations:")

display(
    sample_stock.tail(15)
)


# ------------------------------------------------------------
# 3. Inspect total return index
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOTAL RETURN INDEX")
print("=" * 80)

if "total_return_index" in master.columns:

    print(
        "Data type:",
        master["total_return_index"].dtype
    )

    print(
        "Missing:",
        master["total_return_index"].isna().sum()
    )

    print(
        "Unique values:",
        master["total_return_index"].nunique()
    )

    print(
        "Minimum:",
        master["total_return_index"].min()
    )

    print(
        "Maximum:",
        master["total_return_index"].max()
    )

    print("\nSample values:")

    display(
        master[
            [
                "date",
                "ticker",
                "total_return_index"
            ]
        ]
        .sort_values(
            [
                "ticker",
                "date"
            ]
        )
        .head(30)
    )


# ------------------------------------------------------------
# 4. Calculate monthly return temporarily
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TEMPORARY MONTHLY RETURN CHECK")
print("=" * 80)

if "total_return_index" in master.columns:

    # Sort first
    temp = master[
        [
            "ticker",
            "date",
            "total_return_index"
        ]
    ].copy()

    temp = temp.sort_values(
        [
            "ticker",
            "date"
        ]
    )

    # Calculate one-period return
    temp["monthly_return"] = (
        temp
        .groupby("ticker")["total_return_index"]
        .pct_change()
    )

    print(
        "Monthly return summary:"
    )

    display(
        temp["monthly_return"]
        .describe(
            percentiles=[
                0.01,
                0.05,
                0.25,
                0.50,
                0.75,
                0.95,
                0.99
            ]
        )
    )

    print(
        "\nSample calculated returns:"
    )

    display(
        temp[
            [
                "ticker",
                "date",
                "total_return_index",
                "monthly_return"
            ]
        ]
        .dropna(subset=["monthly_return"])
        .head(30)
    )


# ------------------------------------------------------------
# 5. Check for suspicious return values
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RETURN SANITY CHECK")
print("=" * 80)

if "total_return_index" in master.columns:

    temp_returns = temp[
        "monthly_return"
    ].dropna()

    print(
        "Returns below -100%:",
        (temp_returns < -1).sum()
    )

    print(
        "Returns below -50%:",
        (temp_returns < -0.50).sum()
    )

    print(
        "Returns above +50%:",
        (temp_returns > 0.50).sum()
    )

    print(
        "Returns above +100%:",
        (temp_returns > 1.00).sum()
    )

    print(
        "\nLargest positive returns:"
    )

    display(
        temp[
            [
                "ticker",
                "date",
                "monthly_return"
            ]
        ]
        .sort_values(
            "monthly_return",
            ascending=False
        )
        .head(20)
    )

    print(
        "\nLargest negative returns:"
    )

    display(
        temp[
            [
                "ticker",
                "date",
                "monthly_return"
            ]
        ]
        .sort_values(
            "monthly_return",
            ascending=True
        )
        .head(20)
    )


# ------------------------------------------------------------
# 6. Check date frequency
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATE FREQUENCY")
print("=" * 80)

unique_dates = (
    master["date"]
    .drop_duplicates()
    .sort_values()
)

date_differences = (
    unique_dates
    .diff()
    .dropna()
)

print(
    "Number of unique dates:",
    len(unique_dates)
)

print(
    "\nMost common gaps between observations:"
)

print(
    date_differences
    .value_counts()
    .head(15)
)


# ------------------------------------------------------------
# 7. Check whether dates are month-end
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MONTH-END CHECK")
print("=" * 80)

month_end_check = (
    master["date"]
    .dropna()
    .map(
        lambda x:
        x == x + pd.offsets.MonthEnd(0)
    )
)

print(
    "Observations on month-end dates:",
    month_end_check.sum()
)

print(
    "Observations not on month-end dates:",
    (~month_end_check).sum()
)


# ------------------------------------------------------------
# 8. Inspect all variables with examples
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VARIABLE SAMPLE VALUES")
print("=" * 80)

for col in master.columns:

    print(
        f"\n{col}"
    )

    print(
        master[col]
        .dropna()
        .head(5)
        .tolist()
    )


# ------------------------------------------------------------
# 9. Final variable summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 5 COMPLETE")
print("=" * 80)

print(
    "The master panel contains",
    len(master.columns),
    "columns and",
    len(master),
    "observations."
)

print(
    "The next step will be determined from "
    "the variable/return diagnostics above."
)

In [ ]:
# ============================================================
# STEP 6: VALIDATE EXISTING FEATURES, RETURNS AND S&P 500
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 6 — DATA VALIDATION")
print("=" * 80)


# ============================================================
# 1. MONTHLY RETURN CONSISTENCY CHECK
# ============================================================

print("\n" + "=" * 80)
print("1. MONTHLY RETURN CONSISTENCY CHECK")
print("=" * 80)

check = master[
    [
        "ticker",
        "date",
        "total_return_index",
        "monthly_return"
    ]
].copy()

check = check.sort_values(["ticker", "date"])

check["calculated_return"] = (
    check
    .groupby("ticker")["total_return_index"]
    .pct_change(fill_method=None)
)

check["return_difference"] = (
    check["monthly_return"]
    - check["calculated_return"]
)

valid_return_comparisons = check[
    check["monthly_return"].notna()
    & check["calculated_return"].notna()
].copy()

print("Valid return comparisons:", len(valid_return_comparisons))

print(
    "Mean absolute difference:",
    valid_return_comparisons["return_difference"].abs().mean()
)

print(
    "Maximum absolute difference:",
    valid_return_comparisons["return_difference"].abs().max()
)

print(
    "Comparisons differing by > 0.0001:",
    (
        valid_return_comparisons["return_difference"].abs()
        > 0.0001
    ).sum()
)

print("\nLargest discrepancies:")

display(
    valid_return_comparisons[
        [
            "ticker",
            "date",
            "total_return_index",
            "monthly_return",
            "calculated_return",
            "return_difference"
        ]
    ]
    .sort_values(
        "return_difference",
        key=lambda x: x.abs(),
        ascending=False
    )
    .head(20)
)


# ============================================================
# 2. EXCESS RETURN CHECK
# ============================================================

print("\n" + "=" * 80)
print("2. EXCESS RETURN CHECK")
print("=" * 80)

if {
    "monthly_return",
    "risk_free_rate",
    "excess_return"
}.issubset(master.columns):

    excess_check = master[
        [
            "ticker",
            "date",
            "monthly_return",
            "risk_free_rate",
            "excess_return"
        ]
    ].copy()

    # risk_free_rate is assumed to be an annualized
    # percentage rate. Convert to monthly decimal units.

    excess_check["monthly_rf_decimal"] = (
        excess_check["risk_free_rate"] / 100 / 12
    )

    excess_check["expected_excess_decimal"] = (
        excess_check["monthly_return"]
        - excess_check["monthly_rf_decimal"]
    )

    excess_check["excess_difference"] = (
        excess_check["excess_return"]
        - excess_check["expected_excess_decimal"]
    )

    valid_excess = excess_check[
        excess_check["excess_return"].notna()
        & excess_check["monthly_return"].notna()
        & excess_check["risk_free_rate"].notna()
    ].copy()

    print(
        "Valid excess-return comparisons:",
        len(valid_excess)
    )

    print(
        "Mean absolute excess-return difference:",
        valid_excess["excess_difference"].abs().mean()
    )

    print(
        "Maximum absolute excess-return difference:",
        valid_excess["excess_difference"].abs().max()
    )

    print(
        "Differences > 0.0001:",
        (
            valid_excess["excess_difference"].abs()
            > 0.0001
        ).sum()
    )

    print("\nLargest discrepancies:")

    display(
        valid_excess[
            [
                "ticker",
                "date",
                "monthly_return",
                "risk_free_rate",
                "monthly_rf_decimal",
                "excess_return",
                "expected_excess_decimal",
                "excess_difference"
            ]
        ]
        .sort_values(
            "excess_difference",
            key=lambda x: x.abs(),
            ascending=False
        )
        .head(20)
    )


# ============================================================
# 3. S&P 500 MEMBERSHIP
# ============================================================

print("\n" + "=" * 80)
print("3. S&P 500 MEMBERSHIP")
print("=" * 80)

print(
    "Unique values:",
    master["in_sp500"].dropna().unique()
)

print("\nMembership counts:")

print(
    master["in_sp500"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMembership percentage:")

print(
    master["in_sp500"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)


# ============================================================
# 4. S&P 500 MEMBERSHIP THROUGH TIME
# ============================================================

print("\n" + "=" * 80)
print("4. S&P 500 MEMBERSHIP THROUGH TIME")
print("=" * 80)

membership_by_date = (
    master
    .groupby("date")
    .agg(
        total_tickers=("ticker", "nunique"),
        sp500_members=(
            "in_sp500",
            lambda x: (x == 1).sum()
        )
    )
)

membership_by_date["membership_pct"] = (
    membership_by_date["sp500_members"]
    / membership_by_date["total_tickers"]
    * 100
)

display(
    membership_by_date.head(20)
)

print("\nLatest dates:")

display(
    membership_by_date.tail(20)
)


# ============================================================
# 5. SECTOR / INDUSTRY VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("5. SECTOR / INDUSTRY VALIDATION")
print("=" * 80)

print("gics_sector unique values:")

print(
    master["gics_sector"]
    .dropna()
    .value_counts()
    .head(30)
    .to_string()
)

print("\ngics_industry unique values:")

print(
    master["gics_industry"]
    .dropna()
    .value_counts()
    .head(30)
    .to_string()
)


# ============================================================
# 6. COMPARE GICS DATA WITH CLEAN SECTOR FILE
# ============================================================

print("\n" + "=" * 80)
print("6. COMPARE GICS DATA WITH CLEAN SECTOR FILE")
print("=" * 80)

sector_file = os.path.join(
    DATA_DIR,
    "SP500_sectors_clean.csv"
)

sectors = pd.read_csv(sector_file)

sectors["ticker"] = (
    sectors["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print("Sector file shape:", sectors.shape)

print("\nSector file columns:")
print(sectors.columns.tolist())

print("\nSample sector file:")
display(sectors.head(20))


# ------------------------------------------------------------
# Merge only for comparison
# ------------------------------------------------------------

comparison = master[
    [
        "ticker",
        "gics_sector",
        "gics_industry"
    ]
].drop_duplicates("ticker")

comparison = comparison.merge(
    sectors,
    on="ticker",
    how="left",
    suffixes=("_master", "_clean")
)

print("\nMerged comparison shape:", comparison.shape)

display(comparison.head(30))


# ------------------------------------------------------------
# Compare labels
# ------------------------------------------------------------

comparison["sector_both_missing"] = (
    comparison["gics_sector_master"].isna()
    & comparison["gics_sector_clean"].isna()
)

comparison["industry_both_missing"] = (
    comparison["gics_industry_master"].isna()
    & comparison["gics_industry_clean"].isna()
)

comparison["sector_matches"] = (
    comparison["gics_sector_master"]
    .eq(comparison["gics_sector_clean"])
    | comparison["sector_both_missing"]
)

comparison["industry_matches"] = (
    comparison["gics_industry_master"]
    .eq(comparison["gics_industry_clean"])
    | comparison["industry_both_missing"]
)

sector_true_mismatch = (
    comparison["gics_sector_master"].notna()
    & comparison["gics_sector_clean"].notna()
    & (
        comparison["gics_sector_master"]
        != comparison["gics_sector_clean"]
    )
)

industry_true_mismatch = (
    comparison["gics_industry_master"].notna()
    & comparison["gics_industry_clean"].notna()
    & (
        comparison["gics_industry_master"]
        != comparison["gics_industry_clean"]
    )
)

print("\nSector matches:", comparison["sector_matches"].sum())

print(
    "Sector true mismatches:",
    sector_true_mismatch.sum()
)

print(
    "Sector both missing:",
    comparison["sector_both_missing"].sum()
)

print(
    "\nIndustry matches:",
    comparison["industry_matches"].sum()
)

print(
    "Industry true mismatches:",
    industry_true_mismatch.sum()
)

print(
    "Industry both missing:",
    comparison["industry_both_missing"].sum()
)

print("\nTrue sector mismatches:")

display(
    comparison.loc[
        sector_true_mismatch,
        [
            "ticker",
            "gics_sector_master",
            "gics_sector_clean"
        ]
    ]
)

print("\nTrue industry mismatches:")

display(
    comparison.loc[
        industry_true_mismatch,
        [
            "ticker",
            "gics_industry_master",
            "gics_industry_clean"
        ]
    ]
)


# ============================================================
# 7. EXTREME RETURN INVESTIGATION
# ============================================================

print("\n" + "=" * 80)
print("7. EXTREME RETURN INVESTIGATION")
print("=" * 80)

extreme = master[
    [
        "ticker",
        "date",
        "total_return_index",
        "monthly_return",
        "market_cap",
        "in_sp500"
    ]
].copy()

print(
    "Returns > +100%:",
    (extreme["monthly_return"] > 1).sum()
)

print(
    "Returns < -50%:",
    (extreme["monthly_return"] < -0.50).sum()
)

print("\nExtreme observations:")

display(
    extreme[
        extreme["monthly_return"].abs() > 1
    ]
    .sort_values(
        "monthly_return",
        ascending=False
    )
    .head(30)
)


# ============================================================
# 8. EXTREME RETURN + MARKET CAP CHECK
# ============================================================

print("\n" + "=" * 80)
print("8. EXTREME RETURN + MARKET CAP CHECK")
print("=" * 80)

extreme_check = master[
    master["monthly_return"].abs() > 1
][
    [
        "ticker",
        "date",
        "monthly_return",
        "market_cap",
        "in_sp500"
    ]
].copy()

display(
    extreme_check.head(30)
)

print("\nMarket cap summary for extreme observations:")

display(
    extreme_check["market_cap"].describe()
)


# ============================================================
# 9. FEATURE AVAILABILITY
# ============================================================

print("\n" + "=" * 80)
print("9. FEATURE AVAILABILITY")
print("=" * 80)

feature_columns = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
    "monthly_return",
    "excess_return",
    "bid_ask_spread",
    "dollar_volume"
]

availability = pd.DataFrame({
    "variable": feature_columns,
    "non_missing": [
        master[c].notna().sum()
        for c in feature_columns
    ],
    "missing": [
        master[c].isna().sum()
        for c in feature_columns
    ],
    "missing_pct": [
        master[c].isna().mean() * 100
        for c in feature_columns
    ]
})

availability["missing_pct"] = (
    availability["missing_pct"].round(2)
)

display(
    availability.sort_values(
        "missing_pct",
        ascending=False
    )
)


# ============================================================
# 10. ACTUAL VALID DATA COVERAGE BY TICKER
# ============================================================

print("\n" + "=" * 80)
print("10. TICKER DATE COVERAGE")
print("=" * 80)

ticker_coverage = (
    master
    .loc[
        master["total_return_index"].notna()
    ]
    .groupby("ticker")
    .agg(
        first_valid_date=("date", "min"),
        last_valid_date=("date", "max"),
        valid_observations=("date", "count")
    )
)

print("Coverage summary:")

display(
    ticker_coverage.describe()
)

print("\nTickers with fewer than 24 valid observations:")

display(
    ticker_coverage[
        ticker_coverage["valid_observations"] < 24
    ]
    .sort_values("valid_observations")
    .head(30)
)


# ============================================================
# 11. CHECK FOR FUTURE-DATED DATA
# ============================================================

print("\n" + "=" * 80)
print("11. FUTURE-DATED DATA CHECK")
print("=" * 80)

today = pd.Timestamp.today().normalize()

future_rows = master[
    master["date"] > today
].copy()

print("Today:", today)

print(
    "Rows dated after today:",
    len(future_rows)
)

if len(future_rows) > 0:

    print("\nFuture-date summary:")

    display(
        future_rows[
            [
                "ticker",
                "date",
                "total_return_index",
                "monthly_return",
                "market_cap"
            ]
        ].head(30)
    )

    populated_future = future_rows[
        future_rows[
            [
                "total_return_index",
                "monthly_return",
                "market_cap"
            ]
        ].notna().any(axis=1)
    ]

    print(
        "\nFuture rows containing populated market data:",
        len(populated_future)
    )

else:

    print("No future-dated rows found.")


# ============================================================
# 12. EXTREME RETURN DETAILS
# ============================================================

print("\n" + "=" * 80)
print("12. EXTREME RETURN DETAILS")
print("=" * 80)

extreme_detail = master[
    master["monthly_return"].abs() > 1
].copy()

extreme_detail = extreme_detail.sort_values(
    "monthly_return",
    ascending=False
)

print(
    "Total extreme observations:",
    len(extreme_detail)
)

detail_columns = [
    "ticker",
    "date",
    "total_return_index",
    "monthly_return",
    "market_cap",
    "volume",
    "bid",
    "ask",
    "in_sp500"
]

detail_columns = [
    c for c in detail_columns
    if c in extreme_detail.columns
]

display(
    extreme_detail[detail_columns].head(50)
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 80)
print("STEP 6 COMPLETE")
print("=" * 80)

print(
    "No data has been modified in this step."
)

print(
    "The existing master panel has been validated."
)

print(
    "Return consistency, excess returns, S&P 500 membership, "
    "GICS classification, extreme observations, feature availability, "
    "actual ticker coverage, and future-dated data have been checked."
)

print(
    "Next step will be the construction of the "
    "look-ahead-safe modelling dataset."
)

In [ ]:
# ============================================================
# STEP 7: CONSTRUCT LOOK-AHEAD-SAFE MODELLING DATASET
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 7 — LOOK-AHEAD-SAFE MODELLING DATASET")
print("=" * 80)

# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------
# This step creates a NEW modelling dataset.
# The original `master` dataframe is NOT modified.
#
# Target:
#   Predict next month's excess return.
#
# At month t:
#   Features come from month t.
#   Target is excess return at month t+1.
#
# Therefore:
#   X_t  ->  y_(t+1)
#
# No future information is used in the feature set.
# ------------------------------------------------------------


# ============================================================
# 1. CREATE MODELLING COPY
# ============================================================

model_data = master.copy()

model_data["date"] = pd.to_datetime(
    model_data["date"]
)

model_data = model_data.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

print(
    "Initial modelling panel shape:",
    model_data.shape
)


# ============================================================
# 2. REMOVE FUTURE-DATED OBSERVATIONS
# ============================================================

today = pd.Timestamp.today().normalize()

future_rows = (
    model_data["date"] > today
)

print(
    "\nFuture-dated rows removed:",
    future_rows.sum()
)

model_data = model_data.loc[
    ~future_rows
].copy()


# ============================================================
# 3. DEFINE CORE FEATURE SET
# ============================================================

feature_columns = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
    "bid_ask_spread",
    "dollar_volume"
]

feature_columns = [
    c for c in feature_columns
    if c in model_data.columns
]

print(
    "\nFeatures used:"
)

for c in feature_columns:
    print(" -", c)


# ============================================================
# 4. CREATE NEXT-MONTH TARGET
# ============================================================

model_data["target_excess_return"] = (
    model_data
    .groupby("ticker")["excess_return"]
    .shift(-1)
)

model_data["target_date"] = (
    model_data
    .groupby("ticker")["date"]
    .shift(-1)
)


# ============================================================
# 5. VERIFY TARGET IS ACTUALLY NEXT MONTH
# ============================================================

model_data["month_gap"] = (
    model_data["target_date"]
    .dt.to_period("M")
    - model_data["date"]
    .dt.to_period("M")
)

invalid_target_gap = (
    model_data["target_excess_return"].notna()
    &
    (model_data["month_gap"] != 1)
)

print(
    "\nInvalid target month gaps:",
    invalid_target_gap.sum()
)

if invalid_target_gap.sum() > 0:

    print(
        "\nInvalid target examples:"
    )

    display(
        model_data.loc[
            invalid_target_gap,
            [
                "ticker",
                "date",
                "target_date",
                "month_gap"
            ]
        ].head(20)
    )


# ============================================================
# 6. REQUIRE CURRENT-MONTH RETURN
# ============================================================

# A stock must have a valid current-month return
# before it can enter the modelling dataset.

before_return_filter = len(model_data)

model_data = model_data[
    model_data["monthly_return"].notna()
].copy()

print(
    "\nRows removed because current monthly return is missing:",
    before_return_filter - len(model_data)
)


# ============================================================
# 7. REQUIRE NEXT-MONTH TARGET
# ============================================================

before_target_filter = len(model_data)

model_data = model_data[
    model_data["target_excess_return"].notna()
].copy()

print(
    "Rows removed because next-month target is missing:",
    before_target_filter - len(model_data)
)


# ============================================================
# 8. REQUIRE S&P 500 MEMBERSHIP AT TIME t
# ============================================================

before_sp500_filter = len(model_data)

model_data = model_data[
    model_data["in_sp500"] == 1
].copy()

print(
    "Rows removed because stock was not an S&P 500 member:",
    before_sp500_filter - len(model_data)
)


# ============================================================
# 9. REMOVE OBSERVATIONS WITHOUT BASIC MARKET DATA
# ============================================================

basic_market_columns = [
    "market_cap",
    "monthly_return"
]

basic_market_columns = [
    c for c in basic_market_columns
    if c in model_data.columns
]

before_market_filter = len(model_data)

model_data = model_data.dropna(
    subset=basic_market_columns
)

print(
    "Rows removed because basic market data is missing:",
    before_market_filter - len(model_data)
)


# ============================================================
# 10. CREATE MISSINGNESS INDICATORS
# ============================================================

print("\nCreating feature missingness indicators...")

for col in feature_columns:

    model_data[f"{col}_missing"] = (
        model_data[col].isna().astype(int)
    )


# ============================================================
# 11. CHECK FEATURE AVAILABILITY
# ============================================================

print("\n" + "=" * 80)
print("FEATURE AVAILABILITY IN MODELLING DATA")
print("=" * 80)

model_availability = pd.DataFrame({
    "variable": feature_columns,
    "non_missing": [
        model_data[c].notna().sum()
        for c in feature_columns
    ],
    "missing": [
        model_data[c].isna().sum()
        for c in feature_columns
    ],
    "missing_pct": [
        model_data[c].isna().mean() * 100
        for c in feature_columns
    ]
})

model_availability["missing_pct"] = (
    model_availability["missing_pct"]
    .round(2)
)

display(
    model_availability.sort_values(
        "missing_pct",
        ascending=False
    )
)


# ============================================================
# 12. CHECK FOR DUPLICATE TICKER-DATE OBSERVATIONS
# ============================================================

print("\n" + "=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

duplicates = model_data.duplicated(
    subset=["ticker", "date"],
    keep=False
)

print(
    "Duplicate ticker-date rows:",
    duplicates.sum()
)

if duplicates.sum() > 0:

    display(
        model_data.loc[
            duplicates,
            ["ticker", "date"]
        ].head(30)
    )


# ============================================================
# 13. CHECK FOR LOOK-AHEAD IN TARGET ALIGNMENT
# ============================================================

print("\n" + "=" * 80)
print("LOOK-AHEAD ALIGNMENT CHECK")
print("=" * 80)

alignment_check = model_data[
    [
        "ticker",
        "date",
        "target_date",
        "monthly_return",
        "target_excess_return"
    ]
].head(20)

display(alignment_check)

print(
    "\nFeature date is always <= target date:",
    (
        model_data["date"]
        <= model_data["target_date"]
    ).all()
)

print(
    "Target is exactly one month ahead:",
    (
        model_data["month_gap"] == 1
    ).all()
)


# ============================================================
# 14. CHECK TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION")
print("=" * 80)

display(
    model_data["target_excess_return"].describe()
)

print(
    "\nTarget > +100%:",
    (
        model_data["target_excess_return"] > 1
    ).sum()
)

print(
    "Target < -50%:",
    (
        model_data["target_excess_return"] < -0.50
    ).sum()
)


# ============================================================
# 15. CHECK DATE RANGE
# ============================================================

print("\n" + "=" * 80)
print("MODELLING DATE RANGE")
print("=" * 80)

print(
    "First modelling date:",
    model_data["date"].min()
)

print(
    "Last modelling date:",
    model_data["date"].max()
)

print(
    "Number of unique dates:",
    model_data["date"].nunique()
)

print(
    "Number of unique tickers:",
    model_data["ticker"].nunique()
)


# ============================================================
# 16. CHECK OBSERVATIONS BY DATE
# ============================================================

print("\n" + "=" * 80)
print("OBSERVATIONS BY DATE")
print("=" * 80)

observations_by_date = (
    model_data
    .groupby("date")
    .agg(
        observations=("ticker", "count"),
        unique_tickers=("ticker", "nunique")
    )
)

display(
    observations_by_date.head(20)
)

print("\nLatest dates:")

display(
    observations_by_date.tail(20)
)


# ============================================================
# 17. FINAL MODELLING COLUMNS
# ============================================================

missing_indicator_columns = [
    f"{c}_missing"
    for c in feature_columns
]

final_model_columns = [
    "ticker",
    "date",
    "target_date",
    "target_excess_return",
    "in_sp500",
    "gics_sector",
    "gics_industry"
] + feature_columns + missing_indicator_columns

final_model_columns = [
    c for c in final_model_columns
    if c in model_data.columns
]

model_data = model_data[
    final_model_columns
].copy()


# ============================================================
# 18. SORT FINAL DATASET
# ============================================================

model_data = model_data.sort_values(
    ["date", "ticker"]
).reset_index(drop=True)


# ============================================================
# 19. FINAL SANITY CHECK
# ============================================================

print("\n" + "=" * 80)
print("FINAL MODELLING DATASET")
print("=" * 80)

print(
    "Final shape:",
    model_data.shape
)

print(
    "Unique tickers:",
    model_data["ticker"].nunique()
)

print(
    "Unique dates:",
    model_data["date"].nunique()
)

print(
    "Missing target values:",
    model_data["target_excess_return"].isna().sum()
)

print(
    "Non-S&P 500 observations:",
    (
        model_data["in_sp500"] != 1
    ).sum()
)

print(
    "Duplicate ticker-date rows:",
    model_data.duplicated(
        ["ticker", "date"]
    ).sum()
)

print(
    "Target dates before feature dates:",
    (
        model_data["target_date"]
        < model_data["date"]
    ).sum()
)

print("\nSample:")

display(
    model_data.head(20)
)


# ============================================================
# 20. SAVE MODELLING DATASET
# ============================================================

# ============================================================
# SAVE FINAL MODELLING DATASET
# ============================================================

print("\n" + "=" * 80)
print("SAVING FINAL MODELLING DATASET")
print("=" * 80)

# Kaggle data/raw is READ-ONLY.
# Save generated files under results instead.

MODEL_DATA_FILE = "results/modelling_dataset_step7.csv"

model_data.to_csv(
    MODEL_DATA_FILE,
    index=False
)

print("Saved successfully:")
print(MODEL_DATA_FILE)

print("\nFile shape:")
print(model_data.shape)

print("\nFile size:")
import os
print(
    f"{os.path.getsize(MODEL_DATA_FILE) / (1024**2):.2f} MB"
)

print("\n" + "=" * 80)
print("STEP 7 COMPLETE")
print("=" * 80)

# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 80)
print("STEP 7 COMPLETE")
print("=" * 80)

print(
    "A separate look-ahead-safe modelling dataset has been created."
)

print(
    "The original `master` dataframe was not modified."
)

print(
    "Features are measured at month t."
)

print(
    "The prediction target is next-month excess return."
)

print(
    "S&P 500 membership is evaluated at month t."
)

print(
    "Next step will be the train/validation/test time-series split."
)

In [ ]:
# ============================================================
# STEP 8: CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 8 — CHRONOLOGICAL MODEL SPLIT")
print("=" * 80)


# ============================================================
# 1. LOAD FINAL MODELLING DATASET
# ============================================================

MODEL_DATA_FILE = "results/modelling_dataset_step7.csv"

model_data = pd.read_csv(
    MODEL_DATA_FILE,
    parse_dates=["date", "target_date"]
)

print("Modelling dataset shape:", model_data.shape)


# ============================================================
# 2. SORT CHRONOLOGICALLY
# ============================================================

model_data = model_data.sort_values(
    ["date", "ticker"]
).reset_index(drop=True)


# ============================================================
# 3. CHECK DATE RANGE
# ============================================================

unique_dates = (
    model_data["date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("\nFirst date:", unique_dates.min())
print("Last date:", unique_dates.max())
print("Number of dates:", len(unique_dates))


# ============================================================
# 4. DEFINE CHRONOLOGICAL SPLIT
#
# Training:   first 70%
# Validation: next 15%
# Test:       final 15%
#
# IMPORTANT:
# No random shuffling.
# ============================================================

n_dates = len(unique_dates)

train_end_idx = int(n_dates * 0.70)
validation_end_idx = int(n_dates * 0.85)

train_end_date = unique_dates.iloc[
    train_end_idx - 1
]

validation_end_date = unique_dates.iloc[
    validation_end_idx - 1
]

test_start_date = unique_dates.iloc[
    validation_end_idx
]


# ============================================================
# 5. CREATE SPLITS
# ============================================================

train_data = model_data[
    model_data["date"] <= train_end_date
].copy()

validation_data = model_data[
    (model_data["date"] > train_end_date)
    &
    (model_data["date"] <= validation_end_date)
].copy()

test_data = model_data[
    model_data["date"] >= test_start_date
].copy()


# ============================================================
# 6. PRINT SPLIT INFORMATION
# ============================================================

print("\n" + "=" * 80)
print("SPLIT INFORMATION")
print("=" * 80)

print("\nTRAIN")
print("Start:", train_data["date"].min())
print("End:", train_data["date"].max())
print("Rows:", len(train_data))
print("Unique dates:", train_data["date"].nunique())
print("Unique tickers:", train_data["ticker"].nunique())

print("\nVALIDATION")
print("Start:", validation_data["date"].min())
print("End:", validation_data["date"].max())
print("Rows:", len(validation_data))
print("Unique dates:", validation_data["date"].nunique())
print("Unique tickers:", validation_data["ticker"].nunique())

print("\nTEST")
print("Start:", test_data["date"].min())
print("End:", test_data["date"].max())
print("Rows:", len(test_data))
print("Unique dates:", test_data["date"].nunique())
print("Unique tickers:", test_data["ticker"].nunique())


# ============================================================
# 7. CHECK FOR OVERLAPPING DATES
# ============================================================

print("\n" + "=" * 80)
print("DATE OVERLAP CHECK")
print("=" * 80)

train_dates = set(train_data["date"])
validation_dates = set(validation_data["date"])
test_dates = set(test_data["date"])

print(
    "Train ∩ Validation:",
    len(train_dates & validation_dates)
)

print(
    "Train ∩ Test:",
    len(train_dates & test_dates)
)

print(
    "Validation ∩ Test:",
    len(validation_dates & test_dates)
)


# ============================================================
# 8. CHECK TEMPORAL ORDER
# ============================================================

print("\n" + "=" * 80)
print("TEMPORAL ORDER CHECK")
print("=" * 80)

print(
    "Training ends before validation:",
    train_data["date"].max()
    < validation_data["date"].min()
)

print(
    "Validation ends before test:",
    validation_data["date"].max()
    < test_data["date"].min()
)

print(
    "Training dates before test dates:",
    train_data["date"].max()
    < test_data["date"].min()
)


# ============================================================
# 9. TARGET DISTRIBUTION BY SPLIT
# ============================================================

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION")
print("=" * 80)

target = "target_excess_return"

target_summary = pd.DataFrame({
    "train": train_data[target].describe(),
    "validation": validation_data[target].describe(),
    "test": test_data[target].describe()
})

display(
    target_summary
)


# ============================================================
# 10. EXTREME TARGET CHECK
# ============================================================

print("\n" + "=" * 80)
print("EXTREME TARGET CHECK")
print("=" * 80)

for name, data in [
    ("TRAIN", train_data),
    ("VALIDATION", validation_data),
    ("TEST", test_data)
]:

    print(f"\n{name}")

    print(
        "Target > +100%:",
        (data[target] > 1).sum()
    )

    print(
        "Target < -50%:",
        (data[target] < -0.50).sum()
    )


# ============================================================
# 11. CHECK TARGET / FEATURE DATE RELATIONSHIP
# ============================================================

print("\n" + "=" * 80)
print("LOOK-AHEAD CHECK")
print("=" * 80)

for name, data in [
    ("TRAIN", train_data),
    ("VALIDATION", validation_data),
    ("TEST", test_data)
]:

    print(
        f"{name} — target date after feature date:",
        (data["target_date"] > data["date"]).all()
    )


# ============================================================
# 12. SAVE SPLITS
# ============================================================

TRAIN_FILE = "results/train_data_step8.csv"
VALIDATION_FILE = "results/validation_data_step8.csv"
TEST_FILE = "results/test_data_step8.csv"

train_data.to_csv(
    TRAIN_FILE,
    index=False
)

validation_data.to_csv(
    VALIDATION_FILE,
    index=False
)

test_data.to_csv(
    TEST_FILE,
    index=False
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("STEP 8 COMPLETE")
print("=" * 80)

print("\nTraining file:")
print(TRAIN_FILE)

print("\nValidation file:")
print(VALIDATION_FILE)

print("\nTest file:")
print(TEST_FILE)

print("\nFinal split sizes:")
print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)

In [ ]:
# ============================================================
# STEP 9: TRAINING-ONLY FEATURE PREPROCESSING
# FIXED VERSION
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("=" * 80)
print("STEP 9 — TRAINING-ONLY FEATURE PREPROCESSING")
print("=" * 80)


# ============================================================
# 1. LOAD STEP 8 DATA
# ============================================================

TRAIN_FILE = "results/train_data_step8.csv"
VALIDATION_FILE = "results/validation_data_step8.csv"
TEST_FILE = "results/test_data_step8.csv"

train_data = pd.read_csv(
    TRAIN_FILE,
    parse_dates=["date", "target_date"]
)

validation_data = pd.read_csv(
    VALIDATION_FILE,
    parse_dates=["date", "target_date"]
)

test_data = pd.read_csv(
    TEST_FILE,
    parse_dates=["date", "target_date"]
)

print("Train shape:", train_data.shape)
print("Validation shape:", validation_data.shape)
print("Test shape:", test_data.shape)


# ============================================================
# 2. DEFINE ORIGINAL FEATURES
# ============================================================

original_feature_columns = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
    "bid_ask_spread",
    "dollar_volume"
]


# ============================================================
# 3. IDENTIFY FEATURES WITH ZERO TRAINING OBSERVATIONS
# ============================================================

print("\n" + "=" * 80)
print("CHECK FOR FEATURES WITH NO TRAINING DATA")
print("=" * 80)

zero_observation_features = [
    c
    for c in original_feature_columns
    if train_data[c].notna().sum() == 0
]

print(
    "Features with zero observed training values:"
)

print(
    zero_observation_features
)


# ============================================================
# 4. DROP FEATURES THAT CANNOT BE LEARNED FROM TRAINING
# ============================================================

feature_columns = [
    c
    for c in original_feature_columns
    if c not in zero_observation_features
]

print("\nFeatures retained:")

for c in feature_columns:
    print(" -", c)

print(
    "\nNumber of retained numeric features:",
    len(feature_columns)
)

if zero_observation_features:
    print(
        "\nDropped because they contain no training observations:"
    )

    for c in zero_observation_features:
        print(" -", c)


# ============================================================
# 5. DEFINE MISSINGNESS INDICATORS
# ============================================================

missing_indicator_columns = [
    f"{c}_missing"
    for c in [
        "eps",
        "dividend_yield",
        "short_interest_ratio",
        "best_eps",
        "px_to_sales",
        "price_to_book",
        "oper_margin",
        "momentum_12_1",
        "bid_ask_spread"
    ]
]

available_indicators = [
    c
    for c in missing_indicator_columns
    if c in train_data.columns
]

model_features = (
    feature_columns
    + available_indicators
)

print(
    "\nNumber of missingness indicators:",
    len(available_indicators)
)

print(
    "Total model features:",
    len(model_features)
)


# ============================================================
# 6. VERIFY FEATURES
# ============================================================

missing_features = [
    c
    for c in model_features
    if c not in train_data.columns
]

if missing_features:
    raise ValueError(
        f"Missing required features: {missing_features}"
    )

print("\nAll required features are present.")


# ============================================================
# 7. EXTRACT FEATURES AND TARGET
# ============================================================

TARGET = "target_excess_return"

X_train = train_data[model_features].copy()
X_validation = validation_data[model_features].copy()
X_test = test_data[model_features].copy()

y_train = train_data[TARGET].copy()
y_validation = validation_data[TARGET].copy()
y_test = test_data[TARGET].copy()


# ============================================================
# 8. SEPARATE NUMERIC FEATURES FROM INDICATORS
# ============================================================

numeric_features = feature_columns


# ============================================================
# 9. MISSING VALUES BEFORE IMPUTATION
# ============================================================

print("\n" + "=" * 80)
print("MISSING VALUES BEFORE IMPUTATION")
print("=" * 80)

missing_before = pd.DataFrame({
    "variable": numeric_features,
    "train_missing": [
        X_train[c].isna().sum()
        for c in numeric_features
    ],
    "validation_missing": [
        X_validation[c].isna().sum()
        for c in numeric_features
    ],
    "test_missing": [
        X_test[c].isna().sum()
        for c in numeric_features
    ]
})

display(missing_before)


# ============================================================
# 10. MEDIAN IMPUTATION
#
# FIT ONLY ON TRAINING DATA
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

X_train_numeric = imputer.fit_transform(
    X_train[numeric_features]
)

X_validation_numeric = imputer.transform(
    X_validation[numeric_features]
)

X_test_numeric = imputer.transform(
    X_test[numeric_features]
)


# ============================================================
# 11. CONVERT BACK TO DATAFRAMES
# ============================================================

X_train_numeric = pd.DataFrame(
    X_train_numeric,
    columns=numeric_features,
    index=X_train.index
)

X_validation_numeric = pd.DataFrame(
    X_validation_numeric,
    columns=numeric_features,
    index=X_validation.index
)

X_test_numeric = pd.DataFrame(
    X_test_numeric,
    columns=numeric_features,
    index=X_test.index
)


# ============================================================
# 12. ADD MISSINGNESS INDICATORS
# ============================================================

X_train_processed = X_train_numeric.copy()
X_validation_processed = X_validation_numeric.copy()
X_test_processed = X_test_numeric.copy()

for c in available_indicators:

    X_train_processed[c] = (
        X_train[c]
        .fillna(0)
        .astype(int)
    )

    X_validation_processed[c] = (
        X_validation[c]
        .fillna(0)
        .astype(int)
    )

    X_test_processed[c] = (
        X_test[c]
        .fillna(0)
        .astype(int)
    )


# ============================================================
# 13. VERIFY FEATURE COUNT
# ============================================================

print("\n" + "=" * 80)
print("FEATURE COUNT CHECK")
print("=" * 80)

print(
    "Expected features:",
    len(model_features)
)

print(
    "Actual train features:",
    X_train_processed.shape[1]
)

print(
    "Actual validation features:",
    X_validation_processed.shape[1]
)

print(
    "Actual test features:",
    X_test_processed.shape[1]
)

if (
    X_train_processed.shape[1]
    != len(model_features)
):
    raise ValueError(
        "Training feature count does not match model feature list."
    )


# ============================================================
# 14. CHECK MISSING VALUES AFTER IMPUTATION
# ============================================================

print("\n" + "=" * 80)
print("MISSING VALUES AFTER IMPUTATION")
print("=" * 80)

print(
    "Train missing:",
    X_train_processed.isna().sum().sum()
)

print(
    "Validation missing:",
    X_validation_processed.isna().sum().sum()
)

print(
    "Test missing:",
    X_test_processed.isna().sum().sum()
)


# ============================================================
# 15. STANDARDIZE FEATURES
#
# FIT ONLY ON TRAINING DATA
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_processed
)

X_validation_scaled = scaler.transform(
    X_validation_processed
)

X_test_scaled = scaler.transform(
    X_test_processed
)


# ============================================================
# 16. CONVERT TO DATAFRAMES
# ============================================================

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=model_features,
    index=X_train.index
)

X_validation_scaled = pd.DataFrame(
    X_validation_scaled,
    columns=model_features,
    index=X_validation.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=model_features,
    index=X_test.index
)


# ============================================================
# 17. STANDARDIZATION CHECK
# ============================================================

print("\n" + "=" * 80)
print("TRAINING STANDARDIZATION CHECK")
print("=" * 80)

standardization_check = pd.DataFrame({
    "feature": model_features,
    "mean": X_train_scaled.mean().values,
    "std": X_train_scaled.std(ddof=0).values
})

display(
    standardization_check
)


# ============================================================
# 18. INFINITE VALUE CHECK
# ============================================================

print("\n" + "=" * 80)
print("INFINITE VALUE CHECK")
print("=" * 80)

print(
    "Train infinite values:",
    np.isinf(X_train_scaled).sum().sum()
)

print(
    "Validation infinite values:",
    np.isinf(X_validation_scaled).sum().sum()
)

print(
    "Test infinite values:",
    np.isinf(X_test_scaled).sum().sum()
)


# ============================================================
# 19. FINAL DATASET SHAPES
# ============================================================

print("\n" + "=" * 80)
print("FINAL PREPROCESSED DATA")
print("=" * 80)

print("X_train:", X_train_scaled.shape)
print("y_train:", y_train.shape)

print(
    "X_validation:",
    X_validation_scaled.shape
)

print(
    "y_validation:",
    y_validation.shape
)

print("X_test:", X_test_scaled.shape)
print("y_test:", y_test.shape)


# ============================================================
# 20. SAVE PREPROCESSING OBJECTS
# ============================================================

PREPROCESSING_FILE = (
    "results/"
    "preprocessing_objects_step9.pkl"
)

joblib.dump(
    {
        "imputer": imputer,
        "scaler": scaler,
        "model_features": model_features,
        "numeric_features": numeric_features,
        "missing_indicators": available_indicators,
        "dropped_features": zero_observation_features
    },
    PREPROCESSING_FILE
)

print(
    "\nPreprocessing objects saved:"
)

print(PREPROCESSING_FILE)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 80)
print("STEP 9 COMPLETE")
print("=" * 80)

print(
    "Imputation fitted on TRAIN only: True"
)

print(
    "Scaling fitted on TRAIN only: True"
)

print(
    "Validation/test information used during preprocessing: False"
)

print(
    "Features with zero training observations dropped:",
    zero_observation_features
)

print(
    "Ready for model training."
)

In [ ]:
# ============================================================
# STEP 10: MODEL TRAINING AND BASELINE COMPARISON
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

print("=" * 80)
print("STEP 10 — MODEL TRAINING AND BASELINE COMPARISON")
print("=" * 80)


# ============================================================
# 1. VERIFY STEP 9 OBJECTS EXIST
# ============================================================

required_objects = [
    "X_train_scaled",
    "X_validation_scaled",
    "X_test_scaled",
    "y_train",
    "y_validation",
    "y_test",
    "model_features"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise ValueError(
        f"Missing Step 9 objects: {missing_objects}"
    )

print("\nStep 9 preprocessing objects found.")


# ============================================================
# 2. BASIC DATA CHECK
# ============================================================

print("\n" + "=" * 80)
print("DATA CHECK")
print("=" * 80)

print("Training:", X_train_scaled.shape)
print("Validation:", X_validation_scaled.shape)
print("Test:", X_test_scaled.shape)

print(
    "Training target missing:",
    y_train.isna().sum()
)

print(
    "Validation target missing:",
    y_validation.isna().sum()
)

print(
    "Test target missing:",
    y_test.isna().sum()
)


# ============================================================
# 3. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    X,
    y,
    name
):

    predictions = model.predict(X)

    mse = mean_squared_error(
        y,
        predictions
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y,
        predictions
    )

    r2 = r2_score(
        y,
        predictions
    )

    correlation = np.corrcoef(
        y,
        predictions
    )[0, 1]

    results = {
        "model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Prediction_Correlation": correlation
    }

    return results, predictions


# ============================================================
# 4. RIDGE REGRESSION HYPERPARAMETER SEARCH
# ============================================================

print("\n" + "=" * 80)
print("RIDGE REGRESSION")
print("=" * 80)

ridge_alphas = [
    0.01,
    0.1,
    1,
    10,
    100,
    1000,
    10000
]

ridge_validation_results = []

for alpha in ridge_alphas:

    ridge = Ridge(
        alpha=alpha
    )

    ridge.fit(
        X_train_scaled,
        y_train
    )

    results, _ = evaluate_model(
        ridge,
        X_validation_scaled,
        y_validation,
        f"Ridge_alpha_{alpha}"
    )

    results["alpha"] = alpha

    ridge_validation_results.append(
        results
    )


ridge_validation_results = pd.DataFrame(
    ridge_validation_results
)

ridge_validation_results = (
    ridge_validation_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\nValidation results:")

display(
    ridge_validation_results
)


# ============================================================
# 5. SELECT BEST RIDGE
# ============================================================

best_ridge_alpha = (
    ridge_validation_results
    .iloc[0]["alpha"]
)

print(
    "\nBest Ridge alpha:",
    best_ridge_alpha
)


# ============================================================
# 6. TRAIN FINAL RIDGE
# ============================================================

best_ridge = Ridge(
    alpha=best_ridge_alpha
)

best_ridge.fit(
    X_train_scaled,
    y_train
)

ridge_test_results, ridge_test_predictions = (
    evaluate_model(
        best_ridge,
        X_test_scaled,
        y_test,
        "Ridge"
    )
)

print("\nRidge test performance:")

display(
    pd.DataFrame([
        ridge_test_results
    ])
)


# ============================================================
# 7. RANDOM FOREST
# ============================================================

print("\n" + "=" * 80)
print("RANDOM FOREST")
print("=" * 80)

rf_parameters = [
    {
        "n_estimators": 300,
        "max_depth": 5,
        "min_samples_leaf": 20
    },
    {
        "n_estimators": 300,
        "max_depth": 10,
        "min_samples_leaf": 20
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 20
    },
    {
        "n_estimators": 300,
        "max_depth": 10,
        "min_samples_leaf": 50
    }
]

rf_validation_results = []

for params in rf_parameters:

    print(
        "\nTesting:",
        params
    )

    rf = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    rf.fit(
        X_train_scaled,
        y_train
    )

    results, _ = evaluate_model(
        rf,
        X_validation_scaled,
        y_validation,
        "Random Forest"
    )

    results.update(params)

    rf_validation_results.append(
        results
    )


rf_validation_results = pd.DataFrame(
    rf_validation_results
)

rf_validation_results = (
    rf_validation_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\nRandom Forest validation results:")

display(
    rf_validation_results
)


# ============================================================
# 8. SELECT BEST RANDOM FOREST
# ============================================================

best_rf_params = (
    rf_validation_results
    .iloc[0]
)

print(
    "\nBest Random Forest parameters:"
)

print(
    best_rf_params[
        [
            "n_estimators",
            "max_depth",
            "min_samples_leaf"
        ]
    ]
)


# ============================================================
# 9. TRAIN FINAL RANDOM FOREST
# ============================================================

best_rf = RandomForestRegressor(
    n_estimators=int(
        best_rf_params["n_estimators"]
    ),
    max_depth=int(
        best_rf_params["max_depth"]
    ),
    min_samples_leaf=int(
        best_rf_params["min_samples_leaf"]
    ),
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

best_rf.fit(
    X_train_scaled,
    y_train
)

rf_test_results, rf_test_predictions = (
    evaluate_model(
        best_rf,
        X_test_scaled,
        y_test,
        "Random Forest"
    )
)

print("\nRandom Forest test performance:")

display(
    pd.DataFrame([
        rf_test_results
    ])
)


# ============================================================
# 10. HISTOGRAM GRADIENT BOOSTING
# ============================================================

print("\n" + "=" * 80)
print("HISTOGRAM GRADIENT BOOSTING")
print("=" * 80)

hgb_parameters = [
    {
        "max_iter": 200,
        "learning_rate": 0.03,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 30
    },
    {
        "max_iter": 300,
        "learning_rate": 0.03,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 30
    },
    {
        "max_iter": 300,
        "learning_rate": 0.05,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 30
    },
    {
        "max_iter": 300,
        "learning_rate": 0.03,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 30
    }
]

hgb_validation_results = []

for params in hgb_parameters:

    print(
        "\nTesting:",
        params
    )

    hgb = HistGradientBoostingRegressor(
        max_iter=params["max_iter"],
        learning_rate=params["learning_rate"],
        max_leaf_nodes=params["max_leaf_nodes"],
        min_samples_leaf=params["min_samples_leaf"],
        l2_regularization=1.0,
        random_state=42
    )

    hgb.fit(
        X_train_scaled,
        y_train
    )

    results, _ = evaluate_model(
        hgb,
        X_validation_scaled,
        y_validation,
        "HistGradientBoosting"
    )

    results.update(params)

    hgb_validation_results.append(
        results
    )


hgb_validation_results = pd.DataFrame(
    hgb_validation_results
)

hgb_validation_results = (
    hgb_validation_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print("\nHGB validation results:")

display(
    hgb_validation_results
)


# ============================================================
# 11. SELECT BEST HGB
# ============================================================

best_hgb_params = (
    hgb_validation_results
    .iloc[0]
)

print(
    "\nBest HGB parameters:"
)

print(
    best_hgb_params[
        [
            "max_iter",
            "learning_rate",
            "max_leaf_nodes",
            "min_samples_leaf"
        ]
    ]
)


# ============================================================
# 12. TRAIN FINAL HGB
# ============================================================

best_hgb = HistGradientBoostingRegressor(
    max_iter=int(
        best_hgb_params["max_iter"]
    ),
    learning_rate=float(
        best_hgb_params["learning_rate"]
    ),
    max_leaf_nodes=int(
        best_hgb_params["max_leaf_nodes"]
    ),
    min_samples_leaf=int(
        best_hgb_params["min_samples_leaf"]
    ),
    l2_regularization=1.0,
    random_state=42
)

best_hgb.fit(
    X_train_scaled,
    y_train
)

hgb_test_results, hgb_test_predictions = (
    evaluate_model(
        best_hgb,
        X_test_scaled,
        y_test,
        "HistGradientBoosting"
    )
)

print("\nHGB test performance:")

display(
    pd.DataFrame([
        hgb_test_results
    ])
)


# ============================================================
# 13. FINAL MODEL COMPARISON
# ============================================================

print("\n" + "=" * 80)
print("FINAL TEST-SET MODEL COMPARISON")
print("=" * 80)

test_results = pd.DataFrame([
    ridge_test_results,
    rf_test_results,
    hgb_test_results
])

test_results = (
    test_results
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(
    test_results
)


# ============================================================
# 14. COMPARE AGAINST ZERO-RETURN BASELINE
# ============================================================

print("\n" + "=" * 80)
print("ZERO-RETURN BASELINE")
print("=" * 80)

zero_predictions = np.zeros(
    len(y_test)
)

zero_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        zero_predictions
    )
)

zero_mae = mean_absolute_error(
    y_test,
    zero_predictions
)

zero_r2 = r2_score(
    y_test,
    zero_predictions
)

zero_baseline = pd.DataFrame([
    {
        "model": "Zero-return baseline",
        "RMSE": zero_rmse,
        "MAE": zero_mae,
        "R2": zero_r2,
        "Prediction_Correlation": np.nan
    }
])

display(
    zero_baseline
)


# ============================================================
# 15. COMBINED RESULTS
# ============================================================

all_results = pd.concat(
    [
        zero_baseline,
        test_results
    ],
    ignore_index=True
)

print("\nCombined test results:")

display(
    all_results
)


# ============================================================
# 16. SAVE MODELS
# ============================================================

MODEL_FILE = (
    "results/"
    "models_step10.pkl"
)

joblib.dump(
    {
        "ridge": best_ridge,
        "random_forest": best_rf,
        "hist_gradient_boosting": best_hgb,
        "model_features": model_features,
        "best_ridge_alpha": best_ridge_alpha,
        "best_rf_params": best_rf_params.to_dict(),
        "best_hgb_params": best_hgb_params.to_dict()
    },
    MODEL_FILE
)

print(
    "\nModels saved:"
)

print(MODEL_FILE)


# ============================================================
# 17. SAVE RESULTS
# ============================================================

RESULTS_FILE = (
    "results/"
    "model_comparison_step10.csv"
)

all_results.to_csv(
    RESULTS_FILE,
    index=False
)

print(
    "Results saved:"
)

print(RESULTS_FILE)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 80)
print("STEP 10 COMPLETE")
print("=" * 80)

print(
    "Models trained:",
    [
        "Ridge",
        "Random Forest",
        "HistGradientBoosting"
    ]
)

print(
    "Hyperparameters selected using validation data: True"
)

print(
    "Final evaluation performed on test data: True"
)

print(
    "Test data used for hyperparameter selection: False"
)

In [ ]:
# ============================================================
# STEP 9: TRAINING-ONLY FEATURE PREPROCESSING
# ============================================================

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("=" * 80)
print("STEP 9 — TRAINING-ONLY FEATURE PREPROCESSING")
print("=" * 80)


# ============================================================
# 1. DEFINE FEATURES
# ============================================================

numeric_features = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
    "bid_ask_spread",
    "dollar_volume"
]

missingness_features = [
    "eps_missing",
    "dividend_yield_missing",
    "short_interest_ratio_missing",
    "best_eps_missing",
    "px_to_sales_missing",
    "price_to_book_missing",
    "oper_margin_missing",
    "momentum_12_1_missing"
]

model_features = (
    numeric_features
    + missingness_features
)


# ============================================================
# 2. CHECK DATASETS
# ============================================================

print("\nTrain shape:", train_data.shape)
print("Validation shape:", validation_data.shape)
print("Test shape:", test_data.shape)

print("\nNumber of base features:", len(numeric_features))
print(
    "Number of missingness indicators:",
    len(missingness_features)
)
print(
    "Total intended model features:",
    len(model_features)
)


# ============================================================
# 3. VERIFY ALL FEATURES EXIST
# ============================================================

missing_columns = [
    c for c in model_features
    if c not in train_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required features: {missing_columns}"
    )

print("\nAll required features are present.")


# ============================================================
# 4. MISSING VALUES BEFORE IMPUTATION
# ============================================================

print("\n" + "=" * 80)
print("MISSING VALUES BEFORE IMPUTATION")
print("=" * 80)

missing_report = pd.DataFrame({
    "variable": model_features,

    "train_missing": [
        train_data[c].isna().sum()
        for c in model_features
    ],

    "validation_missing": [
        validation_data[c].isna().sum()
        for c in model_features
    ],

    "test_missing": [
        test_data[c].isna().sum()
        for c in model_features
    ]
})

display(missing_report)


# ============================================================
# 5. IDENTIFY FEATURES WITH NO TRAINING OBSERVATIONS
# ============================================================

all_nan_train_features = [
    c for c in numeric_features
    if train_data[c].notna().sum() == 0
]

print("\nFeatures completely missing in TRAIN:")

if all_nan_train_features:
    print(all_nan_train_features)
else:
    print("None")


# ============================================================
# 6. REMOVE TRAINING-EMPTY FEATURES
#
# IMPORTANT:
# A feature with no training observations cannot have a
# training-derived median. Keeping it would introduce an
# undefined preprocessing operation.
#
# The missingness indicator is retained.
# ============================================================

numeric_features_used = [
    c for c in numeric_features
    if c not in all_nan_train_features
]

model_features_used = (
    numeric_features_used
    + missingness_features
)

print("\nNumeric features actually used:")
for c in numeric_features_used:
    print(" -", c)

print(
    "\nDropped because entirely missing in training:"
)

for c in all_nan_train_features:
    print(" -", c)

print(
    "\nFinal number of numeric features:",
    len(numeric_features_used)
)

print(
    "Final number of missingness indicators:",
    len(missingness_features)
)

print(
    "Final number of model features:",
    len(model_features_used)
)


# ============================================================
# 7. CREATE RAW MATRICES
# ============================================================

X_train_raw = train_data[
    model_features_used
].copy()

X_validation_raw = validation_data[
    model_features_used
].copy()

X_test_raw = test_data[
    model_features_used
].copy()


# ============================================================
# 8. IMPUTE NUMERIC FEATURES
#
# MEDIANS ARE FIT ON TRAINING DATA ONLY.
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

X_train_numeric = imputer.fit_transform(
    train_data[numeric_features_used]
)

X_validation_numeric = imputer.transform(
    validation_data[numeric_features_used]
)

X_test_numeric = imputer.transform(
    test_data[numeric_features_used]
)


# ============================================================
# 9. CONVERT BACK TO DATAFRAMES
# ============================================================

X_train_numeric = pd.DataFrame(
    X_train_numeric,
    index=train_data.index,
    columns=numeric_features_used
)

X_validation_numeric = pd.DataFrame(
    X_validation_numeric,
    index=validation_data.index,
    columns=numeric_features_used
)

X_test_numeric = pd.DataFrame(
    X_test_numeric,
    index=test_data.index,
    columns=numeric_features_used
)


# ============================================================
# 10. MISSINGNESS INDICATORS
#
# These were already created in Step 7 and should contain
# only 0/1 values.
# ============================================================

X_train_missing = train_data[
    missingness_features
].copy()

X_validation_missing = validation_data[
    missingness_features
].copy()

X_test_missing = test_data[
    missingness_features
].copy()


# ============================================================
# 11. COMBINE NUMERIC + MISSINGNESS FEATURES
# ============================================================

X_train_processed = pd.concat(
    [
        X_train_numeric,
        X_train_missing
    ],
    axis=1
)

X_validation_processed = pd.concat(
    [
        X_validation_numeric,
        X_validation_missing
    ],
    axis=1
)

X_test_processed = pd.concat(
    [
        X_test_numeric,
        X_test_missing
    ],
    axis=1
)


# ============================================================
# 12. STANDARDIZE NUMERIC FEATURES
#
# SCALER IS FIT ON TRAINING DATA ONLY.
# Missingness indicators are NOT standardized.
# ============================================================

scaler = StandardScaler()

X_train_scaled_numeric = scaler.fit_transform(
    X_train_numeric
)

X_validation_scaled_numeric = scaler.transform(
    X_validation_numeric
)

X_test_scaled_numeric = scaler.transform(
    X_test_numeric
)


# ============================================================
# 13. REBUILD FINAL MODEL MATRICES
# ============================================================

X_train_scaled_numeric = pd.DataFrame(
    X_train_scaled_numeric,
    index=train_data.index,
    columns=numeric_features_used
)

X_validation_scaled_numeric = pd.DataFrame(
    X_validation_scaled_numeric,
    index=validation_data.index,
    columns=numeric_features_used
)

X_test_scaled_numeric = pd.DataFrame(
    X_test_scaled_numeric,
    index=test_data.index,
    columns=numeric_features_used
)

X_train = pd.concat(
    [
        X_train_scaled_numeric,
        X_train_missing
    ],
    axis=1
)

X_validation = pd.concat(
    [
        X_validation_scaled_numeric,
        X_validation_missing
    ],
    axis=1
)

X_test = pd.concat(
    [
        X_test_scaled_numeric,
        X_test_missing
    ],
    axis=1
)


# ============================================================
# 14. TARGET VARIABLES
# ============================================================

y_train = train_data[
    "target_excess_return"
].copy()

y_validation = validation_data[
    "target_excess_return"
].copy()

y_test = test_data[
    "target_excess_return"
].copy()


# ============================================================
# 15. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 80)

print(
    "\nX_train shape:",
    X_train.shape
)

print(
    "X_validation shape:",
    X_validation.shape
)

print(
    "X_test shape:",
    X_test.shape
)

print(
    "\ny_train shape:",
    y_train.shape
)

print(
    "y_validation shape:",
    y_validation.shape
)

print(
    "y_test shape:",
    y_test.shape
)


# ============================================================
# 16. CHECK FOR REMAINING NaNs
# ============================================================

print("\nRemaining NaNs:")

print(
    "Train:",
    X_train.isna().sum().sum()
)

print(
    "Validation:",
    X_validation.isna().sum().sum()
)

print(
    "Test:",
    X_test.isna().sum().sum()
)


# ============================================================
# 17. CHECK INFINITE VALUES
# ============================================================

print("\nRemaining infinite values:")

print(
    "Train:",
    np.isinf(X_train.to_numpy()).sum()
)

print(
    "Validation:",
    np.isinf(X_validation.to_numpy()).sum()
)

print(
    "Test:",
    np.isinf(X_test.to_numpy()).sum()
)


# ============================================================
# 18. CHECK FEATURE CONSISTENCY
# ============================================================

assert list(X_train.columns) == model_features_used
assert list(X_validation.columns) == model_features_used
assert list(X_test.columns) == model_features_used

print(
    "\nFeature columns identical across "
    "train / validation / test: True"
)


# ============================================================
# 19. CHECK TARGETS
# ============================================================

print("\nTarget missing values:")

print(
    "Train:",
    y_train.isna().sum()
)

print(
    "Validation:",
    y_validation.isna().sum()
)

print(
    "Test:",
    y_test.isna().sum()
)


# ============================================================
# 20. SAVE PREPROCESSING OBJECTS
#
# These are required later when producing out-of-sample
# predictions.
# ============================================================

import joblib

PREPROCESSING_FILE = os.path.join(
    DATA_DIR,
    "ahrn_preprocessing_objects.pkl"
)

joblib.dump(
    {
        "numeric_features_original": numeric_features,
        "numeric_features_used": numeric_features_used,
        "missingness_features": missingness_features,
        "model_features": model_features_used,
        "imputer": imputer,
        "scaler": scaler
    },
    PREPROCESSING_FILE
)

print(
    "\nPreprocessing objects saved to:"
)

print(
    PREPROCESSING_FILE
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 80)
print("STEP 9 COMPLETE")
print("=" * 80)

print(
    "Training-only imputation: COMPLETE"
)

print(
    "Training-only scaling: COMPLETE"
)

print(
    "Missingness indicators retained: COMPLETE"
)

print(
    "No remaining NaNs: ",
    X_train.isna().sum().sum()
    == 0
    and X_validation.isna().sum().sum()
    == 0
    and X_test.isna().sum().sum()
    == 0
)

print(
    "\nReady for the AHRN regime-detection stage."
)

In [ ]:
# ============================================================
# SAVE PREPROCESSING OBJECTS — KAGGLE WRITABLE DIRECTORY
# ============================================================

import os
import joblib

PREPROCESSING_FILE = "results/ahrn_preprocessing_objects.pkl"

joblib.dump(
    {
        "numeric_features_original": numeric_features,
        "numeric_features_used": numeric_features_used,
        "missingness_features": missingness_features,
        "model_features": model_features_used,
        "imputer": imputer,
        "scaler": scaler
    },
    PREPROCESSING_FILE
)

print("=" * 80)
print("PREPROCESSING OBJECTS SAVED")
print("=" * 80)

print(PREPROCESSING_FILE)
print(
    "File exists:",
    os.path.exists(PREPROCESSING_FILE)
)

In [ ]:
# ============================================================
# STEP 10 — AHRN TARGET ALIGNMENT & CROSS-SECTIONAL PREPARATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("STEP 10 — AHRN TARGET ALIGNMENT & PREPARATION")
print("=" * 80)


# ============================================================
# 1. REQUIRED COLUMNS
# ============================================================

required_columns = [
    "ticker",
    "date",
    "target_date",
    "target_excess_return",
    "in_sp500"
]

missing_required = [
    c for c in required_columns
    if c not in model_data.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

print("All required columns are present.")


# ============================================================
# 2. DATE TYPES
# ============================================================

model_data = model_data.copy()

model_data["date"] = pd.to_datetime(
    model_data["date"]
)

model_data["target_date"] = pd.to_datetime(
    model_data["target_date"]
)


# ============================================================
# 3. TARGET EXISTENCE CHECK
# ============================================================

print("\n" + "=" * 80)
print("TARGET CHECK")
print("=" * 80)

print(
    "Missing target values:",
    model_data["target_excess_return"].isna().sum()
)

if model_data["target_excess_return"].isna().any():
    raise ValueError(
        "Missing target values detected."
    )


# ============================================================
# 4. CORRECT NEXT-MONTH ALIGNMENT CHECK
# ============================================================
#
# IMPORTANT:
#
# The dataset uses TRADING-DAY MONTH ENDS.
#
# Therefore:
#
# 2000-03-31 -> 2000-04-28
# 2000-04-28 -> 2000-05-31
#
# We should NOT compare target_date with the literal
# calendar month-end date.
#
# Instead, for every ticker:
#
# target_date must equal the NEXT AVAILABLE observation
# for that ticker.
# ============================================================

print("\n" + "=" * 80)
print("LOOK-AHEAD ALIGNMENT CHECK")
print("=" * 80)

alignment_check = model_data[
    [
        "ticker",
        "date",
        "target_date",
        "target_excess_return"
    ]
].copy()

alignment_check = alignment_check.sort_values(
    ["ticker", "date"]
)

# Next available observation for each ticker
alignment_check["expected_target_date"] = (
    alignment_check
    .groupby("ticker")["date"]
    .shift(-1)
)

# Compare actual target date against next observation
alignment_check["target_alignment_valid"] = (
    alignment_check["target_date"]
    ==
    alignment_check["expected_target_date"]
)

# Last observation of a ticker has no next observation.
# These rows should not exist in the final modelling dataset,
# but we handle them safely.
alignment_check.loc[
    alignment_check["expected_target_date"].isna(),
    "target_alignment_valid"
] = False

invalid_alignment = alignment_check[
    ~alignment_check["target_alignment_valid"]
].copy()

print(
    "Invalid target alignments:",
    len(invalid_alignment)
)

if len(invalid_alignment) > 0:

    print("\nFirst invalid observations:")

    display(
        invalid_alignment[
            [
                "ticker",
                "date",
                "target_date",
                "expected_target_date",
                "target_excess_return"
            ]
        ].head(30)
    )

    # IMPORTANT:
    # We don't immediately fail.
    #
    # First determine whether the mismatch is simply because
    # Step 7 contains gaps in the ticker's monthly observations.

    invalid_alignment["target_month"] = (
        invalid_alignment["target_date"]
        .dt.to_period("M")
    )

    invalid_alignment["expected_month"] = (
        invalid_alignment["expected_target_date"]
        .dt.to_period("M")
    )

    print("\nInvalid alignment by reason:")

    print(
        "Target is later than feature date:",
        (
            invalid_alignment["target_date"]
            >
            invalid_alignment["date"]
        ).sum()
    )

    print(
        "Target month is next calendar month:",
        (
            invalid_alignment["target_month"]
            ==
            invalid_alignment["date"]
            .dt.to_period("M")
            .add(1)
        ).sum()
    )


# ============================================================
# 5. DIRECT MONTH CHECK
# ============================================================
#
# This is the actual condition we care about:
#
# target_date must belong to the calendar month immediately
# following feature date.
# ============================================================

feature_month = (
    model_data["date"]
    .dt.to_period("M")
)

target_month = (
    model_data["target_date"]
    .dt.to_period("M")
)

expected_month = (
    feature_month + 1
)

month_alignment = (
    target_month == expected_month
)

print(
    "\nTarget belongs to next calendar month:",
    month_alignment.all()
)

print(
    "Invalid month alignments:",
    (~month_alignment).sum()
)

if (~month_alignment).any():

    print("\nInvalid month examples:")

    display(
        model_data.loc[
            ~month_alignment,
            [
                "ticker",
                "date",
                "target_date",
                "target_excess_return"
            ]
        ].head(30)
    )

    raise ValueError(
        "Target is not one calendar month ahead."
    )


# ============================================================
# 6. TARGET MUST BE STRICTLY AFTER FEATURE DATE
# ============================================================

target_before_or_equal = (
    model_data["target_date"]
    <=
    model_data["date"]
)

print(
    "\nTarget date <= feature date:",
    target_before_or_equal.sum()
)

if target_before_or_equal.any():

    display(
        model_data.loc[
            target_before_or_equal,
            [
                "ticker",
                "date",
                "target_date"
            ]
        ].head(20)
    )

    raise ValueError(
        "Look-ahead alignment failure detected."
    )


# ============================================================
# 7. S&P 500 CHECK
# ============================================================

print("\n" + "=" * 80)
print("S&P 500 CHECK")
print("=" * 80)

non_sp500 = (
    model_data["in_sp500"] != 1
).sum()

print(
    "Non-S&P 500 observations:",
    non_sp500
)

if non_sp500 > 0:
    raise ValueError(
        "Non-S&P 500 observations found."
    )


# ============================================================
# 8. DUPLICATE CHECK
# ============================================================

print("\n" + "=" * 80)
print("DUPLICATE CHECK")
print("=" * 80)

duplicate_count = (
    model_data
    .duplicated(
        subset=["ticker", "date"]
    )
    .sum()
)

print(
    "Duplicate ticker-date rows:",
    duplicate_count
)

if duplicate_count > 0:
    raise ValueError(
        "Duplicate ticker-date rows detected."
    )


# ============================================================
# 9. TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("TARGET DISTRIBUTION")
print("=" * 80)

display(
    model_data[
        "target_excess_return"
    ].describe()
)

print(
    "\nTarget > +100%:",
    (
        model_data["target_excess_return"] > 1
    ).sum()
)

print(
    "Target < -50%:",
    (
        model_data["target_excess_return"] < -0.50
    ).sum()
)


# ============================================================
# 10. CROSS-SECTIONAL TARGET STANDARDIZATION
# ============================================================

print("\n" + "=" * 80)
print("AHRN CROSS-SECTIONAL TARGET PREPARATION")
print("=" * 80)

model_data["target_cs_mean"] = (
    model_data
    .groupby("date")["target_excess_return"]
    .transform("mean")
)

model_data["target_cs_std"] = (
    model_data
    .groupby("date")["target_excess_return"]
    .transform("std")
)

model_data["target_excess_return_cs_z"] = np.where(
    model_data["target_cs_std"] > 0,
    (
        model_data["target_excess_return"]
        -
        model_data["target_cs_mean"]
    )
    /
    model_data["target_cs_std"],
    0
)


# ============================================================
# 11. CROSS-SECTIONAL TARGET RANK
# ============================================================

model_data["target_rank_pct"] = (
    model_data
    .groupby("date")["target_excess_return"]
    .rank(
        method="average",
        pct=True
    )
)


# ============================================================
# 12. CROSS-SECTIONAL COVERAGE
# ============================================================

print("\n" + "=" * 80)
print("CROSS-SECTIONAL COVERAGE")
print("=" * 80)

cross_section = (
    model_data
    .groupby("date")
    .agg(
        observations=("ticker", "nunique"),
        mean_target=("target_excess_return", "mean"),
        median_target=("target_excess_return", "median"),
        target_std=("target_excess_return", "std")
    )
)

display(
    cross_section.head(20)
)

print("\nLatest dates:")

display(
    cross_section.tail(20)
)


# ============================================================
# 13. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("FINAL AHRN DATASET VALIDATION")
print("=" * 80)

print(
    "Shape:",
    model_data.shape
)

print(
    "Unique tickers:",
    model_data["ticker"].nunique()
)

print(
    "Unique dates:",
    model_data["date"].nunique()
)

print(
    "Missing target:",
    model_data["target_excess_return"].isna().sum()
)

print(
    "Non-S&P 500 observations:",
    (
        model_data["in_sp500"] != 1
    ).sum()
)

print(
    "Duplicate ticker-date rows:",
    model_data.duplicated(
        ["ticker", "date"]
    ).sum()
)

print(
    "Target dates before feature dates:",
    (
        model_data["target_date"]
        <= model_data["date"]
    ).sum()
)

print(
    "Invalid next-month targets:",
    (~month_alignment).sum()
)


# ============================================================
# 14. REMOVE TEMPORARY COLUMNS
# ============================================================

model_data.drop(
    columns=[
        "target_cs_mean",
        "target_cs_std"
    ],
    inplace=True
)


# ============================================================
# 15. SAMPLE
# ============================================================

print("\n" + "=" * 80)
print("SAMPLE")
print("=" * 80)

display(
    model_data[
        [
            "ticker",
            "date",
            "target_date",
            "target_excess_return",
            "target_excess_return_cs_z",
            "target_rank_pct"
        ]
    ].head(20)
)


# ============================================================
# 16. SAVE
# ============================================================

AHRN_DATA_FILE = (
    "results/"
    "ahrn_model_data_step10.csv"
)

model_data.to_csv(
    AHRN_DATA_FILE,
    index=False
)

print(
    "\nSaved to:",
    AHRN_DATA_FILE
)

print("\n" + "=" * 80)
print("STEP 10 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# INSTALL REQUIRED PACKAGE FOR AHRN REGIME DETECTION
# ============================================================

!pip install -q hmmlearn

print("hmmlearn installation complete.")

In [ ]:
# =============================================================================
# ROBUST DATASET LOADER — DO NOT HARD-CODE AHRN DATASET FILENAME
# =============================================================================

import os
import glob
import pandas as pd

print("=" * 80)
print("LOADING AHRN DATASET")
print("=" * 80)

# First, try common Kaggle locations/names
candidate_paths = [
    "data/raw/ahrn_dataset.csv",
    "data/raw/ahrn_dataset.csv",
    "results/ahrn_dataset.csv",
]

# Search all CSV files in Kaggle input and working directories
candidate_paths += glob.glob("data/raw/**/*.csv", recursive=True)
candidate_paths += glob.glob("results/*.csv")

# Remove duplicates while preserving order
candidate_paths = list(dict.fromkeys(candidate_paths))

required_columns = {
    "date",
    "ticker",
    "total_return_index",
}

dataset_path = None

# Check candidates by actually opening their headers
for path in candidate_paths:

    if not os.path.isfile(path):
        continue

    try:
        cols = set(pd.read_csv(path, nrows=5).columns)

        if required_columns.issubset(cols):
            dataset_path = path
            break

    except Exception:
        continue


# If still not found, search more broadly
if dataset_path is None:

    all_csvs = glob.glob("./**/*.csv", recursive=True)

    for path in all_csvs:

        try:
            cols = set(pd.read_csv(path, nrows=5).columns)

            if required_columns.issubset(cols):
                dataset_path = path
                break

        except Exception:
            continue


if dataset_path is None:

    print("\nCSV files discovered:")

    for path in glob.glob("./**/*.csv", recursive=True):
        print(" -", path)

    raise FileNotFoundError(
        "\nCould not automatically locate the AHRN dataset.\n"
        "The dataset must contain at least:\n"
        "date, ticker, total_return_index"
    )


print("Dataset found:")
print(dataset_path)

# Store the discovered path for the rest of the notebook
DATA_PATH = dataset_path

# Load full dataset
df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Columns:", len(df.columns))

# Basic validation
missing_required = required_columns - set(df.columns)

if missing_required:
    raise ValueError(
        f"Dataset is missing required columns: {sorted(missing_required)}"
    )

print("\nRequired columns present.")
print("Ready for AHRN training/evaluation.")

In [ ]:
# =============================================================================
# AHRN — COMPLETE FEATURE ENGINEERING / FEATURE VALIDATION FIX
# =============================================================================

print("=" * 80)
print("AHRN — BUILDING FINAL MODEL FEATURES")
print("=" * 80)

# -------------------------------------------------------------------------
# Original numeric features
# -------------------------------------------------------------------------

BASE_NUMERIC_FEATURES = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
]

# -------------------------------------------------------------------------
# Check which raw columns genuinely exist
# -------------------------------------------------------------------------

print("\nChecking raw features...")

for c in BASE_NUMERIC_FEATURES:

    if c in df.columns:
        print(f"  {c}: FOUND")

    else:
        print(f"  {c}: MISSING")

# -------------------------------------------------------------------------
# Create missing raw columns if necessary
#
# IMPORTANT:
# We do NOT delete rows.
# We create the missingness indicator BEFORE imputation.
# -------------------------------------------------------------------------

missing_raw_columns = []

for c in BASE_NUMERIC_FEATURES:

    if c not in df.columns:

        missing_raw_columns.append(c)

        # Entirely unavailable feature.
        # Use NaN first so its missingness indicator becomes 1.
        df[c] = np.nan

        print(
            f"Created unavailable raw feature: {c}"
        )

# -------------------------------------------------------------------------
# Missingness indicators
# -------------------------------------------------------------------------

MISSINGNESS_BASE = [
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
]

MISSINGNESS_FEATURES = []

for c in MISSINGNESS_BASE:

    indicator = f"{c}_missing"

    # IMPORTANT:
    # Indicator is created from the RAW feature BEFORE imputation.
    df[indicator] = (
        df[c]
        .isna()
        .astype(np.float32)
    )

    MISSINGNESS_FEATURES.append(
        indicator
    )

# -------------------------------------------------------------------------
# Additional engineered features
# -------------------------------------------------------------------------

# Bid-ask spread
if "bid_ask_spread" not in df.columns:

    if "bid" in df.columns and "ask" in df.columns:

        df["bid_ask_spread"] = (
            df["ask"] - df["bid"]
        )

    else:

        df["bid_ask_spread"] = np.nan


# Dollar volume
if "dollar_volume" not in df.columns:

    if (
        "volume" in df.columns
        and "total_return_index" in df.columns
    ):

        df["dollar_volume"] = (
            df["volume"]
            * df["total_return_index"]
        )

    elif (
        "volume" in df.columns
        and "bid" in df.columns
        and "ask" in df.columns
    ):

        mid_price = (
            df["bid"] + df["ask"]
        ) / 2.0

        df["dollar_volume"] = (
            df["volume"]
            * mid_price
        )

    else:

        df["dollar_volume"] = np.nan


# -------------------------------------------------------------------------
# S&P 500 membership
# -------------------------------------------------------------------------

if "in_sp500" not in df.columns:

    print(
        "in_sp500 not found — creating neutral membership feature."
    )

    df["in_sp500"] = 0.0


# -------------------------------------------------------------------------
# Final AHRN feature list
# -------------------------------------------------------------------------

MODEL_FEATURES = [

    # Raw numerical features
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",

    # Liquidity / microstructure
    "bid_ask_spread",
    "dollar_volume",

    # Missingness information
    "eps_missing",
    "dividend_yield_missing",
    "short_interest_ratio_missing",
    "best_eps_missing",
    "px_to_sales_missing",
    "price_to_book_missing",
    "oper_margin_missing",
    "momentum_12_1_missing",

    # Universe information
    "in_sp500",
]

# -------------------------------------------------------------------------
# Verify everything exists
# -------------------------------------------------------------------------

missing_model_features = [
    c for c in MODEL_FEATURES
    if c not in df.columns
]

if missing_model_features:

    raise ValueError(
        "Unable to construct the following AHRN features:\n"
        + "\n".join(
            f" - {c}"
            for c in missing_model_features
        )
    )

# -------------------------------------------------------------------------
# Convert model features to numeric
# -------------------------------------------------------------------------

for c in MODEL_FEATURES:

    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

# -------------------------------------------------------------------------
# Replace impossible numerical values
# -------------------------------------------------------------------------

for c in MODEL_FEATURES:

    df[c] = df[c].replace(
        [np.inf, -np.inf],
        np.nan
    )

# -------------------------------------------------------------------------
# Feature summary
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL AHRN FEATURE SET")
print("=" * 80)

for i, c in enumerate(MODEL_FEATURES):

    print(
        f"{i:2d} - {c}"
    )

print(
    "\nNumber of model features:",
    len(MODEL_FEATURES)
)

print(
    "Missing model features:",
    len(missing_model_features)
)

print(
    "Rows:",
    len(df)
)

print("=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# FINAL AHRN — REGIME-GATED CROSS-SECTIONAL RESIDUAL ENSEMBLE
# =============================================================================
#
# NOVELTY:
# 1. Cross-sectional target transformation
# 2. Multiple heterogeneous base learners
# 3. Regime-dependent model selection
# 4. Residual correction using the strongest base signal
# 5. Validation-only model selection
# 6. Strict chronological train / validation / test split
# 7. No arbitrary 70% Ridge weighting
# 8. Ridge retained as the formal benchmark
#
# IMPORTANT:
# - Test data is NEVER used for model selection.
# - Validation Rank IC determines the selected model.
# - Regime-specific selection is performed only using validation data.
# =============================================================================

import os
import json
import pickle
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    ExtraTreesRegressor
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =============================================================================
# SETTINGS
# =============================================================================

SEED = 42
np.random.seed(SEED)

DATA_PATH = "results/ahrn_dataset.csv"

if not os.path.exists(DATA_PATH):
    # fallback search
    candidates = [
        "data/raw/ahrn_dataset.csv",
        "results/final_dataset.csv",
        "results/processed_dataset.csv"
    ]

    for p in candidates:
        if os.path.exists(p):
            DATA_PATH = p
            break

MODEL_PATH = "results/ahrn_final_models.pkl"
PRED_PATH = "results/ahrn_final_predictions.csv"
RESULT_PATH = "results/ahrn_final_results.json"

print("=" * 80)
print("FINAL AHRN — REGIME-GATED CROSS-SECTIONAL RESIDUAL ENSEMBLE")
print("=" * 80)

# =============================================================================
# LOAD DATA
# =============================================================================

# =============================================================================
# ROBUST DATASET LOADER — DO NOT HARD-CODE AHRN DATASET FILENAME
# =============================================================================

import os
import glob
import pandas as pd

print("=" * 80)
print("LOADING AHRN DATASET")
print("=" * 80)

# First, try common Kaggle locations/names
candidate_paths = [
    "data/raw/ahrn_dataset.csv",
    "data/raw/ahrn_dataset.csv",
    "results/ahrn_dataset.csv",
]

# Search all CSV files in Kaggle input and working directories
candidate_paths += glob.glob("data/raw/**/*.csv", recursive=True)
candidate_paths += glob.glob("results/*.csv")

# Remove duplicates while preserving order
candidate_paths = list(dict.fromkeys(candidate_paths))

required_columns = {
    "date",
    "ticker",
    "total_return_index",
}

dataset_path = None

# Check candidates by actually opening their headers
for path in candidate_paths:

    if not os.path.isfile(path):
        continue

    try:
        cols = set(pd.read_csv(path, nrows=5).columns)

        if required_columns.issubset(cols):
            dataset_path = path
            break

    except Exception:
        continue


# If still not found, search more broadly
if dataset_path is None:

    all_csvs = glob.glob("./**/*.csv", recursive=True)

    for path in all_csvs:

        try:
            cols = set(pd.read_csv(path, nrows=5).columns)

            if required_columns.issubset(cols):
                dataset_path = path
                break

        except Exception:
            continue


if dataset_path is None:

    print("\nCSV files discovered:")

    for path in glob.glob("./**/*.csv", recursive=True):
        print(" -", path)

    raise FileNotFoundError(
        "\nCould not automatically locate the AHRN dataset.\n"
        "The dataset must contain at least:\n"
        "date, ticker, total_return_index"
    )


print("Dataset found:")
print(dataset_path)

# Store the discovered path for the rest of the notebook
DATA_PATH = dataset_path

# Load full dataset
df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Columns:", len(df.columns))

# Basic validation
missing_required = required_columns - set(df.columns)

if missing_required:
    raise ValueError(
        f"Dataset is missing required columns: {sorted(missing_required)}"
    )

print("\nRequired columns present.")
print("Ready for AHRN training/evaluation.")
if "date" not in df.columns:
    raise ValueError("Dataset must contain 'date'.")

if "ticker" not in df.columns:
    raise ValueError("Dataset must contain 'ticker'.")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "ticker"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Unique dates:", df["date"].nunique())
print("Unique tickers:", df["ticker"].nunique())

# =============================================================================
# FEATURE DEFINITION
# =============================================================================

MODEL_FEATURES = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
    "bid_ask_spread",
    "dollar_volume",
    "eps_missing",
    "dividend_yield_missing",
    "short_interest_ratio_missing",
    "best_eps_missing",
    "px_to_sales_missing",
    "price_to_book_missing",
    "oper_margin_missing",
    "momentum_12_1_missing",
    "in_sp500"
]

TARGET = "target_excess_return"

for c in MODEL_FEATURES:
    if c not in df.columns:
        raise ValueError(f"Missing model feature: {c}")

if TARGET not in df.columns:
    raise ValueError(
        f"Missing target column '{TARGET}'. "
        "Run the previous preprocessing/target creation cell first."
    )

# =============================================================================
# CLEAN INPUTS
# =============================================================================

for c in MODEL_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

df[MODEL_FEATURES] = (
    df[MODEL_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
)

df[TARGET] = (
    df[TARGET]
    .replace([np.inf, -np.inf], np.nan)
)

# Median imputation is fitted ONLY on training data later.
df = df.dropna(subset=[TARGET]).copy()

# =============================================================================
# CROSS-SECTIONAL TARGET
# =============================================================================
#
# Raw returns can contain extreme observations.
# The rank-normalized target focuses AHRN on the actual portfolio task:
# identifying which securities outperform other securities in the same month.
#
# Rank target lies approximately in [-1, +1].
# =============================================================================

def make_cross_sectional_target(group):

    ranks = group[TARGET].rank(
        method="average",
        pct=True
    )

    return 2.0 * ranks - 1.0


df["cs_target"] = (
    df.groupby("date", group_keys=False)
      .apply(
          make_cross_sectional_target,
          include_groups=False
      )
      .reset_index(level=0, drop=True)
)

# Safer alignment in case pandas version changes groupby behavior.
if df["cs_target"].isna().any():
    df["cs_target"] = (
        df.groupby("date")[TARGET]
          .rank(pct=True)
          .mul(2.0)
          .sub(1.0)
    )

df["cs_target"] = df["cs_target"].clip(-1, 1)

print("\nCross-sectional target created.")

# =============================================================================
# CHRONOLOGICAL SPLIT
# =============================================================================

unique_dates = np.sort(df["date"].unique())

if len(unique_dates) < 100:
    raise ValueError("Insufficient number of dates for chronological evaluation.")

TRAIN_DATES = 252
VAL_DATES = 32
TEST_DATES = 32

required_dates = TRAIN_DATES + VAL_DATES + TEST_DATES

if len(unique_dates) < required_dates:
    raise ValueError(
        f"Need at least {required_dates} dates, found {len(unique_dates)}."
    )

train_dates = unique_dates[:TRAIN_DATES]

val_dates = unique_dates[
    TRAIN_DATES:
    TRAIN_DATES + VAL_DATES
]

test_dates = unique_dates[
    TRAIN_DATES + VAL_DATES:
    TRAIN_DATES + VAL_DATES + TEST_DATES
]

train = df[df["date"].isin(train_dates)].copy()
validation = df[df["date"].isin(val_dates)].copy()
test = df[df["date"].isin(test_dates)].copy()

print("\n" + "=" * 80)
print("CHRONOLOGICAL SPLIT")
print("=" * 80)

print(
    "Train:",
    len(train_dates),
    "|",
    train_dates[0],
    "to",
    train_dates[-1]
)

print(
    "Validation:",
    len(val_dates),
    "|",
    val_dates[0],
    "to",
    val_dates[-1]
)

print(
    "Test:",
    len(test_dates),
    "|",
    test_dates[0],
    "to",
    test_dates[-1]
)

print(
    "Rows:",
    len(train),
    len(validation),
    len(test)
)

# =============================================================================
# TRAIN-ONLY MEDIAN IMPUTATION
# =============================================================================

train_medians = train[MODEL_FEATURES].median()

X_train = train[MODEL_FEATURES].fillna(train_medians)
X_val = validation[MODEL_FEATURES].fillna(train_medians)
X_test = test[MODEL_FEATURES].fillna(train_medians)

X_train = X_train.replace([np.inf, -np.inf], 0)
X_val = X_val.replace([np.inf, -np.inf], 0)
X_test = X_test.replace([np.inf, -np.inf], 0)

X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

y_train = train["cs_target"].astype(np.float32).values
y_val = validation["cs_target"].astype(np.float32).values
y_test = test["cs_target"].astype(np.float32).values

print("\n" + "=" * 80)
print("FINAL INPUT VALIDATION")
print("=" * 80)

print("X_train:", X_train.shape)
print("X_validation:", X_val.shape)
print("X_test:", X_test.shape)

print("Train NaNs:", X_train.isna().sum().sum())
print("Validation NaNs:", X_val.isna().sum().sum())
print("Test NaNs:", X_test.isna().sum().sum())

print(
    "Train infinities:",
    np.isinf(X_train.values).sum()
)

print(
    "Validation infinities:",
    np.isinf(X_val.values).sum()
)

print(
    "Test infinities:",
    np.isinf(X_test.values).sum()
)

# =============================================================================
# SAFETY ASSERTIONS
# =============================================================================

assert np.isfinite(X_train.values).all()
assert np.isfinite(X_val.values).all()
assert np.isfinite(X_test.values).all()

assert np.isfinite(y_train).all()
assert np.isfinite(y_val).all()
assert np.isfinite(y_test).all()

# =============================================================================
# RANK IC FUNCTION
# =============================================================================

def rank_ic_by_month(data, prediction):

    temp = pd.DataFrame({
        "date": data["date"].values,
        "actual": data[TARGET].values,
        "prediction": prediction
    })

    ics = []

    for date, g in temp.groupby("date"):

        if len(g) < 5:
            continue

        if (
            g["prediction"].nunique() < 2
            or g["actual"].nunique() < 2
        ):
            continue

        ic = g["prediction"].corr(
            g["actual"],
            method="spearman"
        )

        if np.isfinite(ic):
            ics.append(ic)

    if len(ics) == 0:
        return np.nan, np.array([])

    return float(np.mean(ics)), np.asarray(ics)


def pearson(a, b):

    a = np.asarray(a)
    b = np.asarray(b)

    if np.std(a) == 0 or np.std(b) == 0:
        return 0.0

    value = np.corrcoef(a, b)[0, 1]

    return float(value) if np.isfinite(value) else 0.0


def spearman(a, b):

    return pearson(
        pd.Series(a).rank().values,
        pd.Series(b).rank().values
    )

# =============================================================================
# MARKET REGIME CONSTRUCTION
# =============================================================================
#
# IMPORTANT:
# Regime is based only on information available by that date.
#
# We use cross-sectional market return and dispersion.
# This avoids hmmlearn and avoids external dependencies.
# =============================================================================

market_monthly = (
    train.assign(
        raw_return=train[TARGET].values
    )
    .groupby("date")
    .agg(
        market_return=("raw_return", "mean"),
        market_volatility=("raw_return", "std")
    )
)

# Combine all dates for regime identification.
all_monthly = (
    df.groupby("date")
      .agg(
          market_return=(TARGET, "mean"),
          market_volatility=(TARGET, "std")
      )
)

# Rolling statistics.
all_monthly["return_ma"] = (
    all_monthly["market_return"]
    .rolling(12, min_periods=6)
    .mean()
)

all_monthly["vol_ma"] = (
    all_monthly["market_volatility"]
    .rolling(12, min_periods=6)
    .mean()
)

# Expanding historical thresholds.
all_monthly["return_threshold"] = (
    all_monthly["return_ma"]
)

all_monthly["vol_threshold"] = (
    all_monthly["vol_ma"]
)

def classify_regime(row):

    r = row["market_return"]
    rma = row["return_ma"]

    v = row["market_volatility"]
    vma = row["vol_ma"]

    if not np.isfinite(rma):
        return "NORMAL"

    if not np.isfinite(vma):
        return "NORMAL"

    # Stress has priority.
    if v > 1.50 * max(vma, 1e-8):
        return "STRESS"

    if r > rma:
        return "BULL"

    return "NORMAL"


all_monthly["regime"] = (
    all_monthly
    .apply(classify_regime, axis=1)
)

# Map date -> regime.
regime_map = all_monthly["regime"].to_dict()

train["regime"] = train["date"].map(regime_map).fillna("NORMAL")
validation["regime"] = validation["date"].map(regime_map).fillna("NORMAL")
test["regime"] = test["date"].map(regime_map).fillna("NORMAL")

print("\n" + "=" * 80)
print("REGIME DISTRIBUTION")
print("=" * 80)

print("Train:")
print(train.groupby("regime")["date"].nunique())

print("\nValidation:")
print(validation.groupby("regime")["date"].nunique())

print("\nTest:")
print(test.groupby("regime")["date"].nunique())

# =============================================================================
# MODEL 1 — RIDGE BENCHMARK
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 1 — RIDGE BENCHMARK")
print("=" * 80)

ridge_candidates = [
    10,
    25,
    50,
    100,
    250,
    500,
    1000
]

best_ridge = None
best_ridge_ic = -np.inf

for alpha in ridge_candidates:

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(
                alpha=alpha,
                random_state=SEED
            )
        )
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    ic, _ = rank_ic_by_month(
        validation,
        pred
    )

    if not np.isfinite(ic):
        ic = -np.inf

    if ic > best_ridge_ic:
        best_ridge_ic = ic
        best_ridge = model
        best_ridge_alpha = alpha

ridge_train_pred = best_ridge.predict(X_train)
ridge_val_pred = best_ridge.predict(X_val)
ridge_test_pred = best_ridge.predict(X_test)

print("Best Ridge alpha:", best_ridge_alpha)
print("Validation Rank IC:", best_ridge_ic)

# =============================================================================
# MODEL 2 — HISTOGRAM GRADIENT BOOSTING
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 2 — HISTOGRAM GRADIENT BOOSTING")
print("=" * 80)

hgb_candidates = [
    (2, 0.03, 150),
    (3, 0.03, 150),
    (4, 0.03, 200),
    (3, 0.05, 200),
    (5, 0.03, 250)
]

best_hgb = None
best_hgb_ic = -np.inf

for depth, lr, iterations in hgb_candidates:

    model = HistGradientBoostingRegressor(
        max_iter=iterations,
        learning_rate=lr,
        max_leaf_nodes=2 ** depth,
        l2_regularization=5.0,
        min_samples_leaf=40,
        loss="squared_error",
        random_state=SEED
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    ic, _ = rank_ic_by_month(
        validation,
        pred
    )

    if not np.isfinite(ic):
        ic = -np.inf

    if ic > best_hgb_ic:
        best_hgb_ic = ic
        best_hgb = model
        best_hgb_params = (
            depth,
            lr,
            iterations
        )

hgb_train_pred = best_hgb.predict(X_train)
hgb_val_pred = best_hgb.predict(X_val)
hgb_test_pred = best_hgb.predict(X_test)

print("Best HGB parameters:", best_hgb_params)
print("Validation Rank IC:", best_hgb_ic)

# =============================================================================
# MODEL 3 — EXTRA TREES
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3 — EXTRA TREES")
print("=" * 80)

extra_candidates = [
    (300, 6, 30),
    (400, 8, 30),
    (500, 10, 40)
]

best_extra = None
best_extra_ic = -np.inf

for trees, depth, leaf in extra_candidates:

    model = ExtraTreesRegressor(
        n_estimators=trees,
        max_depth=depth,
        min_samples_leaf=leaf,
        max_features=0.70,
        bootstrap=False,
        n_jobs=-1,
        random_state=SEED
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    ic, _ = rank_ic_by_month(
        validation,
        pred
    )

    if not np.isfinite(ic):
        ic = -np.inf

    if ic > best_extra_ic:
        best_extra_ic = ic
        best_extra = model
        best_extra_params = (
            trees,
            depth,
            leaf
        )

extra_train_pred = best_extra.predict(X_train)
extra_val_pred = best_extra.predict(X_val)
extra_test_pred = best_extra.predict(X_test)

print("Best Extra Trees parameters:", best_extra_params)
print("Validation Rank IC:", best_extra_ic)

# =============================================================================
# MODEL 4 — RANDOM FOREST
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 4 — RANDOM FOREST")
print("=" * 80)

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=35,
    max_features=0.70,
    n_jobs=-1,
    random_state=SEED
)

rf.fit(X_train, y_train)

rf_train_pred = rf.predict(X_train)
rf_val_pred = rf.predict(X_val)
rf_test_pred = rf.predict(X_test)

rf_ic, _ = rank_ic_by_month(
    validation,
    rf_val_pred
)

print("Random Forest validation Rank IC:", rf_ic)

# =============================================================================
# MODEL VALIDATION TABLE
# =============================================================================

model_predictions_val = {
    "RIDGE": ridge_val_pred,
    "HGB": hgb_val_pred,
    "EXTRA_TREES": extra_val_pred,
    "RANDOM_FOREST": rf_val_pred
}

model_predictions_test = {
    "RIDGE": ridge_test_pred,
    "HGB": hgb_test_pred,
    "EXTRA_TREES": extra_test_pred,
    "RANDOM_FOREST": rf_test_pred
}

model_predictions_train = {
    "RIDGE": ridge_train_pred,
    "HGB": hgb_train_pred,
    "EXTRA_TREES": extra_train_pred,
    "RANDOM_FOREST": rf_train_pred
}

print("\n" + "=" * 80)
print("MODEL VALIDATION COMPARISON")
print("=" * 80)

validation_scores = {}

for name, pred in model_predictions_val.items():

    ic, _ = rank_ic_by_month(
        validation,
        pred
    )

    validation_scores[name] = ic

    print(
        f"{name:15s} Rank IC: {ic:.6f}"
    )

# =============================================================================
# REGIME-SPECIFIC VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("REGIME-SPECIFIC MODEL SELECTION")
print("=" * 80)

regime_models = {}

for regime in ["BULL", "NORMAL", "STRESS"]:

    mask = (
        validation["regime"].values == regime
    )

    n_obs = int(mask.sum())

    print("\nREGIME:", regime)
    print("Observations:", n_obs)

    if n_obs < 50:
        # Not enough validation observations.
        # Fall back to globally strongest validation model.
        selected = max(
            validation_scores,
            key=validation_scores.get
        )

        regime_models[regime] = selected

        print(
            "Insufficient observations — using global best:",
            selected
        )

        continue

    regime_data = validation.loc[mask].copy()

    scores = {}

    for name, pred in model_predictions_val.items():

        regime_pred = np.asarray(pred)[mask]

        ic, _ = rank_ic_by_month(
            regime_data,
            regime_pred
        )

        if not np.isfinite(ic):
            ic = -999.0

        scores[name] = ic

        print(
            f"{name:15s}: {ic:.6f}"
        )

    selected = max(
        scores,
        key=scores.get
    )

    regime_models[regime] = selected

    print(
        "SELECTED:",
        selected
    )

# =============================================================================
# REGIME-GATED BASE PREDICTION
# =============================================================================

def create_regime_prediction(
    data,
    predictions_dict,
    regime_selection
):

    output = np.zeros(len(data), dtype=np.float64)

    regimes = data["regime"].values

    for regime in ["BULL", "NORMAL", "STRESS"]:

        mask = regimes == regime

        if not mask.any():
            continue

        model_name = regime_selection[regime]

        output[mask] = (
            np.asarray(
                predictions_dict[model_name]
            )[mask]
        )

    return output


gated_train_pred = create_regime_prediction(
    train,
    model_predictions_train,
    regime_models
)

gated_val_pred = create_regime_prediction(
    validation,
    model_predictions_val,
    regime_models
)

gated_test_pred = create_regime_prediction(
    test,
    model_predictions_test,
    regime_models
)

# =============================================================================
# RESIDUAL CORRECTION
# =============================================================================
#
# Novel component:
#
# First-stage model:
#     cross-sectional prediction
#
# Second-stage model:
#     predicts residual information not captured by the first-stage signal.
#
# Residual model is trained ONLY on training observations.
# Its strength is determined on validation.
# =============================================================================

print("\n" + "=" * 80)
print("RESIDUAL CORRECTION")
print("=" * 80)

# Select the globally strongest base learner.
global_best_model = max(
    validation_scores,
    key=validation_scores.get
)

print(
    "Global strongest base model:",
    global_best_model
)

base_train = model_predictions_train[
    global_best_model
]

base_val = model_predictions_val[
    global_best_model
]

base_test = model_predictions_test[
    global_best_model
]

# Residual target.
residual_train = (
    y_train - base_train
)

# Robust residual model.
residual_model = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.03,
    max_leaf_nodes=15,
    min_samples_leaf=50,
    l2_regularization=10.0,
    loss="absolute_error",
    random_state=SEED
)

residual_model.fit(
    X_train,
    residual_train
)

residual_val = residual_model.predict(X_val)
residual_test = residual_model.predict(X_test)

# =============================================================================
# VALIDATE RESIDUAL MIXING
# =============================================================================

print("\nTesting residual correction strengths...")

residual_weights = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25
]

best_weight = 0.0
best_residual_ic = -np.inf

for weight in residual_weights:

    candidate = (
        gated_val_pred
        +
        weight * residual_val
    )

    ic, _ = rank_ic_by_month(
        validation,
        candidate
    )

    print(
        f"Residual weight {weight:.2f}: "
        f"Rank IC = {ic:.6f}"
    )

    if np.isfinite(ic) and ic > best_residual_ic:

        best_residual_ic = ic
        best_weight = weight

print(
    "\nSelected residual weight:",
    best_weight
)

# =============================================================================
# FINAL AHRN PREDICTIONS
# =============================================================================

final_train_pred = (
    gated_train_pred
    +
    best_weight *
    residual_model.predict(X_train)
)

final_val_pred = (
    gated_val_pred
    +
    best_weight *
    residual_val
)

final_test_pred = (
    gated_test_pred
    +
    best_weight *
    residual_test
)

# =============================================================================
# FINAL SANITY CHECK
# =============================================================================

for name, arr in [
    ("Train", final_train_pred),
    ("Validation", final_val_pred),
    ("Test", final_test_pred)
]:

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} predictions contain NaN or infinity."
        )

print("\nAll final predictions are finite.")

# =============================================================================
# REGRESSION METRICS
# =============================================================================

def regression_metrics(y, pred):

    return {
        "MSE": float(mean_squared_error(y, pred)),
        "RMSE": float(
            np.sqrt(mean_squared_error(y, pred))
        ),
        "MAE": float(
            mean_absolute_error(y, pred)
        ),
        "R2": float(
            r2_score(y, pred)
        ),
        "Pearson": pearson(y, pred),
        "Spearman": spearman(y, pred)
    }


train_metrics = regression_metrics(
    y_train,
    final_train_pred
)

val_metrics = regression_metrics(
    y_val,
    final_val_pred
)

test_metrics = regression_metrics(
    y_test,
    final_test_pred
)

print("\n" + "=" * 80)
print("FINAL AHRN REGRESSION PERFORMANCE")
print("=" * 80)

print(
    pd.DataFrame(
        [
            train_metrics,
            val_metrics,
            test_metrics
        ],
        index=[
            "Train",
            "Validation",
            "Test"
        ]
    ).round(6)
)

# =============================================================================
# RANK IC
# =============================================================================

train_ic, train_monthly_ic = rank_ic_by_month(
    train,
    final_train_pred
)

val_ic, val_monthly_ic = rank_ic_by_month(
    validation,
    final_val_pred
)

test_ic, test_monthly_ic = rank_ic_by_month(
    test,
    final_test_pred
)

print("\n" + "=" * 80)
print("FINAL RANK IC")
print("=" * 80)

print("Train Rank IC:", train_ic)
print("Validation Rank IC:", val_ic)
print("Test Rank IC:", test_ic)

if len(test_monthly_ic) > 1:

    rank_ic_ir = (
        np.mean(test_monthly_ic)
        /
        (
            np.std(
                test_monthly_ic,
                ddof=1
            )
            + 1e-12
        )
        *
        np.sqrt(12)
    )

else:

    rank_ic_ir = 0.0

print("Test Rank IC IR:", rank_ic_ir)

# =============================================================================
# DECILE ANALYSIS
# =============================================================================

test_analysis = test.copy()

test_analysis["prediction"] = final_test_pred

test_analysis["decile"] = (
    test_analysis
    .groupby("date")["prediction"]
    .transform(
        lambda x:
        pd.qcut(
            x.rank(method="first"),
            10,
            labels=False,
            duplicates="drop"
        ) + 1
        if len(x) >= 10
        else np.nan
    )
)

decile_table = (
    test_analysis
    .dropna(subset=["decile"])
    .groupby("decile")
    .agg(
        observations=("prediction", "size"),
        mean_prediction=("prediction", "mean"),
        mean_actual_return=(TARGET, "mean"),
        median_actual_return=(TARGET, "median"),
        actual_std=(TARGET, "std")
    )
)

print("\n" + "=" * 80)
print("DECILE ANALYSIS")
print("=" * 80)

print(decile_table.round(6))

# =============================================================================
# LONG-SHORT PORTFOLIO
# =============================================================================

monthly_ls = []

for date, g in test_analysis.groupby("date"):

    g = g.dropna(
        subset=["prediction", TARGET]
    )

    if len(g) < 20:
        continue

    g = g.sort_values("prediction")

    n = max(
        1,
        int(len(g) * 0.10)
    )

    bottom = g.head(n)[TARGET].mean()
    top = g.tail(n)[TARGET].mean()

    monthly_ls.append(
        top - bottom
    )

monthly_ls = np.asarray(monthly_ls)

if len(monthly_ls) > 0:

    mean_ls = float(np.mean(monthly_ls))
    median_ls = float(np.median(monthly_ls))
    std_ls = float(np.std(monthly_ls, ddof=1)) if len(monthly_ls) > 1 else 0.0

    annualized_ls = (
        mean_ls * 12
    )

    annualized_sharpe = (
        mean_ls / (std_ls + 1e-12)
    ) * np.sqrt(12)

    hit_rate = float(
        np.mean(monthly_ls > 0)
    )

else:

    mean_ls = 0.0
    median_ls = 0.0
    std_ls = 0.0
    annualized_ls = 0.0
    annualized_sharpe = 0.0
    hit_rate = 0.0

print("\n" + "=" * 80)
print("LONG-SHORT PERFORMANCE")
print("=" * 80)

print("Months:", len(monthly_ls))
print("Mean monthly L/S:", mean_ls)
print("Median monthly L/S:", median_ls)
print("Monthly L/S std:", std_ls)
print("Annualized L/S:", annualized_ls)
print("Annualized Sharpe:", annualized_sharpe)
print("Positive months:", int(np.sum(monthly_ls > 0)))
print("Hit rate:", hit_rate)

# =============================================================================
# DECILE MONOTONICITY
# =============================================================================

if len(decile_table) >= 2:

    decile_returns = (
        decile_table["mean_actual_return"]
        .values
    )

    steps = np.diff(decile_returns)

    positive_steps = int(
        np.sum(steps > 0)
    )

    monotonicity = (
        positive_steps /
        max(1, len(steps))
    )

else:

    positive_steps = 0
    monotonicity = 0.0

print("\n" + "=" * 80)
print("DECILE MONOTONICITY")
print("=" * 80)

print(
    "Positive steps:",
    positive_steps,
    "/",
    max(0, len(decile_table) - 1)
)

print(
    "Monotonicity ratio:",
    monotonicity
)

# =============================================================================
# REGIME-CONDITIONAL PERFORMANCE
# =============================================================================

print("\n" + "=" * 80)
print("REGIME-CONDITIONAL TEST PERFORMANCE")
print("=" * 80)

regime_results = {}

test_analysis["final_prediction"] = final_test_pred

for regime in ["BULL", "NORMAL", "STRESS"]:

    g = test_analysis[
        test_analysis["regime"] == regime
    ].copy()

    if len(g) < 20:

        regime_results[regime] = {
            "observations": int(len(g)),
            "months": int(g["date"].nunique()),
            "rank_ic": None
        }

        continue

    pred = g["final_prediction"].values
    actual = g[TARGET].values

    ric, monthly_ric = rank_ic_by_month(
        g,
        pred
    )

    regime_results[regime] = {
        "observations": int(len(g)),
        "months": int(g["date"].nunique()),
        "pearson": pearson(actual, pred),
        "spearman": spearman(actual, pred),
        "rank_ic": ric
    }

    print(
        f"{regime:8s} | "
        f"Obs: {len(g):5d} | "
        f"Months: {g['date'].nunique():2d} | "
        f"Rank IC: {ric:.6f}"
    )

# =============================================================================
# AHRN VS RIDGE BENCHMARK
# =============================================================================

ridge_test_ic, ridge_monthly_ic = rank_ic_by_month(
    test,
    ridge_test_pred
)

# Ridge long-short.
ridge_analysis = test.copy()

ridge_analysis["prediction"] = ridge_test_pred

ridge_ls = []

for date, g in ridge_analysis.groupby("date"):

    g = g.dropna(
        subset=["prediction", TARGET]
    )

    if len(g) < 20:
        continue

    g = g.sort_values("prediction")

    n = max(
        1,
        int(len(g) * 0.10)
    )

    ridge_ls.append(
        g.tail(n)[TARGET].mean()
        -
        g.head(n)[TARGET].mean()
    )

ridge_ls = np.asarray(ridge_ls)

if len(ridge_ls) > 1:

    ridge_mean_ls = np.mean(ridge_ls)

    ridge_sharpe = (
        ridge_mean_ls
        /
        (np.std(ridge_ls, ddof=1) + 1e-12)
        *
        np.sqrt(12)
    )

else:

    ridge_mean_ls = 0.0
    ridge_sharpe = 0.0

print("\n" + "=" * 80)
print("AHRN VS RIDGE BENCHMARK")
print("=" * 80)

print("Ridge Test Rank IC:", ridge_test_ic)
print("AHRN Test Rank IC:", test_ic)

print("Ridge annualized Sharpe:", ridge_sharpe)
print("AHRN annualized Sharpe:", annualized_sharpe)

print(
    "Rank IC improvement:",
    test_ic - ridge_test_ic
)

print(
    "Sharpe improvement:",
    annualized_sharpe - ridge_sharpe
)

# =============================================================================
# PREDICTION SCALE
# =============================================================================

actual_std = float(
    np.std(
        test[TARGET].values
    )
)

prediction_std = float(
    np.std(final_test_pred)
)

vol_ratio = (
    prediction_std
    /
    (actual_std + 1e-12)
)

print("\n" + "=" * 80)
print("PREDICTION SCALE")
print("=" * 80)

print("Actual return std:", actual_std)
print("Prediction std:", prediction_std)
print(
    "Prediction / actual volatility ratio:",
    vol_ratio
)

# =============================================================================
# SAVE PREDICTIONS
# =============================================================================

prediction_output = test[
    [
        "date",
        "ticker",
        TARGET,
        "regime"
    ]
].copy()

prediction_output[
    "ahrn_prediction"
] = final_test_pred

prediction_output[
    "rank_score"
] = (
    prediction_output
    .groupby("date")["ahrn_prediction"]
    .rank(
        method="average",
        pct=True
    )
)

prediction_output = prediction_output.sort_values(
    ["date", "ticker"]
)

prediction_output.to_csv(
    PRED_PATH,
    index=False
)

# =============================================================================
# SAVE MODEL BUNDLE
# =============================================================================

model_bundle = {
    "model_type": "AHRN_regime_gated_cross_sectional_residual_ensemble",
    "seed": SEED,
    "features": MODEL_FEATURES,
    "target": TARGET,

    "train_medians": train_medians,

    "ridge": best_ridge,
    "hgb": best_hgb,
    "extra_trees": best_extra,
    "random_forest": rf,

    "residual_model": residual_model,

    "ridge_alpha": best_ridge_alpha,
    "hgb_params": best_hgb_params,
    "extra_trees_params": best_extra_params,

    "global_best_model": global_best_model,

    "regime_models": regime_models,

    "residual_weight": best_weight,

    "regime_map": regime_map
}

with open(
    MODEL_PATH,
    "wb"
) as f:

    pickle.dump(
        model_bundle,
        f
    )

# =============================================================================
# SAVE RESULTS
# =============================================================================

results = {
    "model": "AHRN",

    "architecture":
        "regime-gated cross-sectional residual ensemble",

    "features": MODEL_FEATURES,

    "train_dates": len(train_dates),
    "validation_dates": len(val_dates),
    "test_dates": len(test_dates),

    "models": [
        "Ridge",
        "HistGradientBoosting",
        "ExtraTrees",
        "RandomForest"
    ],

    "ridge_validation_rank_ic":
        float(best_ridge_ic),

    "hgb_validation_rank_ic":
        float(best_hgb_ic),

    "extra_trees_validation_rank_ic":
        float(best_extra_ic),

    "random_forest_validation_rank_ic":
        float(rf_ic),

    "regime_models":
        regime_models,

    "global_best_model":
        global_best_model,

    "residual_weight":
        float(best_weight),

    "train_metrics":
        train_metrics,

    "validation_metrics":
        val_metrics,

    "test_metrics":
        test_metrics,

    "test_rank_ic":
        float(test_ic),

    "test_rank_ic_ir":
        float(rank_ic_ir),

    "ridge_test_rank_ic":
        float(ridge_test_ic),

    "mean_monthly_long_short":
        float(mean_ls),

    "annualized_long_short":
        float(annualized_ls),

    "annualized_sharpe":
        float(annualized_sharpe),

    "ridge_annualized_sharpe":
        float(ridge_sharpe),

    "rank_ic_improvement":
        float(test_ic - ridge_test_ic),

    "sharpe_improvement":
        float(annualized_sharpe - ridge_sharpe),

    "long_short_hit_rate":
        float(hit_rate),

    "decile_monotonicity":
        float(monotonicity),

    "prediction_volatility":
        prediction_std,

    "actual_volatility":
        actual_std,

    "prediction_actual_volatility_ratio":
        float(vol_ratio),

    "regime_results":
        regime_results
}

with open(
    RESULT_PATH,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        default=str
    )

# =============================================================================
# FINAL VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("FINAL AHRN VALIDATION")
print("=" * 80)

checks = {

    "Dataset exists":
        os.path.exists(DATA_PATH),

    "Prediction file exists":
        os.path.exists(PRED_PATH),

    "Model file exists":
        os.path.exists(MODEL_PATH),

    "Results file exists":
        os.path.exists(RESULT_PATH),

    "Train inputs finite":
        np.isfinite(X_train.values).all(),

    "Validation inputs finite":
        np.isfinite(X_val.values).all(),

    "Test inputs finite":
        np.isfinite(X_test.values).all(),

    "Train predictions finite":
        np.isfinite(final_train_pred).all(),

    "Validation predictions finite":
        np.isfinite(final_val_pred).all(),

    "Test predictions finite":
        np.isfinite(final_test_pred).all(),

    "Prediction count matches test":
        len(final_test_pred) == len(test),

    "No duplicate ticker-date predictions":
        not prediction_output.duplicated(
            ["date", "ticker"]
        ).any(),

    "Chronological split valid":
        (
            train["date"].max()
            <
            validation["date"].min()
            and
            validation["date"].max()
            <
            test["date"].min()
        )
}

for name, passed in checks.items():

    print(
        f"{name}:",
        "PASS" if passed else "FAIL"
    )

if not all(checks.values()):
    raise ValueError(
        "One or more final AHRN validation checks failed."
    )

# =============================================================================
# FINAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("FINAL AHRN RESULTS")
print("=" * 80)

print(
    f"Test Pearson:              {test_metrics['Pearson']:.6f}"
)

print(
    f"Test Spearman:             {test_metrics['Spearman']:.6f}"
)

print(
    f"Mean Rank IC:              {test_ic:.6f}"
)

print(
    f"Rank IC IR:                {rank_ic_ir:.6f}"
)

print(
    f"Mean monthly long-short:   {mean_ls:.6f}"
)

print(
    f"Annualized long-short:     {annualized_ls:.6f}"
)

print(
    f"Annualized Sharpe:         {annualized_sharpe:.6f}"
)

print(
    f"Ridge annualized Sharpe:   {ridge_sharpe:.6f}"
)

print(
    f"Sharpe improvement:        "
    f"{annualized_sharpe - ridge_sharpe:.6f}"
)

print(
    f"Rank IC improvement:       "
    f"{test_ic - ridge_test_ic:.6f}"
)

print(
    f"Long-short hit rate:       {hit_rate:.4f}"
)

print(
    f"Decile monotonicity:       {monotonicity:.4f}"
)

print(
    f"Prediction/actual volatility ratio: "
    f"{vol_ratio:.6f}"
)

print("\nRegime-selected models:")
for regime, model_name in regime_models.items():
    print(
        f"  {regime}: {model_name}"
    )

print("\nPrediction file saved:")
print(PRED_PATH)

print("\nModel bundle saved:")
print(MODEL_PATH)

print("\nResults saved:")
print(RESULT_PATH)

print("\n" + "=" * 80)
print("AHRN FINAL MODEL COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# AHRN — COMPLETE FEATURE ENGINEERING / FEATURE VALIDATION FIX
# =============================================================================

print("=" * 80)
print("AHRN — BUILDING FINAL MODEL FEATURES")
print("=" * 80)

# -------------------------------------------------------------------------
# Original numeric features
# -------------------------------------------------------------------------

BASE_NUMERIC_FEATURES = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
]

# -------------------------------------------------------------------------
# Check which raw columns genuinely exist
# -------------------------------------------------------------------------

print("\nChecking raw features...")

for c in BASE_NUMERIC_FEATURES:

    if c in df.columns:
        print(f"  {c}: FOUND")

    else:
        print(f"  {c}: MISSING")

# -------------------------------------------------------------------------
# Create missing raw columns if necessary
#
# IMPORTANT:
# We do NOT delete rows.
# We create the missingness indicator BEFORE imputation.
# -------------------------------------------------------------------------

missing_raw_columns = []

for c in BASE_NUMERIC_FEATURES:

    if c not in df.columns:

        missing_raw_columns.append(c)

        # Entirely unavailable feature.
        # Use NaN first so its missingness indicator becomes 1.
        df[c] = np.nan

        print(
            f"Created unavailable raw feature: {c}"
        )

# -------------------------------------------------------------------------
# Missingness indicators
# -------------------------------------------------------------------------

MISSINGNESS_BASE = [
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
]

MISSINGNESS_FEATURES = []

for c in MISSINGNESS_BASE:

    indicator = f"{c}_missing"

    # IMPORTANT:
    # Indicator is created from the RAW feature BEFORE imputation.
    df[indicator] = (
        df[c]
        .isna()
        .astype(np.float32)
    )

    MISSINGNESS_FEATURES.append(
        indicator
    )

# -------------------------------------------------------------------------
# Additional engineered features
# -------------------------------------------------------------------------

# Bid-ask spread
if "bid_ask_spread" not in df.columns:

    if "bid" in df.columns and "ask" in df.columns:

        df["bid_ask_spread"] = (
            df["ask"] - df["bid"]
        )

    else:

        df["bid_ask_spread"] = np.nan


# Dollar volume
if "dollar_volume" not in df.columns:

    if (
        "volume" in df.columns
        and "total_return_index" in df.columns
    ):

        df["dollar_volume"] = (
            df["volume"]
            * df["total_return_index"]
        )

    elif (
        "volume" in df.columns
        and "bid" in df.columns
        and "ask" in df.columns
    ):

        mid_price = (
            df["bid"] + df["ask"]
        ) / 2.0

        df["dollar_volume"] = (
            df["volume"]
            * mid_price
        )

    else:

        df["dollar_volume"] = np.nan


# -------------------------------------------------------------------------
# S&P 500 membership
# -------------------------------------------------------------------------

if "in_sp500" not in df.columns:

    print(
        "in_sp500 not found — creating neutral membership feature."
    )

    df["in_sp500"] = 0.0


# -------------------------------------------------------------------------
# Final AHRN feature list
# -------------------------------------------------------------------------

MODEL_FEATURES = [

    # Raw numerical features
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",

    # Liquidity / microstructure
    "bid_ask_spread",
    "dollar_volume",

    # Missingness information
    "eps_missing",
    "dividend_yield_missing",
    "short_interest_ratio_missing",
    "best_eps_missing",
    "px_to_sales_missing",
    "price_to_book_missing",
    "oper_margin_missing",
    "momentum_12_1_missing",

    # Universe information
    "in_sp500",
]

# -------------------------------------------------------------------------
# Verify everything exists
# -------------------------------------------------------------------------

missing_model_features = [
    c for c in MODEL_FEATURES
    if c not in df.columns
]

if missing_model_features:

    raise ValueError(
        "Unable to construct the following AHRN features:\n"
        + "\n".join(
            f" - {c}"
            for c in missing_model_features
        )
    )

# -------------------------------------------------------------------------
# Convert model features to numeric
# -------------------------------------------------------------------------

for c in MODEL_FEATURES:

    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

# -------------------------------------------------------------------------
# Replace impossible numerical values
# -------------------------------------------------------------------------

for c in MODEL_FEATURES:

    df[c] = df[c].replace(
        [np.inf, -np.inf],
        np.nan
    )

# -------------------------------------------------------------------------
# Feature summary
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL AHRN FEATURE SET")
print("=" * 80)

for i, c in enumerate(MODEL_FEATURES):

    print(
        f"{i:2d} - {c}"
    )

print(
    "\nNumber of model features:",
    len(MODEL_FEATURES)
)

print(
    "Missing model features:",
    len(missing_model_features)
)

print(
    "Rows:",
    len(df)
)

print("=" * 80)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 80)

In [ ]:
# =============================================================================
# FINAL AHRN MODEL
# Regime-Gated Cross-Sectional Residual Learning
#
# DESIGN:
#   1. Robust dataset discovery
#   2. 23-feature AHRN representation
#   3. Cross-sectional target
#   4. Ridge benchmark
#   5. Nonlinear residual learner
#   6. Regime detection without hmmlearn
#   7. Validation-only model selection
#   8. Regime-gated residual adjustment
#   9. Automatic fallback to Ridge if AHRN does not improve
#  10. Strict chronological out-of-sample evaluation
#
# RANDOM SEED: 42
# =============================================================================

import os
import glob
import json
import pickle
import warnings
import random

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from scipy.stats import spearmanr, pearsonr

warnings.filterwarnings("ignore")

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

print("=" * 80)
print("FINAL AHRN — REGIME-GATED CROSS-SECTIONAL RESIDUAL MODEL")
print("=" * 80)

# =============================================================================
# 1. DATASET DISCOVERY
# =============================================================================

print("\nSearching for dataset...")

search_roots = [
    "data/raw",
    "results",
    "/content",
    "."
]

csv_files = []

for root in search_roots:

    if not os.path.exists(root):
        continue

    for pattern in ("**/*.csv", "**/*.CSV"):

        csv_files.extend(
            glob.glob(
                os.path.join(root, pattern),
                recursive=True
            )
        )

csv_files = sorted(
    list(dict.fromkeys(csv_files))
)

if not csv_files:

    raise FileNotFoundError(
        "No CSV file found in Kaggle input/working directories."
    )

required_core = {
    "date",
    "ticker",
    "total_return_index"
}

candidates = []

for path in csv_files:

    try:

        sample = pd.read_csv(
            path,
            nrows=10
        )

        score = len(
            required_core.intersection(
                set(sample.columns)
            )
        )

        candidates.append(
            (
                score,
                len(sample.columns),
                path
            )
        )

    except Exception:
        pass

candidates = [
    x for x in candidates
    if x[0] >= 2
]

if not candidates:

    raise FileNotFoundError(
        "No CSV containing the required AHRN dataset columns was found."
    )

candidates.sort(
    key=lambda x: (-x[0], -x[1], x[2])
)

DATA_PATH = candidates[0][2]

print("Selected dataset:")
print(DATA_PATH)

df = pd.read_csv(DATA_PATH)

print(
    "Dataset shape:",
    df.shape
)

# =============================================================================
# 2. BASIC VALIDATION
# =============================================================================

for c in [
    "date",
    "ticker",
    "total_return_index"
]:

    if c not in df.columns:

        raise ValueError(
            f"Required column missing: {c}"
        )

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

if df["date"].isna().any():

    raise ValueError(
        "Invalid dates detected."
    )

df = (
    df
    .sort_values(
        ["date", "ticker"]
    )
    .drop_duplicates(
        ["date", "ticker"],
        keep="first"
    )
    .reset_index(drop=True)
)

print(
    "Date range:",
    df["date"].min(),
    "to",
    df["date"].max()
)

print(
    "Unique dates:",
    df["date"].nunique()
)

print(
    "Unique tickers:",
    df["ticker"].nunique()
)

# =============================================================================
# 3. FEATURE ENGINEERING
# =============================================================================

print("\n" + "=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

RAW_FEATURES = [

    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1"
]

MISSINGNESS_BASE = [

    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1"
]

# -------------------------------------------------------------------------
# Make sure raw features exist
# -------------------------------------------------------------------------

for c in RAW_FEATURES:

    if c not in df.columns:

        df[c] = np.nan

# -------------------------------------------------------------------------
# Missingness indicators BEFORE imputation
# -------------------------------------------------------------------------

for c in MISSINGNESS_BASE:

    df[f"{c}_missing"] = (
        df[c]
        .isna()
        .astype(np.float32)
    )

# -------------------------------------------------------------------------
# Bid-ask spread
# -------------------------------------------------------------------------

if (
    "bid" in df.columns
    and "ask" in df.columns
):

    df["bid_ask_spread"] = (
        df["ask"] - df["bid"]
    )

else:

    df["bid_ask_spread"] = np.nan

# -------------------------------------------------------------------------
# Dollar volume
# -------------------------------------------------------------------------

if (
    "volume" in df.columns
    and "total_return_index" in df.columns
):

    df["dollar_volume"] = (
        df["volume"]
        * df["total_return_index"]
    )

else:

    df["dollar_volume"] = np.nan

# -------------------------------------------------------------------------
# S&P membership
# -------------------------------------------------------------------------

if "in_sp500" not in df.columns:

    df["in_sp500"] = 0.0

# -------------------------------------------------------------------------
# EXACT 23-FEATURE AHRN INPUT
# -------------------------------------------------------------------------

MODEL_FEATURES = [

    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",

    "bid_ask_spread",
    "dollar_volume",

    "eps_missing",
    "dividend_yield_missing",
    "short_interest_ratio_missing",
    "best_eps_missing",
    "px_to_sales_missing",
    "price_to_book_missing",
    "oper_margin_missing",
    "momentum_12_1_missing",

    "in_sp500"
]

assert len(MODEL_FEATURES) == 23

# -------------------------------------------------------------------------
# Numeric conversion
# -------------------------------------------------------------------------

for c in MODEL_FEATURES:

    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

    df[c] = df[c].replace(
        [np.inf, -np.inf],
        np.nan
    )

print(
    "AHRN features:",
    len(MODEL_FEATURES)
)

# =============================================================================
# 4. TARGET CONSTRUCTION
# =============================================================================

print("\n" + "=" * 80)
print("TARGET CONSTRUCTION")
print("=" * 80)

# One-period forward total-return-index return.
#
# shift(-1) is performed within ticker, preventing one company's
# next observation from becoming another company's target.

df["forward_tri"] = (
    df
    .groupby("ticker")["total_return_index"]
    .shift(-1)
)

df["target_return"] = (
    df["forward_tri"]
    / df["total_return_index"]
    - 1.0
)

# -------------------------------------------------------------------------
# Cross-sectional target
#
# Ranking/z-scoring by date prevents the model from merely learning
# market-wide return levels.
# -------------------------------------------------------------------------

def cross_sectional_zscore(s):

    std = s.std(ddof=0)

    if not np.isfinite(std) or std < 1e-12:

        return pd.Series(
            0.0,
            index=s.index
        )

    return (
        (s - s.mean())
        / std
    )

df["target_cs"] = (
    df
    .groupby("date")["target_return"]
    .transform(cross_sectional_zscore)
)

# Robust clipping of extreme targets.

df["target_cs"] = df["target_cs"].clip(
    -4.0,
    4.0
)

# Economic target retained for final evaluation.

df = df[
    np.isfinite(df["target_return"])
    & np.isfinite(df["target_cs"])
].copy()

df = df.reset_index(drop=True)

print(
    "Rows after target validation:",
    len(df)
)

# =============================================================================
# 5. CHRONOLOGICAL SPLIT
# =============================================================================

print("\n" + "=" * 80)
print("CHRONOLOGICAL SPLIT")
print("=" * 80)

unique_dates = np.array(
    sorted(df["date"].unique())
)

n_dates = len(unique_dates)

if n_dates < 100:

    raise ValueError(
        "Too few dates for robust chronological evaluation."
    )

# Last 64 dates = 32 validation + 32 test.
# Everything before = training.

TEST_DATES = 32
VALIDATION_DATES = 32

if n_dates <= TEST_DATES + VALIDATION_DATES:

    raise ValueError(
        "Insufficient dates for train/validation/test split."
    )

train_dates = unique_dates[
    :-(TEST_DATES + VALIDATION_DATES)
]

validation_dates = unique_dates[
    -(
        TEST_DATES + VALIDATION_DATES
    ):-TEST_DATES
]

test_dates = unique_dates[
    -TEST_DATES:
]

train = df[
    df["date"].isin(train_dates)
].copy()

validation = df[
    df["date"].isin(validation_dates)
].copy()

test = df[
    df["date"].isin(test_dates)
].copy()

print(
    "Train dates:",
    len(train_dates),
    "|",
    train_dates[0],
    "to",
    train_dates[-1]
)

print(
    "Validation dates:",
    len(validation_dates),
    "|",
    validation_dates[0],
    "to",
    validation_dates[-1]
)

print(
    "Test dates:",
    len(test_dates),
    "|",
    test_dates[0],
    "to",
    test_dates[-1]
)

print(
    "\nRows:",
    len(train),
    len(validation),
    len(test)
)

# =============================================================================
# 6. PREPROCESSING
# =============================================================================

print("\n" + "=" * 80)
print("PREPROCESSING")
print("=" * 80)

imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)

scaler = StandardScaler()

X_train_raw = train[
    MODEL_FEATURES
].copy()

X_validation_raw = validation[
    MODEL_FEATURES
].copy()

X_test_raw = test[
    MODEL_FEATURES
].copy()

X_train_imp = imputer.fit_transform(
    X_train_raw
)

X_validation_imp = imputer.transform(
    X_validation_raw
)

X_test_imp = imputer.transform(
    X_test_raw
)

X_train = scaler.fit_transform(
    X_train_imp
)

X_validation = scaler.transform(
    X_validation_imp
)

X_test = scaler.transform(
    X_test_imp
)

# Final finite guarantee.

X_train = np.nan_to_num(
    X_train,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_validation = np.nan_to_num(
    X_validation,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_test = np.nan_to_num(
    X_test,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

y_train = train["target_cs"].values.astype(
    np.float32
)

y_validation = validation["target_cs"].values.astype(
    np.float32
)

y_test = test["target_cs"].values.astype(
    np.float32
)

assert X_train.shape[1] == 23
assert np.isfinite(X_train).all()
assert np.isfinite(X_validation).all()
assert np.isfinite(X_test).all()

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print(
    "All model inputs finite: PASS"
)

# =============================================================================
# 7. CROSS-SECTIONAL RANK IC
# =============================================================================

def monthly_rank_ic(
    dates,
    predictions,
    actual
):

    temp = pd.DataFrame({
        "date": pd.to_datetime(dates),
        "prediction": predictions,
        "actual": actual
    })

    ics = []

    for _, g in temp.groupby("date"):

        if len(g) < 10:
            continue

        if (
            g["prediction"].nunique() < 2
            or
            g["actual"].nunique() < 2
        ):
            continue

        ic = spearmanr(
            g["prediction"],
            g["actual"]
        ).statistic

        if np.isfinite(ic):

            ics.append(float(ic))

    if len(ics) == 0:

        return {
            "mean": 0.0,
            "median": 0.0,
            "std": 0.0,
            "ir": 0.0,
            "positive": 0,
            "months": 0
        }

    arr = np.asarray(ics)

    std = arr.std(ddof=1) if len(arr) > 1 else 0.0

    ir = (
        arr.mean() / std * np.sqrt(len(arr))
        if std > 1e-12
        else 0.0
    )

    return {
        "mean": float(arr.mean()),
        "median": float(np.median(arr)),
        "std": float(std),
        "ir": float(ir),
        "positive": int((arr > 0).sum()),
        "months": int(len(arr))
    }

# =============================================================================
# 8. RIDGE BENCHMARK
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 1 — RIDGE BENCHMARK")
print("=" * 80)

ridge_alphas = [
    10.0,
    25.0,
    50.0,
    100.0,
    250.0,
    500.0,
    1000.0
]

best_ridge = None
best_ridge_ic = -np.inf

for alpha in ridge_alphas:

    model = Ridge(
        alpha=alpha,
        random_state=SEED
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_validation
    )

    ic = monthly_rank_ic(
        validation["date"],
        pred,
        y_validation
    )["mean"]

    if ic > best_ridge_ic:

        best_ridge_ic = ic
        best_ridge = model
        best_ridge_alpha = alpha

ridge_val_pred = best_ridge.predict(
    X_validation
)

ridge_test_pred = best_ridge.predict(
    X_test
)

print(
    "Best Ridge alpha:",
    best_ridge_alpha
)

print(
    "Validation Rank IC:",
    best_ridge_ic
)

# =============================================================================
# 9. NONLINEAR RESIDUAL LEARNER
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 2 — NONLINEAR RESIDUAL LEARNER")
print("=" * 80)

# Ridge captures the stable linear cross-sectional component.
#
# The nonlinear model is trained on RIDGE RESIDUALS rather than the
# raw target. This prevents the nonlinear learner from simply
# replacing a strong linear benchmark with a noisy prediction.

ridge_train_pred = best_ridge.predict(
    X_train
)

train_residual = (
    y_train - ridge_train_pred
)

hgb_configs = [

    (2, 0.03, 100, 10),
    (2, 0.03, 200, 15),
    (3, 0.03, 150, 10),
    (3, 0.02, 250, 15),
    (4, 0.02, 200, 15),
    (4, 0.03, 150, 20),
    (5, 0.02, 200, 20)

]

best_hgb = None
best_hgb_ic = -np.inf
best_hgb_config = None

for (
    max_iter_depth,
    learning_rate,
    max_iter,
    min_samples_leaf
) in hgb_configs:

    model = HistGradientBoostingRegressor(

        max_iter=max_iter,

        learning_rate=learning_rate,

        max_leaf_nodes=2 ** max_iter_depth,

        min_samples_leaf=min_samples_leaf,

        l2_regularization=5.0,

        loss="squared_error",

        random_state=SEED
    )

    model.fit(
        X_train,
        train_residual
    )

    residual_pred = model.predict(
        X_validation
    )

    candidate = (
        ridge_val_pred
        + residual_pred
    )

    ic = monthly_rank_ic(
        validation["date"],
        candidate,
        y_validation
    )["mean"]

    if ic > best_hgb_ic:

        best_hgb_ic = ic
        best_hgb = model
        best_hgb_config = (
            max_iter_depth,
            learning_rate,
            max_iter,
            min_samples_leaf
        )

print(
    "Best HGB configuration:",
    best_hgb_config
)

print(
    "Validation Rank IC:",
    best_hgb_ic
)

# =============================================================================
# 10. REGIME DETECTION
# =============================================================================

print("\n" + "=" * 80)
print("REGIME DETECTION")
print("=" * 80)

# No hmmlearn.
#
# Regime detection is deliberately based ONLY on historical information.
# We classify each month using market return and volatility.
#
# This preserves the regime-gated novelty without requiring an HMM.

market_monthly = (
    df
    .groupby("date")["target_return"]
    .agg(
        market_return="mean",
        market_volatility="std"
    )
)

market_monthly["market_volatility"] = (
    market_monthly["market_volatility"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(
        market_monthly["market_volatility"]
        .median()
    )
)

# Rolling historical z-scores.

rolling_mean_return = (
    market_monthly["market_return"]
    .rolling(
        24,
        min_periods=12
    )
    .mean()
)

rolling_std_return = (
    market_monthly["market_return"]
    .rolling(
        24,
        min_periods=12
    )
    .std()
)

rolling_mean_vol = (
    market_monthly["market_volatility"]
    .rolling(
        24,
        min_periods=12
    )
    .mean()
)

rolling_std_vol = (
    market_monthly["market_volatility"]
    .rolling(
        24,
        min_periods=12
    )
    .std()
)

return_z = (
    market_monthly["market_return"]
    - rolling_mean_return
) / rolling_std_return.replace(0, np.nan)

vol_z = (
    market_monthly["market_volatility"]
    - rolling_mean_vol
) / rolling_std_vol.replace(0, np.nan)

return_z = return_z.fillna(0.0)
vol_z = vol_z.fillna(0.0)

# Simple interpretable regime rule:
#
# BULL:
#   strong positive return environment
#
# STRESS:
#   unusually high volatility OR strongly negative return
#
# NORMAL:
#   everything else

regime = pd.Series(
    "NORMAL",
    index=market_monthly.index
)

regime[
    (return_z >= 0.75)
    & (vol_z < 1.50)
] = "BULL"

regime[
    (vol_z >= 1.50)
    | (return_z <= -0.75)
] = "STRESS"

regime_df = pd.DataFrame({
    "regime": regime
})

def attach_regime(data):

    out = data.copy()

    out["regime"] = (
        out["date"]
        .map(regime_df["regime"])
        .fillna("NORMAL")
    )

    return out

train = attach_regime(train)
validation = attach_regime(validation)
test = attach_regime(test)

print(
    "Train regimes:"
)

print(
    train["regime"].value_counts()
)

print(
    "\nValidation regimes:"
)

print(
    validation["regime"].value_counts()
)

print(
    "\nTest regimes:"
)

print(
    test["regime"].value_counts()
)

# =============================================================================
# 11. REGIME-GATED RESIDUAL STRENGTH
# =============================================================================

print("\n" + "=" * 80)
print("REGIME-GATED RESIDUAL OPTIMIZATION")
print("=" * 80)

hgb_val_residual = best_hgb.predict(
    X_validation
)

# Candidate residual multipliers.
#
# Small values deliberately avoid the catastrophic extrapolation
# seen in the earlier RF/HGB ensemble.

candidate_strengths = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50
]

regime_strength_candidates = []

for bull_strength in candidate_strengths:

    for normal_strength in candidate_strengths:

        for stress_strength in candidate_strengths:

            pred = ridge_val_pred.copy()

            regime_array = (
                validation["regime"]
                .values
            )

            pred += np.where(
                regime_array == "BULL",
                bull_strength * hgb_val_residual,
                np.where(
                    regime_array == "STRESS",
                    stress_strength * hgb_val_residual,
                    normal_strength * hgb_val_residual
                )
            )

            ic = monthly_rank_ic(
                validation["date"],
                pred,
                y_validation
            )["mean"]

            regime_strength_candidates.append(
                (
                    ic,
                    bull_strength,
                    normal_strength,
                    stress_strength
                )
            )

regime_strength_candidates.sort(
    key=lambda x: x[0],
    reverse=True
)

best_candidate = (
    regime_strength_candidates[0]
)

best_candidate_ic = best_candidate[0]

bull_strength = best_candidate[1]
normal_strength = best_candidate[2]
stress_strength = best_candidate[3]

print(
    "Best validation Rank IC:",
    best_candidate_ic
)

print(
    "BULL residual strength:",
    bull_strength
)

print(
    "NORMAL residual strength:",
    normal_strength
)

print(
    "STRESS residual strength:",
    stress_strength
)

# =============================================================================
# 12. SAFETY GATE
# =============================================================================

print("\n" + "=" * 80)
print("AHRN MODEL SELECTION")
print("=" * 80)

# Require actual improvement over Ridge.
#
# If nonlinear residual learning does not improve validation Rank IC,
# AHRN automatically falls back to Ridge.
#
# This prevents us from reporting a "novel" but empirically inferior
# model simply because it is more complicated.

IMPROVEMENT_TOLERANCE = 0.0005

if (
    best_candidate_ic
    >
    best_ridge_ic + IMPROVEMENT_TOLERANCE
):

    USE_AHRN = True

else:

    USE_AHRN = False

print(
    "Ridge validation Rank IC:",
    best_ridge_ic
)

print(
    "Regime-gated AHRN validation Rank IC:",
    best_candidate_ic
)

print(
    "Required improvement:",
    IMPROVEMENT_TOLERANCE
)

if USE_AHRN:

    print(
        "\nAHRN selected: PASS"
    )

else:

    print(
        "\nAHRN nonlinear component rejected."
    )

    print(
        "Using Ridge benchmark as final predictor."
    )

# =============================================================================
# 13. FINAL TEST PREDICTIONS
# =============================================================================

print("\n" + "=" * 80)
print("FINAL OUT-OF-SAMPLE TEST PREDICTIONS")
print("=" * 80)

hgb_test_residual = best_hgb.predict(
    X_test
)

if USE_AHRN:

    test_regimes = (
        test["regime"]
        .values
    )

    final_test_pred = (
        ridge_test_pred
        +
        np.where(
            test_regimes == "BULL",
            bull_strength * hgb_test_residual,
            np.where(
                test_regimes == "STRESS",
                stress_strength * hgb_test_residual,
                normal_strength * hgb_test_residual
            )
        )
    )

else:

    final_test_pred = ridge_test_pred.copy()

# =============================================================================
# 14. FINAL METRICS
# =============================================================================

def regression_metrics(
    y_true,
    y_pred
):

    mse = mean_squared_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    pearson = pearsonr(
        y_true,
        y_pred
    ).statistic

    spearman = spearmanr(
        y_true,
        y_pred
    ).statistic

    return {
        "MSE": float(mse),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "R2": float(r2),
        "Pearson": float(pearson),
        "Spearman": float(spearman)
    }

metrics = regression_metrics(
    y_test,
    final_test_pred
)

rank_ic = monthly_rank_ic(
    test["date"],
    final_test_pred,
    y_test
)

print(
    "\nFinal test regression:"
)

for k, v in metrics.items():

    print(
        f"{k:10s}: {v:.8f}"
    )

print(
    "\nFinal Rank IC:"
)

for k, v in rank_ic.items():

    print(
        f"{k:10s}: {v}"
    )

# =============================================================================
# 15. ECONOMIC DECILE ANALYSIS
# =============================================================================

print("\n" + "=" * 80)
print("FINAL DECILE ANALYSIS")
print("=" * 80)

evaluation = pd.DataFrame({

    "date": test["date"].values,

    "ticker": test["ticker"].values,

    "prediction": final_test_pred,

    "actual_return": test["target_return"].values
})

evaluation["decile"] = (
    evaluation
    .groupby("date")["prediction"]
    .transform(
        lambda x:
        pd.qcut(
            x.rank(
                method="first"
            ),
            10,
            labels=False
        ) + 1
    )
)

decile_table = (
    evaluation
    .groupby("decile")
    .agg(
        observations=("actual_return", "size"),
        mean_prediction=("prediction", "mean"),
        mean_actual_return=("actual_return", "mean"),
        median_actual_return=("actual_return", "median"),
        actual_std=("actual_return", "std")
    )
)

print(
    decile_table
)

# =============================================================================
# 16. LONG-SHORT PORTFOLIO
# =============================================================================

monthly_ls = []

for date, g in evaluation.groupby("date"):

    if len(g) < 20:
        continue

    g = g.sort_values(
        "prediction"
    )

    n = max(
        1,
        int(len(g) * 0.10)
    )

    bottom = (
        g
        .iloc[:n]["actual_return"]
        .mean()
    )

    top = (
        g
        .iloc[-n:]["actual_return"]
        .mean()
    )

    monthly_ls.append(
        {
            "date": date,
            "long_short": top - bottom
        }
    )

monthly_ls = pd.DataFrame(
    monthly_ls
)

if len(monthly_ls) > 0:

    ls_mean = monthly_ls[
        "long_short"
    ].mean()

    ls_std = monthly_ls[
        "long_short"
    ].std(ddof=1)

    annualized_ls = (
        ls_mean * 12
    )

    annualized_sharpe = (
        np.sqrt(12)
        * ls_mean
        / ls_std
        if ls_std > 1e-12
        else 0.0
    )

    hit_rate = (
        monthly_ls["long_short"] > 0
    ).mean()

else:

    ls_mean = 0.0
    ls_std = 0.0
    annualized_ls = 0.0
    annualized_sharpe = 0.0
    hit_rate = 0.0

print("\n" + "=" * 80)
print("FINAL LONG-SHORT PERFORMANCE")
print("=" * 80)

print(
    "Months:",
    len(monthly_ls)
)

print(
    "Mean monthly L/S:",
    ls_mean
)

print(
    "Annualized L/S:",
    annualized_ls
)

print(
    "Annualized Sharpe:",
    annualized_sharpe
)

print(
    "Hit rate:",
    hit_rate
)

# =============================================================================
# 17. DECILE MONOTONICITY
# =============================================================================

decile_returns = (
    decile_table[
        "mean_actual_return"
    ]
    .values
)

positive_steps = int(
    np.sum(
        np.diff(decile_returns) > 0
    )
)

monotonicity = (
    positive_steps / 9.0
)

print("\n" + "=" * 80)
print("DECILE MONOTONICITY")
print("=" * 80)

print(
    "Positive steps:",
    positive_steps,
    "/ 9"
)

print(
    "Monotonicity ratio:",
    monotonicity
)

# =============================================================================
# 18. RIDGE VS AHRN COMPARISON
# =============================================================================

ridge_test_ic = monthly_rank_ic(
    test["date"],
    ridge_test_pred,
    y_test
)

ridge_test_ls = []

for date, g in pd.DataFrame({

    "date": test["date"].values,

    "prediction": ridge_test_pred,

    "actual": test["target_return"].values

}).groupby("date"):

    if len(g) < 20:
        continue

    g = g.sort_values(
        "prediction"
    )

    n = max(
        1,
        int(len(g) * 0.10)
    )

    ridge_test_ls.append(
        g.iloc[-n:]["actual"].mean()
        -
        g.iloc[:n]["actual"].mean()
    )

ridge_test_ls = np.asarray(
    ridge_test_ls
)

ridge_ls_mean = (
    ridge_test_ls.mean()
    if len(ridge_test_ls)
    else 0.0
)

ridge_ls_std = (
    ridge_test_ls.std(ddof=1)
    if len(ridge_test_ls) > 1
    else 0.0
)

ridge_sharpe = (
    np.sqrt(12)
    * ridge_ls_mean
    / ridge_ls_std
    if ridge_ls_std > 1e-12
    else 0.0
)

print("\n" + "=" * 80)
print("AHRN VS RIDGE BENCHMARK")
print("=" * 80)

print(
    "Ridge test Rank IC:",
    ridge_test_ic["mean"]
)

print(
    "AHRN test Rank IC:",
    rank_ic["mean"]
)

print(
    "Ridge annualized Sharpe:",
    ridge_sharpe
)

print(
    "AHRN annualized Sharpe:",
    annualized_sharpe
)

print(
    "Rank IC difference:",
    rank_ic["mean"]
    - ridge_test_ic["mean"]
)

print(
    "Sharpe difference:",
    annualized_sharpe
    - ridge_sharpe
)

# =============================================================================
# 19. PREDICTION SCALE
# =============================================================================

actual_std = float(
    np.std(
        test["target_return"].values
    )
)

prediction_std = float(
    np.std(
        final_test_pred
    )
)

vol_ratio = (
    prediction_std / actual_std
    if actual_std > 1e-12
    else np.nan
)

print("\n" + "=" * 80)
print("PREDICTION SCALE")
print("=" * 80)

print(
    "Actual return std:",
    actual_std
)

print(
    "Prediction std:",
    prediction_std
)

print(
    "Prediction / actual volatility ratio:",
    vol_ratio
)

# =============================================================================
# 20. REGIME-CONDITIONAL PERFORMANCE
# =============================================================================

print("\n" + "=" * 80)
print("REGIME-CONDITIONAL AHRN PERFORMANCE")
print("=" * 80)

regime_results = []

for r in sorted(
    test["regime"].unique()
):

    mask = (
        test["regime"].values == r
    )

    if mask.sum() < 10:
        continue

    p = final_test_pred[mask]

    y = y_test[mask]

    pearson = pearsonr(
        y,
        p
    ).statistic

    spearman = spearmanr(
        y,
        p
    ).statistic

    regime_results.append({

        "regime": r,

        "observations": int(mask.sum()),

        "pearson": float(pearson),

        "spearman": float(spearman),

        "actual_mean": float(
            test.loc[
                mask,
                "target_return"
            ].mean()
        ),

        "prediction_mean": float(
            p.mean()
        )
    })

regime_results_df = pd.DataFrame(
    regime_results
)

print(
    regime_results_df
)

# =============================================================================
# 21. SAVE PREDICTIONS
# =============================================================================

prediction_file = (
    "results/"
    "ahrn_final_predictions.csv"
)

output_predictions = test[
    [
        "date",
        "ticker"
    ]
].copy()

output_predictions[
    "ahrn_prediction"
] = final_test_pred

output_predictions[
    "actual_return"
] = test[
    "target_return"
].values

output_predictions[
    "regime"
] = test[
    "regime"
].values

output_predictions.to_csv(
    prediction_file,
    index=False
)

# =============================================================================
# 22. SAVE MODEL BUNDLE
# =============================================================================

model_file = (
    "results/"
    "ahrn_final_models.pkl"
)

model_bundle = {

    "model_type":
        "regime_gated_ridge_residual",

    "use_ahrn":
        bool(USE_AHRN),

    "ridge":
        best_ridge,

    "hgb":
        best_hgb,

    "imputer":
        imputer,

    "scaler":
        scaler,

    "model_features":
        MODEL_FEATURES,

    "ridge_alpha":
        best_ridge_alpha,

    "hgb_config":
        best_hgb_config,

    "regime_strengths": {

        "BULL":
            float(bull_strength),

        "NORMAL":
            float(normal_strength),

        "STRESS":
            float(stress_strength)
    },

    "seed":
        SEED
}

with open(
    model_file,
    "wb"
) as f:

    pickle.dump(
        model_bundle,
        f
    )

# =============================================================================
# 23. SAVE RESULTS
# =============================================================================

results_file = (
    "results/"
    "ahrn_final_results.json"
)

results = {

    "model":
        "Regime-Gated Cross-Sectional Residual AHRN",

    "seed":
        SEED,

    "data_path":
        DATA_PATH,

    "n_features":
        len(MODEL_FEATURES),

    "features":
        MODEL_FEATURES,

    "train_rows":
        int(len(train)),

    "validation_rows":
        int(len(validation)),

    "test_rows":
        int(len(test)),

    "ridge_validation_rank_ic":
        float(best_ridge_ic),

    "ahrn_validation_rank_ic":
        float(best_candidate_ic),

    "ahrn_selected":
        bool(USE_AHRN),

    "test_metrics":
        metrics,

    "test_rank_ic":
        rank_ic,

    "ridge_test_rank_ic":
        ridge_test_ic,

    "mean_monthly_long_short":
        float(ls_mean),

    "annualized_long_short":
        float(annualized_ls),

    "annualized_sharpe":
        float(annualized_sharpe),

    "hit_rate":
        float(hit_rate),

    "decile_monotonicity":
        float(monotonicity),

    "prediction_volatility":
        prediction_std,

    "actual_volatility":
        actual_std,

    "prediction_actual_volatility_ratio":
        float(vol_ratio),

    "ridge_sharpe":
        float(ridge_sharpe),

    "sharpe_difference":
        float(
            annualized_sharpe
            - ridge_sharpe
        ),

    "rank_ic_difference":
        float(
            rank_ic["mean"]
            - ridge_test_ic["mean"]
        ),

    "regime_results":
        regime_results
}

with open(
    results_file,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        default=str
    )

# =============================================================================
# 24. FINAL VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("FINAL AHRN VALIDATION")
print("=" * 80)

checks = {

    "Dataset exists":
        os.path.exists(DATA_PATH),

    "Prediction file exists":
        os.path.exists(prediction_file),

    "Model file exists":
        os.path.exists(model_file),

    "Results file exists":
        os.path.exists(results_file),

    "Train inputs finite":
        bool(np.isfinite(X_train).all()),

    "Validation inputs finite":
        bool(np.isfinite(X_validation).all()),

    "Test inputs finite":
        bool(np.isfinite(X_test).all()),

    "Test predictions finite":
        bool(np.isfinite(final_test_pred).all()),

    "Prediction count matches test":
        len(final_test_pred) == len(test),

    "No duplicate ticker-date predictions":
        not output_predictions.duplicated(
            ["date", "ticker"]
        ).any(),

    "Chronological split valid":
        (
            train["date"].max()
            <
            validation["date"].min()
            and
            validation["date"].max()
            <
            test["date"].min()
        ),

    "Feature count = 23":
        len(MODEL_FEATURES) == 23
}

for name, passed in checks.items():

    print(
        f"{name}:",
        "PASS" if passed else "FAIL"
    )

if not all(checks.values()):

    raise RuntimeError(
        "FINAL AHRN VALIDATION FAILED."
    )

# =============================================================================
# 25. FINAL OUTPUT
# =============================================================================

print("\n" + "=" * 80)
print("FINAL AHRN RESULTS")
print("=" * 80)

print(
    "AHRN nonlinear component selected:",
    USE_AHRN
)

print(
    "Test Pearson:",
    f"{metrics['Pearson']:.6f}"
)

print(
    "Test Spearman:",
    f"{metrics['Spearman']:.6f}"
)

print(
    "Mean Rank IC:",
    f"{rank_ic['mean']:.6f}"
)

print(
    "Rank IC IR:",
    f"{rank_ic['ir']:.6f}"
)

print(
    "Mean monthly long-short:",
    f"{ls_mean:.6f}"
)

print(
    "Annualized long-short:",
    f"{annualized_ls:.6f}"
)

print(
    "Annualized Sharpe:",
    f"{annualized_sharpe:.6f}"
)

print(
    "Long-short hit rate:",
    f"{hit_rate:.4f}"
)

print(
    "Decile monotonicity:",
    f"{monotonicity:.4f}"
)

print(
    "Prediction/actual volatility ratio:",
    f"{vol_ratio:.6f}"
)

print("\nFiles saved:")

print(
    prediction_file
)

print(
    model_file
)

print(
    results_file
)

print("\n" + "=" * 80)
print("AHRN FINAL MODEL COMPLETE")
print("=" * 80)

print(
    """
Novel component:
Regime-gated nonlinear residual learning.

The Ridge model remains the benchmark and captures the
stable linear cross-sectional component.

The nonlinear learner is trained ONLY on Ridge residuals.

The nonlinear residual is applied with regime-specific
strengths selected exclusively using validation data.

A strict validation gate prevents the nonlinear AHRN
component from replacing Ridge when it does not improve
cross-sectional predictive performance.

The final test period is untouched during model selection.
"""
)

In [ ]:
# =============================================================================
# FINAL AHRN — ADAPTIVE CROSS-SECTIONAL RESIDUAL + NONLINEAR MODEL
# =============================================================================
#
# Research design:
#
#   1. Cross-sectional normalization of characteristics
#   2. Ridge benchmark
#   3. Nonlinear residual learner
#   4. Shallow neural network residual learner
#   5. Market-state / regime interactions
#   6. Validation-selected adaptive combination
#   7. Rank-based portfolio construction
#   8. Strict chronological test evaluation
#
# Motivation:
#   - Gu, Kelly & Xiu (2020): nonlinear interactions matter
#   - Kelly, Pruitt & Su (2019): characteristics can condition exposures
#   - Cross-sectional ranking is aligned with long-short portfolio construction
#
# IMPORTANT:
#   The TEST SET is never used for model selection.
# =============================================================================

import os
import glob
import json
import pickle
import warnings
import random

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# =============================================================================
# SETTINGS
# =============================================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 80)
print("FINAL AHRN — ADAPTIVE CROSS-SECTIONAL RESIDUAL MODEL")
print("=" * 80)
print("Random seed:", SEED)


# =============================================================================
# 1. FIND DATASET AUTOMATICALLY
# =============================================================================

candidate_paths = [
    "data/raw/SP500_master_panel.csv",
    "data/raw/SP500_master_panel.csv",
    "results/SP500_master_panel.csv",
    "results/ahrn_dataset.csv"
]

DATA_PATH = None

for p in candidate_paths:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:

    csvs = glob.glob(
        "data/raw/**/*.csv",
        recursive=True
    )

    csvs += glob.glob(
        "results/*.csv"
    )

    preferred = [
        x for x in csvs
        if "master" in os.path.basename(x).lower()
        or "sp500" in os.path.basename(x).lower()
    ]

    if preferred:
        DATA_PATH = preferred[0]
    elif csvs:
        DATA_PATH = csvs[0]

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not automatically locate the Bloomberg dataset."
    )

print("\nDataset selected:")
print(DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)


# =============================================================================
# 2. BASIC CLEANING
# =============================================================================

if "date" not in df.columns:
    raise ValueError("Dataset must contain 'date'.")

if "ticker" not in df.columns:
    raise ValueError("Dataset must contain 'ticker'.")

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

df = df.dropna(
    subset=["date", "ticker"]
).copy()

df = df.sort_values(
    ["date", "ticker"]
).reset_index(drop=True)

print(
    "Date range:",
    df["date"].min(),
    "to",
    df["date"].max()
)

print(
    "Unique dates:",
    df["date"].nunique()
)

print(
    "Unique tickers:",
    df["ticker"].nunique()
)


# =============================================================================
# 3. FEATURE ENGINEERING
# =============================================================================

BASE_NUMERIC = [
    "volume",
    "volume_avg_30d",
    "market_cap",
    "bid",
    "ask",
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1",
    "bid_ask_spread",
    "dollar_volume"
]

# Only use columns that actually exist.
BASE_NUMERIC = [
    c for c in BASE_NUMERIC
    if c in df.columns
]

print("\nBase numeric features:")
print(BASE_NUMERIC)


# =============================================================================
# 4. MISSINGNESS INDICATORS
# =============================================================================

MISSING_BASE = [
    "eps",
    "dividend_yield",
    "short_interest_ratio",
    "best_eps",
    "px_to_sales",
    "price_to_book",
    "oper_margin",
    "momentum_12_1"
]

for c in MISSING_BASE:

    if c in df.columns:

        df[c + "_missing"] = (
            df[c].isna().astype(float)
        )

    else:

        df[c + "_missing"] = 1.0


# =============================================================================
# 5. ECONOMIC FEATURE ENGINEERING
# =============================================================================

# -------------------------------------------------------------------------
# Liquidity / spread
# -------------------------------------------------------------------------

if "bid" in df.columns and "ask" in df.columns:

    mid = (
        df["bid"] + df["ask"]
    ) / 2.0

    df["relative_spread"] = (
        (df["ask"] - df["bid"])
        / mid.replace(0, np.nan)
    )

else:

    df["relative_spread"] = np.nan


# -------------------------------------------------------------------------
# Size
# -------------------------------------------------------------------------

if "market_cap" in df.columns:

    df["log_market_cap"] = np.log1p(
        df["market_cap"].clip(lower=0)
    )

else:

    df["log_market_cap"] = np.nan


# -------------------------------------------------------------------------
# Liquidity
# -------------------------------------------------------------------------

if "dollar_volume" in df.columns:

    df["log_dollar_volume"] = np.log1p(
        df["dollar_volume"].clip(lower=0)
    )

else:

    df["log_dollar_volume"] = np.nan


# -------------------------------------------------------------------------
# Value transforms
# -------------------------------------------------------------------------

for c in [
    "px_to_sales",
    "price_to_book"
]:

    if c in df.columns:

        df["log_" + c] = np.sign(
            df[c]
        ) * np.log1p(
            np.abs(df[c])
        )


# -------------------------------------------------------------------------
# Earnings transform
# -------------------------------------------------------------------------

if "eps" in df.columns:

    df["eps_signed_log"] = (
        np.sign(df["eps"])
        * np.log1p(np.abs(df["eps"]))
    )

else:

    df["eps_signed_log"] = np.nan


# =============================================================================
# 6. CROSS-SECTIONAL RANK TRANSFORM
# =============================================================================
#
# This is important.
#
# The final portfolio cares about relative stock ordering, not absolute
# prediction magnitude.
#
# Each characteristic is transformed into a within-month percentile.
# =============================================================================

RANK_FEATURES = []

candidate_rank_features = list(
    set(
        BASE_NUMERIC
        + [
            "relative_spread",
            "log_market_cap",
            "log_dollar_volume",
            "log_px_to_sales",
            "log_price_to_book",
            "eps_signed_log"
        ]
    )
)

candidate_rank_features = [
    c for c in candidate_rank_features
    if c in df.columns
]

for c in candidate_rank_features:

    rank_name = c + "_cs_rank"

    df[rank_name] = (
        df.groupby("date")[c]
        .rank(
            pct=True,
            method="average"
        )
    )

    RANK_FEATURES.append(rank_name)


# =============================================================================
# 7. MOMENTUM / VALUE / QUALITY COMPOSITES
# =============================================================================

def cross_sectional_rank(series):
    return series.rank(
        pct=True,
        method="average"
    )


# Momentum
if "momentum_12_1" in df.columns:

    df["momentum_signal"] = (
        df.groupby("date")["momentum_12_1"]
        .transform(cross_sectional_rank)
    )

else:

    df["momentum_signal"] = 0.5


# Value
value_components = []

if "px_to_sales" in df.columns:

    value_components.append(
        1.0 -
        df.groupby("date")["px_to_sales"]
        .transform(cross_sectional_rank)
    )

if "price_to_book" in df.columns:

    value_components.append(
        1.0 -
        df.groupby("date")["price_to_book"]
        .transform(cross_sectional_rank)
    )

if "eps" in df.columns:

    value_components.append(
        df.groupby("date")["eps"]
        .transform(cross_sectional_rank)
    )

if value_components:

    df["value_signal"] = np.nanmean(
        np.vstack(value_components),
        axis=0
    )

else:

    df["value_signal"] = 0.5


# Quality
quality_components = []

if "oper_margin" in df.columns:

    quality_components.append(
        df.groupby("date")["oper_margin"]
        .transform(cross_sectional_rank)
    )

if "best_eps" in df.columns:

    quality_components.append(
        df.groupby("date")["best_eps"]
        .transform(cross_sectional_rank)
    )

if quality_components:

    df["quality_signal"] = np.nanmean(
        np.vstack(quality_components),
        axis=0
    )

else:

    df["quality_signal"] = 0.5


# Liquidity
liquidity_components = []

if "market_cap" in df.columns:

    liquidity_components.append(
        df.groupby("date")["market_cap"]
        .transform(cross_sectional_rank)
    )

if "dollar_volume" in df.columns:

    liquidity_components.append(
        df.groupby("date")["dollar_volume"]
        .transform(cross_sectional_rank)
    )

if liquidity_components:

    df["liquidity_signal"] = np.nanmean(
        np.vstack(liquidity_components),
        axis=0
    )

else:

    df["liquidity_signal"] = 0.5


# =============================================================================
# 8. MARKET STATE VARIABLES
# =============================================================================
#
# These are calculated only from information observable at the date.
# =============================================================================

if "momentum_12_1" in df.columns:

    market_momentum = (
        df.groupby("date")["momentum_12_1"]
        .median()
    )

else:

    market_momentum = pd.Series(
        0.0,
        index=df["date"].drop_duplicates()
    )


if "bid_ask_spread" in df.columns:

    market_spread = (
        df.groupby("date")["bid_ask_spread"]
        .median()
    )

else:

    market_spread = pd.Series(
        0.0,
        index=df["date"].drop_duplicates()
    )


market_state = pd.DataFrame({
    "market_momentum": market_momentum,
    "market_spread": market_spread
})

market_state["market_momentum_z"] = (
    market_state["market_momentum"]
    - market_state["market_momentum"].rolling(
        36,
        min_periods=12
    ).mean()
) / (
    market_state["market_momentum"].rolling(
        36,
        min_periods=12
    ).std()
)

market_state["market_spread_z"] = (
    market_state["market_spread"]
    - market_state["market_spread"].rolling(
        36,
        min_periods=12
    ).mean()
) / (
    market_state["market_spread"].rolling(
        36,
        min_periods=12
    ).std()
)

market_state = market_state.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

df = df.merge(
    market_state[
        [
            "market_momentum_z",
            "market_spread_z"
        ]
    ],
    left_on="date",
    right_index=True,
    how="left"
)


# =============================================================================
# 9. REGIME CLASSIFICATION
# =============================================================================

regime_score = (
    df["market_momentum_z"]
    - df["market_spread_z"]
)

date_regime_score = (
    df.groupby("date")["market_momentum_z"]
    .first()
    -
    df.groupby("date")["market_spread_z"]
    .first()
)

q1 = date_regime_score.quantile(0.33)
q2 = date_regime_score.quantile(0.67)

regime_map = {}

for d, s in date_regime_score.items():

    if s <= q1:
        regime_map[d] = "STRESS"

    elif s >= q2:
        regime_map[d] = "BULL"

    else:
        regime_map[d] = "NORMAL"

df["regime"] = df["date"].map(
    regime_map
)


# =============================================================================
# 10. TARGET CONSTRUCTION
# =============================================================================

if "target_excess_return" in df.columns:

    TARGET = "target_excess_return"

else:

    if "total_return_index" not in df.columns:
        raise ValueError(
            "Neither target_excess_return nor total_return_index exists."
        )

    df = df.sort_values(
        ["ticker", "date"]
    )

    df["forward_return"] = (
        df.groupby("ticker")["total_return_index"]
        .shift(-1)
        / df["total_return_index"]
        - 1.0
    )

    df["target_excess_return"] = (
        df["forward_return"]
        -
        df.groupby("date")["forward_return"]
        .transform("mean")
    )

    TARGET = "target_excess_return"


df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="coerce"
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna(
    subset=[TARGET]
).copy()

print("\nRows after target validation:", len(df))


# =============================================================================
# 11. FINAL MODEL FEATURES
# =============================================================================

MODEL_FEATURES = []

# Original characteristics
for c in BASE_NUMERIC:

    if c in df.columns:
        MODEL_FEATURES.append(c)


# Missingness
for c in MISSING_BASE:

    name = c + "_missing"

    if name in df.columns:
        MODEL_FEATURES.append(name)


# Cross-sectional representations
for c in RANK_FEATURES:

    if c in df.columns:
        MODEL_FEATURES.append(c)


# Economic composites
for c in [
    "momentum_signal",
    "value_signal",
    "quality_signal",
    "liquidity_signal",
    "market_momentum_z",
    "market_spread_z"
]:

    if c in df.columns:
        MODEL_FEATURES.append(c)


# Remove duplicates
MODEL_FEATURES = list(
    dict.fromkeys(MODEL_FEATURES)
)

print("\nTotal AHRN features:", len(MODEL_FEATURES))

print(
    "\n".join(
        f"{i:2d} - {c}"
        for i, c in enumerate(MODEL_FEATURES)
    )
)


# =============================================================================
# 12. IMPUTATION
# =============================================================================

for c in MODEL_FEATURES:

    if c not in df.columns:
        df[c] = 0.0

    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

    # Cross-sectional median imputation
    df[c] = (
        df.groupby("date")[c]
        .transform(
            lambda x: x.fillna(x.median())
        )
    )

    # Global fallback
    df[c] = df[c].fillna(
        df[c].median()
    )

    df[c] = df[c].replace(
        [np.inf, -np.inf],
        np.nan
    )

    df[c] = df[c].fillna(0.0)


# =============================================================================
# 13. CHRONOLOGICAL SPLIT
# =============================================================================

dates = sorted(
    df["date"].unique()
)

n_dates = len(dates)

train_end = int(
    n_dates * 0.80
)

validation_end = int(
    n_dates * 0.90
)

train_dates = dates[:train_end]

validation_dates = dates[
    train_end:validation_end
]

test_dates = dates[
    validation_end:
]

train = df[
    df["date"].isin(train_dates)
].copy()

validation = df[
    df["date"].isin(validation_dates)
].copy()

test = df[
    df["date"].isin(test_dates)
].copy()

print("\n" + "=" * 80)
print("CHRONOLOGICAL SPLIT")
print("=" * 80)

print(
    "Train:",
    len(train_dates),
    train_dates[0],
    "to",
    train_dates[-1]
)

print(
    "Validation:",
    len(validation_dates),
    validation_dates[0],
    "to",
    validation_dates[-1]
)

print(
    "Test:",
    len(test_dates),
    test_dates[0],
    "to",
    test_dates[-1]
)

print(
    "Rows:",
    len(train),
    len(validation),
    len(test)
)


# =============================================================================
# 14. CROSS-SECTIONAL TARGET
# =============================================================================
#
# The raw return is extremely noisy.
#
# For the learning problem we use the cross-sectional z-score.
# Economic evaluation continues to use the actual return.
# =============================================================================

def make_cs_target(frame):

    grouped = frame.groupby("date")[TARGET]

    mean = grouped.transform("mean")

    std = grouped.transform("std")

    z = (
        frame[TARGET] - mean
    ) / std.replace(0, np.nan)

    return z.replace(
        [np.inf, -np.inf],
        np.nan
    ).fillna(0.0)


train["y_cs"] = make_cs_target(train)
validation["y_cs"] = make_cs_target(validation)
test["y_cs"] = make_cs_target(test)


# Clip extreme training targets only
lower = train["y_cs"].quantile(0.01)
upper = train["y_cs"].quantile(0.99)

train["y_cs"] = train["y_cs"].clip(
    lower,
    upper
)

print("\nTarget clipping:")
print(lower, upper)


# =============================================================================
# 15. STANDARDIZATION
# =============================================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(
    train[MODEL_FEATURES]
)

X_validation = scaler.transform(
    validation[MODEL_FEATURES]
)

X_test = scaler.transform(
    test[MODEL_FEATURES]
)

y_train = train["y_cs"].values
y_validation = validation["y_cs"].values
y_test = test["y_cs"].values

print("\nInput shapes:")
print(X_train.shape)
print(X_validation.shape)
print(X_test.shape)

print(
    "Finite:",
    np.isfinite(X_train).all(),
    np.isfinite(X_validation).all(),
    np.isfinite(X_test).all()
)


# =============================================================================
# 16. METRICS
# =============================================================================

def monthly_rank_ic(
    frame,
    prediction_column,
    target_column=TARGET
):

    values = []

    for _, g in frame.groupby("date"):

        if len(g) < 10:
            continue

        a = g[prediction_column]
        b = g[target_column]

        c = a.corr(
            b,
            method="spearman"
        )

        if np.isfinite(c):
            values.append(c)

    return np.asarray(values)


def long_short_series(
    frame,
    prediction_column,
    q=0.10
):

    ls = []

    for _, g in frame.groupby("date"):

        if len(g) < 20:
            continue

        p = g[prediction_column]

        lo = p.quantile(q)
        hi = p.quantile(1.0 - q)

        long = g.loc[
            p >= hi,
            TARGET
        ]

        short = g.loc[
            p <= lo,
            TARGET
        ]

        if len(long) == 0 or len(short) == 0:
            continue

        ls.append(
            long.mean() - short.mean()
        )

    return np.asarray(ls)


def annualized_sharpe(x):

    x = np.asarray(x)

    x = x[np.isfinite(x)]

    if len(x) < 2:
        return -999.0

    s = x.std(ddof=1)

    if s <= 1e-12:
        return -999.0

    return (
        np.sqrt(12.0)
        * x.mean()
        / s
    )


def rank_ic_score(
    frame,
    pred
):

    temp = frame[
        ["date", TARGET]
    ].copy()

    temp["prediction"] = pred

    ic = monthly_rank_ic(
        temp,
        "prediction"
    )

    if len(ic) == 0:
        return -999.0

    return np.mean(ic)


def portfolio_score(
    frame,
    pred
):

    temp = frame[
        ["date", TARGET]
    ].copy()

    temp["prediction"] = pred

    ic = monthly_rank_ic(
        temp,
        "prediction"
    )

    ls = long_short_series(
        temp,
        "prediction"
    )

    if len(ls) < 6:
        return -999.0

    sharpe = annualized_sharpe(ls)

    rank_ic = (
        np.mean(ic)
        if len(ic)
        else 0.0
    )

    positive_months = (
        np.mean(ls > 0)
        if len(ls)
        else 0.0
    )

    # Primary objective = Sharpe
    # Secondary objective = IC stability
    # Small hit-rate tie-breaker.
    score = (
        0.70 * sharpe
        +
        0.20 * (
            rank_ic * 20.0
        )
        +
        0.10 * (
            positive_months - 0.5
        )
    )

    return score


# =============================================================================
# 17. RIDGE BENCHMARK
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 1 — RIDGE BENCHMARK")
print("=" * 80)

ridge_candidates = [
    1.0,
    5.0,
    10.0,
    25.0,
    50.0,
    100.0,
    250.0,
    500.0,
    1000.0
]

best_ridge = None
best_ridge_score = -999999

ridge_validation_prediction = None

for alpha in ridge_candidates:

    model = Ridge(
        alpha=alpha
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_validation
    )

    score = portfolio_score(
        validation,
        pred
    )

    if score > best_ridge_score:

        best_ridge_score = score

        best_ridge = model

        ridge_validation_prediction = pred.copy()

print(
    "Best Ridge alpha:",
    best_ridge.alpha
)

print(
    "Ridge validation portfolio score:",
    best_ridge_score
)

print(
    "Ridge validation Rank IC:",
    rank_ic_score(
        validation,
        ridge_validation_prediction
    )
)

ridge_train_prediction = best_ridge.predict(
    X_train
)

ridge_test_prediction = best_ridge.predict(
    X_test
)


# =============================================================================
# 18. RIDGE RESIDUALS
# =============================================================================

residual_target = (
    y_train
    - ridge_train_prediction
)

# Winsorize residuals
rlo = np.quantile(
    residual_target,
    0.01
)

rhi = np.quantile(
    residual_target,
    0.99
)

residual_target = np.clip(
    residual_target,
    rlo,
    rhi
)


# =============================================================================
# 19. NONLINEAR MODEL — HGB
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 2 — NONLINEAR RESIDUAL HGB")
print("=" * 80)

hgb_configs = [
    (2, 0.02, 150, 15),
    (3, 0.02, 200, 20),
    (4, 0.02, 250, 20),
    (5, 0.02, 200, 20),
    (3, 0.03, 250, 20),
    (5, 0.01, 300, 20)
]

best_hgb = None
best_hgb_score = -999999

best_hgb_validation = None

for depth, lr, ntrees, leaf in hgb_configs:

    model = HistGradientBoostingRegressor(
        max_iter=ntrees,
        learning_rate=lr,
        max_leaf_nodes=2 ** depth,
        min_samples_leaf=leaf,
        l2_regularization=5.0,
        random_state=SEED
    )

    model.fit(
        X_train,
        residual_target
    )

    residual_pred = model.predict(
        X_validation
    )

    # residual prediction is added to Ridge
    for strength in [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.75,
        1.00
    ]:

        combined = (
            ridge_validation_prediction
            +
            strength * residual_pred
        )

        score = portfolio_score(
            validation,
            combined
        )

        if score > best_hgb_score:

            best_hgb_score = score
            best_hgb = (
                model,
                strength,
                depth,
                lr,
                ntrees,
                leaf
            )
            best_hgb_validation = combined.copy()

print(
    "Best HGB:",
    best_hgb[2:]
)

print(
    "Residual strength:",
    best_hgb[1]
)

print(
    "Validation Rank IC:",
    rank_ic_score(
        validation,
        best_hgb_validation
    )
)


# =============================================================================
# 20. EXTRA TREES RESIDUAL MODEL
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 3 — EXTRA TREES RESIDUAL LEARNER")
print("=" * 80)

et_configs = [
    (200, 5, 10),
    (300, 6, 10),
    (400, 7, 15),
    (500, 8, 15),
    (400, 10, 20)
]

best_et = None
best_et_score = -999999
best_et_validation = None

for ntrees, depth, leaf in et_configs:

    model = ExtraTreesRegressor(
        n_estimators=ntrees,
        max_depth=depth,
        min_samples_leaf=leaf,
        max_features=0.70,
        random_state=SEED,
        n_jobs=-1
    )

    model.fit(
        X_train,
        residual_target
    )

    residual_pred = model.predict(
        X_validation
    )

    for strength in [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.75,
        1.00
    ]:

        combined = (
            ridge_validation_prediction
            +
            strength * residual_pred
        )

        score = portfolio_score(
            validation,
            combined
        )

        if score > best_et_score:

            best_et_score = score

            best_et = (
                model,
                strength,
                ntrees,
                depth,
                leaf
            )

            best_et_validation = combined.copy()

print(
    "Best ExtraTrees:",
    best_et[2:]
)

print(
    "Residual strength:",
    best_et[1]
)

print(
    "Validation Rank IC:",
    rank_ic_score(
        validation,
        best_et_validation
    )
)


# =============================================================================
# 21. SHALLOW NEURAL NETWORK RESIDUAL LEARNER
# =============================================================================
#
# The network is deliberately shallow.
#
# Asset-pricing research finds that shallow nonlinear models often perform
# better than excessively deep networks because the signal-to-noise ratio
# is extremely low.
# =============================================================================

print("\n" + "=" * 80)
print("MODEL 4 — SHALLOW AHRN NEURAL RESIDUAL")
print("=" * 80)


def build_network(
    n_features,
    width
):

    inputs = keras.Input(
        shape=(n_features,)
    )

    x = layers.Dense(
        width,
        activation="relu"
    )(inputs)

    x = layers.BatchNormalization()(x)

    x = layers.Dropout(
        0.10
    )(x)

    x = layers.Dense(
        width // 2,
        activation="relu"
    )(x)

    x = layers.Dropout(
        0.05
    )(x)

    outputs = layers.Dense(
        1,
        activation="linear"
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="AHRN_residual_network"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.0005
        ),
        loss=keras.losses.Huber(
            delta=1.0
        )
    )

    return model


nn_widths = [
    32,
    48,
    64
]

best_nn = None
best_nn_score = -999999
best_nn_validation = None

for width in nn_widths:

    tf.keras.backend.clear_session()

    nn = build_network(
        X_train.shape[1],
        width
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    nn.fit(
        X_train,
        residual_target,
        validation_split=0.10,
        epochs=60,
        batch_size=1024,
        verbose=0,
        callbacks=[early_stop]
    )

    residual_pred = nn.predict(
        X_validation,
        batch_size=4096,
        verbose=0
    ).ravel()

    for strength in [
        0.05,
        0.10,
        0.15,
        0.20,
        0.30,
        0.40,
        0.50
    ]:

        combined = (
            ridge_validation_prediction
            +
            strength * residual_pred
        )

        score = portfolio_score(
            validation,
            combined
        )

        if score > best_nn_score:

            best_nn_score = score

            best_nn = (
                nn,
                strength,
                width
            )

            best_nn_validation = combined.copy()

print(
    "Best NN width:",
    best_nn[2]
)

print(
    "NN residual strength:",
    best_nn[1]
)

print(
    "NN validation Rank IC:",
    rank_ic_score(
        validation,
        best_nn_validation
    )
)


# =============================================================================
# 22. ADAPTIVE REGIME GATING
# =============================================================================
#
# Instead of assuming that the nonlinear model works everywhere, estimate
# separate residual strengths for BULL / NORMAL / STRESS.
#
# These strengths are selected using validation only.
# =============================================================================

print("\n" + "=" * 80)
print("REGIME-GATED ADAPTIVE COMBINATION")
print("=" * 80)


hgb_model = best_hgb[0]
et_model = best_et[0]
nn_model = best_nn[0]

hgb_val_residual = hgb_model.predict(
    X_validation
)

et_val_residual = et_model.predict(
    X_validation
)

nn_val_residual = nn_model.predict(
    X_validation
)


# Normalize residual predictions before combining.
def standardize_signal(x):

    x = np.asarray(x)

    s = np.std(x)

    if s < 1e-12:
        return np.zeros_like(x)

    return (
        x - np.mean(x)
    ) / s


hgb_val_residual = standardize_signal(
    hgb_val_residual
)

et_val_residual = standardize_signal(
    et_val_residual
)

nn_val_residual = standardize_signal(
    nn_val_residual
)


# Base nonlinear signal
nonlinear_val = (
    0.40 * hgb_val_residual
    +
    0.30 * et_val_residual
    +
    0.30 * nn_val_residual
)


# Standardize Ridge too
ridge_val_std = standardize_signal(
    ridge_validation_prediction
)


# Candidate nonlinear strengths
strength_grid = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60
]

regimes = [
    "BULL",
    "NORMAL",
    "STRESS"
]

# Start with zero nonlinear contribution
best_strengths = {
    r: 0.0
    for r in regimes
}


# -------------------------------------------------------------------------
# Coordinate-search regime strengths.
#
# We optimize one regime at a time while holding the others fixed.
# -------------------------------------------------------------------------

current_score = portfolio_score(
    validation,
    ridge_val_std
)

for iteration in range(3):

    improved = False

    for regime in regimes:

        best_local_strength = best_strengths[
            regime
        ]

        best_local_score = current_score

        for candidate in strength_grid:

            temp_strengths = dict(
                best_strengths
            )

            temp_strengths[
                regime
            ] = candidate

            pred = ridge_val_std.copy()

            for r in regimes:

                mask = (
                    validation["regime"].values
                    == r
                )

                pred[mask] += (
                    temp_strengths[r]
                    *
                    nonlinear_val[mask]
                )

            score = portfolio_score(
                validation,
                pred
            )

            if score > best_local_score:

                best_local_score = score
                best_local_strength = candidate

        if (
            best_local_score
            >
            current_score
            + 1e-10
        ):

            best_strengths[
                regime
            ] = best_local_strength

            current_score = best_local_score

            improved = True

    if not improved:
        break


print("\nSelected regime strengths:")

for r in regimes:
    print(
        r,
        ":",
        best_strengths[r]
    )

print(
    "Validation portfolio score:",
    current_score
)


# =============================================================================
# 23. FINAL VALIDATION PREDICTION
# =============================================================================

final_validation_prediction = (
    ridge_val_std.copy()
)

for r in regimes:

    mask = (
        validation["regime"].values
        == r
    )

    final_validation_prediction[
        mask
    ] += (
        best_strengths[r]
        *
        nonlinear_val[mask]
    )


validation_rank_ic = rank_ic_score(
    validation,
    final_validation_prediction
)

validation_ls = long_short_series(
    pd.DataFrame({
        "date": validation["date"].values,
        TARGET: validation[TARGET].values,
        "prediction":
            final_validation_prediction
    }),
    "prediction"
)

validation_sharpe = annualized_sharpe(
    validation_ls
)

ridge_validation_ls = long_short_series(
    pd.DataFrame({
        "date": validation["date"].values,
        TARGET: validation[TARGET].values,
        "prediction":
            ridge_val_std
    }),
    "prediction"
)

ridge_validation_sharpe = annualized_sharpe(
    ridge_validation_ls
)

print("\n" + "=" * 80)
print("VALIDATION MODEL SELECTION")
print("=" * 80)

print(
    "Ridge validation Rank IC:",
    rank_ic_score(
        validation,
        ridge_val_std
    )
)

print(
    "AHRN validation Rank IC:",
    validation_rank_ic
)

print(
    "Ridge validation Sharpe:",
    ridge_validation_sharpe
)

print(
    "AHRN validation Sharpe:",
    validation_sharpe
)


# =============================================================================
# 24. FINAL MODEL — REFIT ON TRAIN + VALIDATION
# =============================================================================
#
# The test set remains untouched.
# =============================================================================

print("\n" + "=" * 80)
print("REFITTING FINAL AHRN ON TRAIN + VALIDATION")
print("=" * 80)

train_val = pd.concat(
    [
        train,
        validation
    ],
    axis=0
).copy()

X_train_val = scaler.fit_transform(
    train_val[MODEL_FEATURES]
)

X_test_final = scaler.transform(
    test[MODEL_FEATURES]
)

y_train_val = train_val[
    "y_cs"
].values


# -------------------------------------------------------------------------
# Ridge final
# -------------------------------------------------------------------------

final_ridge = Ridge(
    alpha=best_ridge.alpha
)

final_ridge.fit(
    X_train_val,
    y_train_val
)

ridge_tv_pred = final_ridge.predict(
    X_train_val
)

ridge_test_pred = final_ridge.predict(
    X_test_final
)


# -------------------------------------------------------------------------
# Residual target
# -------------------------------------------------------------------------

final_residual = (
    y_train_val
    - ridge_tv_pred
)

lo = np.quantile(
    final_residual,
    0.01
)

hi = np.quantile(
    final_residual,
    0.99
)

final_residual = np.clip(
    final_residual,
    lo,
    hi
)


# -------------------------------------------------------------------------
# Final HGB
# -------------------------------------------------------------------------

hgb_depth = best_hgb[2]
hgb_lr = best_hgb[3]
hgb_ntrees = best_hgb[4]
hgb_leaf = best_hgb[5]

final_hgb = HistGradientBoostingRegressor(
    max_iter=hgb_ntrees,
    learning_rate=hgb_lr,
    max_leaf_nodes=2 ** hgb_depth,
    min_samples_leaf=hgb_leaf,
    l2_regularization=5.0,
    random_state=SEED
)

final_hgb.fit(
    X_train_val,
    final_residual
)

hgb_test_residual = final_hgb.predict(
    X_test_final
)


# -------------------------------------------------------------------------
# Final ExtraTrees
# -------------------------------------------------------------------------

et_ntrees = best_et[2]
et_depth = best_et[3]
et_leaf = best_et[4]

final_et = ExtraTreesRegressor(
    n_estimators=et_ntrees,
    max_depth=et_depth,
    min_samples_leaf=et_leaf,
    max_features=0.70,
    random_state=SEED,
    n_jobs=-1
)

final_et.fit(
    X_train_val,
    final_residual
)

et_test_residual = final_et.predict(
    X_test_final
)


# -------------------------------------------------------------------------
# Final neural network
# -------------------------------------------------------------------------

final_nn = build_network(
    X_train_val.shape[1],
    best_nn[2]
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

final_nn.fit(
    X_train_val,
    final_residual,
    validation_split=0.10,
    epochs=60,
    batch_size=1024,
    verbose=0,
    callbacks=[early_stop]
)

nn_test_residual = final_nn.predict(
    X_test_final,
    batch_size=4096,
    verbose=0
).ravel()


# =============================================================================
# 25. FINAL NONLINEAR SIGNAL
# =============================================================================

hgb_test_residual = standardize_signal(
    hgb_test_residual
)

et_test_residual = standardize_signal(
    et_test_residual
)

nn_test_residual = standardize_signal(
    nn_test_residual
)

nonlinear_test = (
    0.40 * hgb_test_residual
    +
    0.30 * et_test_residual
    +
    0.30 * nn_test_residual
)

ridge_test_std = standardize_signal(
    ridge_test_pred
)

final_prediction = (
    ridge_test_std.copy()
)

for r in regimes:

    mask = (
        test["regime"].values
        == r
    )

    final_prediction[
        mask
    ] += (
        best_strengths[r]
        *
        nonlinear_test[mask]
    )


# =============================================================================
# 26. CROSS-SECTIONAL RANK NORMALIZATION
# =============================================================================
#
# Final score is converted to within-month percentile.
#
# This removes unstable prediction scale and directly targets stock ordering.
# =============================================================================

test_result = test[
    [
        "date",
        "ticker",
        TARGET,
        "regime"
    ]
].copy()

test_result[
    "ahrn_prediction_raw"
] = final_prediction

test_result[
    "ahrn_prediction"
] = (
    test_result
    .groupby("date")[
        "ahrn_prediction_raw"
    ]
    .rank(
        pct=True,
        method="average"
    )
)

# Center around zero
test_result[
    "ahrn_prediction"
] = (
    test_result[
        "ahrn_prediction"
    ] - 0.5
)


# =============================================================================
# 27. TEST PERFORMANCE
# =============================================================================

actual = test_result[TARGET].values
prediction = test_result[
    "ahrn_prediction"
].values

mse = mean_squared_error(
    actual,
    prediction
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    actual,
    prediction
)

r2 = r2_score(
    actual,
    prediction
)

pearson = np.corrcoef(
    actual,
    prediction
)[0, 1]

spearman = pd.Series(
    actual
).corr(
    pd.Series(prediction),
    method="spearman"
)

test_rank_ic = monthly_rank_ic(
    test_result,
    "ahrn_prediction"
)

test_ls = long_short_series(
    test_result,
    "ahrn_prediction"
)

test_sharpe = annualized_sharpe(
    test_ls
)

annualized_ls = (
    test_ls.mean()
    * 12.0
)

hit_rate = np.mean(
    test_ls > 0
)


# =============================================================================
# 28. RIDGE TEST BENCHMARK
# =============================================================================

ridge_result = test[
    [
        "date",
        TARGET
    ]
].copy()

ridge_result[
    "prediction"
] = ridge_test_std

ridge_test_ic = monthly_rank_ic(
    ridge_result,
    "prediction"
)

ridge_test_ls = long_short_series(
    ridge_result,
    "prediction"
)

ridge_test_sharpe = annualized_sharpe(
    ridge_test_ls
)


# =============================================================================
# 29. DECILE ANALYSIS
# =============================================================================

test_result[
    "decile"
] = test_result.groupby(
    "date"
)[
    "ahrn_prediction"
].transform(
    lambda x: pd.qcut(
        x.rank(
            method="first"
        ),
        10,
        labels=False
    ) + 1
)

decile_table = (
    test_result
    .groupby("decile")
    .agg(
        observations=(
            TARGET,
            "count"
        ),
        mean_prediction=(
            "ahrn_prediction",
            "mean"
        ),
        mean_actual_return=(
            TARGET,
            "mean"
        ),
        median_actual_return=(
            TARGET,
            "median"
        ),
        actual_std=(
            TARGET,
            "std"
        )
    )
)


# =============================================================================
# 30. REGIME-CONDITIONAL RESULTS
# =============================================================================

regime_rows = []

for regime in regimes:

    sub = test_result[
        test_result["regime"] == regime
    ].copy()

    if len(sub) == 0:
        continue

    regime_ic = monthly_rank_ic(
        sub,
        "ahrn_prediction"
    )

    regime_ls = long_short_series(
        sub,
        "ahrn_prediction"
    )

    regime_rows.append({
        "regime": regime,
        "observations": len(sub),
        "rank_ic": (
            np.mean(regime_ic)
            if len(regime_ic)
            else np.nan
        ),
        "sharpe": annualized_sharpe(
            regime_ls
        ),
        "mean_ls": (
            regime_ls.mean()
            if len(regime_ls)
            else np.nan
        )
    })

regime_results = pd.DataFrame(
    regime_rows
)


# =============================================================================
# 31. PRINT FINAL RESULTS
# =============================================================================

print("\n" + "=" * 80)
print("FINAL AHRN TEST PERFORMANCE")
print("=" * 80)

print(
    f"MSE       : {mse:.8f}"
)

print(
    f"RMSE      : {rmse:.8f}"
)

print(
    f"MAE       : {mae:.8f}"
)

print(
    f"R2        : {r2:.8f}"
)

print(
    f"Pearson   : {pearson:.8f}"
)

print(
    f"Spearman  : {spearman:.8f}"
)


print("\n" + "=" * 80)
print("FINAL RANK IC")
print("=" * 80)

print(
    "Mean:",
    np.mean(test_rank_ic)
)

print(
    "Median:",
    np.median(test_rank_ic)
)

print(
    "Std:",
    np.std(test_rank_ic, ddof=1)
)

print(
    "IR:",
    np.mean(test_rank_ic)
    /
    (
        np.std(
            test_rank_ic,
            ddof=1
        ) / np.sqrt(
            len(test_rank_ic)
        )
    )
)

print(
    "Positive months:",
    np.sum(test_rank_ic > 0),
    "/",
    len(test_rank_ic)
)


print("\n" + "=" * 80)
print("FINAL LONG-SHORT PERFORMANCE")
print("=" * 80)

print(
    "Months:",
    len(test_ls)
)

print(
    "Mean monthly L/S:",
    test_ls.mean()
)

print(
    "Annualized L/S:",
    annualized_ls
)

print(
    "Annualized Sharpe:",
    test_sharpe
)

print(
    "Hit rate:",
    hit_rate
)


print("\n" + "=" * 80)
print("AHRN VS RIDGE")
print("=" * 80)

print(
    "Ridge test Rank IC:",
    np.mean(ridge_test_ic)
)

print(
    "AHRN test Rank IC:",
    np.mean(test_rank_ic)
)

print(
    "Ridge Sharpe:",
    ridge_test_sharpe
)

print(
    "AHRN Sharpe:",
    test_sharpe
)

print(
    "Sharpe improvement:",
    test_sharpe
    - ridge_test_sharpe
)

print(
    "Rank IC improvement:",
    np.mean(test_rank_ic)
    -
    np.mean(ridge_test_ic)
)


print("\n" + "=" * 80)
print("DECILE ANALYSIS")
print("=" * 80)

print(
    decile_table
)


print("\n" + "=" * 80)
print("REGIME-CONDITIONAL PERFORMANCE")
print("=" * 80)

print(
    regime_results
)


# =============================================================================
# 32. MONOTONICITY
# =============================================================================

decile_returns = (
    decile_table[
        "mean_actual_return"
    ]
    .values
)

positive_steps = np.sum(
    np.diff(decile_returns) > 0
)

monotonicity = (
    positive_steps / 9.0
)

print("\n" + "=" * 80)
print("DECILE MONOTONICITY")
print("=" * 80)

print(
    "Positive steps:",
    positive_steps,
    "/ 9"
)

print(
    "Monotonicity:",
    monotonicity
)


# =============================================================================
# 33. PREDICTION SCALE
# =============================================================================

actual_std = np.std(
    actual
)

prediction_std = np.std(
    prediction
)

print("\n" + "=" * 80)
print("PREDICTION SCALE")
print("=" * 80)

print(
    "Actual return std:",
    actual_std
)

print(
    "Prediction std:",
    prediction_std
)

print(
    "Prediction / actual:",
    prediction_std
    / actual_std
    if actual_std > 0
    else np.nan
)


# =============================================================================
# 34. SAVE PREDICTIONS
# =============================================================================

prediction_path = (
    "results/"
    "ahrn_final_predictions.csv"
)

test_result.to_csv(
    prediction_path,
    index=False
)


# =============================================================================
# 35. SAVE MODEL BUNDLE
# =============================================================================

model_bundle = {

    "ridge": final_ridge,

    "hgb": final_hgb,

    "extra_trees": final_et,

    "neural_network": final_nn,

    "scaler": scaler,

    "features": MODEL_FEATURES,

    "regime_strengths":
        best_strengths,

    "ridge_alpha":
        best_ridge.alpha,

    "hgb_config":
        best_hgb[2:],

    "et_config":
        best_et[2:],

    "nn_width":
        best_nn[2],

    "seed":
        SEED
}

model_path = (
    "results/"
    "ahrn_final_models.pkl"
)

with open(
    model_path,
    "wb"
) as f:

    pickle.dump(
        model_bundle,
        f
    )


# =============================================================================
# 36. SAVE RESULTS
# =============================================================================

results = {

    "model":
        "Adaptive Cross-Sectional Residual AHRN",

    "features":
        MODEL_FEATURES,

    "ridge_alpha":
        float(best_ridge.alpha),

    "regime_strengths":
        {
            k: float(v)
            for k, v in best_strengths.items()
        },

    "validation_rank_ic":
        float(validation_rank_ic),

    "validation_sharpe":
        float(validation_sharpe),

    "ridge_validation_rank_ic":
        float(
            rank_ic_score(
                validation,
                ridge_val_std
            )
        ),

    "ridge_validation_sharpe":
        float(
            ridge_validation_sharpe
        ),

    "test_rank_ic":
        float(np.mean(test_rank_ic)),

    "test_rank_ic_median":
        float(np.median(test_rank_ic)),

    "test_rank_ic_std":
        float(
            np.std(
                test_rank_ic,
                ddof=1
            )
        ),

    "test_rank_ic_ir":
        float(
            np.mean(test_rank_ic)
            /
            (
                np.std(
                    test_rank_ic,
                    ddof=1
                )
                /
                np.sqrt(
                    len(test_rank_ic)
                )
            )
        ),

    "test_sharpe":
        float(test_sharpe),

    "ridge_test_sharpe":
        float(ridge_test_sharpe),

    "sharpe_improvement":
        float(
            test_sharpe
            - ridge_test_sharpe
        ),

    "test_annualized_ls":
        float(annualized_ls),

    "test_hit_rate":
        float(hit_rate),

    "test_pearson":
        float(pearson),

    "test_spearman":
        float(spearman),

    "decile_monotonicity":
        float(monotonicity),

    "prediction_actual_volatility_ratio":
        float(
            prediction_std
            / actual_std
        )
}

results_path = (
    "results/"
    "ahrn_final_results.json"
)

with open(
    results_path,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# =============================================================================
# 37. FINAL VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("FINAL AHRN VALIDATION")
print("=" * 80)

checks = {

    "Dataset exists":
        os.path.exists(DATA_PATH),

    "Prediction file":
        os.path.exists(
            prediction_path
        ),

    "Model file":
        os.path.exists(
            model_path
        ),

    "Results file":
        os.path.exists(
            results_path
        ),

    "Train inputs finite":
        np.isfinite(
            X_train
        ).all(),

    "Validation inputs finite":
        np.isfinite(
            X_validation
        ).all(),

    "Test inputs finite":
        np.isfinite(
            X_test_final
        ).all(),

    "Test predictions finite":
        np.isfinite(
            prediction
        ).all(),

    "Prediction count":
        len(test_result)
        == len(test),

    "No duplicate ticker-date":
        not test_result[
            ["ticker", "date"]
        ].duplicated().any(),

    "Chronological split":
        max(train_dates)
        <
        min(validation_dates)
        <
        min(test_dates),

    "Feature count":
        len(MODEL_FEATURES) >= 20
}

for name, passed in checks.items():

    print(
        f"{name}:",
        "PASS" if passed else "FAIL"
    )


# =============================================================================
# 38. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("FINAL AHRN RESULTS")
print("=" * 80)

print(
    f"Test Rank IC:              "
    f"{np.mean(test_rank_ic):.6f}"
)

print(
    f"Rank IC IR:                "
    f"{np.mean(test_rank_ic) / (np.std(test_rank_ic, ddof=1) / np.sqrt(len(test_rank_ic))):.6f}"
)

print(
    f"Annualized L/S:            "
    f"{annualized_ls:.6f}"
)

print(
    f"AHRN Sharpe:               "
    f"{test_sharpe:.6f}"
)

print(
    f"Ridge Sharpe:              "
    f"{ridge_test_sharpe:.6f}"
)

print(
    f"Sharpe improvement:        "
    f"{test_sharpe - ridge_test_sharpe:.6f}"
)

print(
    f"Long-short hit rate:       "
    f"{hit_rate:.4f}"
)

print(
    f"Decile monotonicity:       "
    f"{monotonicity:.4f}"
)

print(
    f"Prediction/actual vol:     "
    f"{prediction_std / actual_std:.6f}"
)

print("\nFiles saved:")
print(prediction_path)
print(model_path)
print(results_path)

print("\n" + "=" * 80)
print("AHRN FINAL MODEL COMPLETE")
print("=" * 80)